# MMA3001 — Notebook 02: Fresh Historical-Method Development Reconstruction

**Status: restored-source candidate, not yet a completed full rerun.** This notebook executes the **original Gaussian-process Bayesian methodology from scratch** and derives Stage 9 parameters from the new Bayesian output. It is **not** the 128-samples/profile finite-grid reconstruction. This is development-only: it stops **before any final six-model fitting and before any held-out profile prediction**.

### Provenance rules

- Primary source: `Permanent_Magnet_Temperature_Estimation_Run_All_Full_Data.ipynb`, code cells 2–102, in their original order.
- Later Stage 9: `02_Experimental_Development_EvidenceIndexed(1).ipynb`, code cells 108–147.
- The principal algorithms, objectives, model definitions, folds, sample budgets, candidate spaces and Gaussian-process optimiser settings come from the historical source. **Only data provenance handoffs and computational packaging are changed**: Stage 5 and Stage 9 consume *freshly selected* SVR winners; full-data OLS is seeded by the *freshly selected* expanded-pilot OLS spans. No finite-grid replacement is used.
- The source restoration retains original algorithm code; its exceptions now include dataset acquisition, Colab setup, the three fresh-parameter handoffs, and results/documentation export. A source-cell change log details them. Parameter values may differ from the archived run because searches are freshly executed.
- Historical prose may report results of earlier runs; **do not treat those old numbers as outputs of this fresh run**. Fresh figures, trial CSVs and the final manifest are the authoritative outputs.
- Stage 9 **consumes this execution's newly selected larger-sample Bayesian SVR winner**, including its temporal spans, `C` and `epsilon`. It does not load or copy any historical fitted model or selected numerical result. The Stage 9 *algorithms and relative penalty sweeps* are the historical ones; only their baseline input is now determined afresh.

**Run instructions:** Open in a fresh Colab Python runtime; use **Runtime → Run all**. Expect substantially longer execution and higher memory use than the finite-grid notebook. Download the final ZIP and executed `.ipynb` afterward. Set `PMSM_LOCAL_CSV` to the path of a verified copy of the original `measures_v2.csv` if already available, otherwise public KaggleHub download is used.

**Acceptance limitation:** Exact reconstruction of experiment *code* does not guarantee identical results if the original and current Python/Scikit-learn/NumPy/Scikit-Optimize versions differ. Version hashes and outputs are recorded; deviations must be interpreted, not concealed.


## Scientific aim, research questions and interpretation

**Engineering objective:** estimate the permanent-magnet temperature (°C) of a PMSM using observable electrical, mechanical and thermal operating conditions.

**Research aim:** investigate accurate and computationally feasible regression methods using profile-disjoint validation and historical numerical experiments.

1. **RQ1 — Leakage-safe time-series validation:** How should temporally correlated motor recordings be processed and assigned to folds without overlapping profiles or using future samples in causal features?
2. **RQ2 — Baseline model accuracy–cost:** Under the original Bayesian optimisations, what accuracy and computational costs arise for OLS, k-NN, Random Forest, Extra Trees and MLP?
3. **RQ3 — SVR feasibility:** How do exact RBF-SVR, matched-kernel Nyström approximations and direct LinearSVR compare as the number of training observations increases?
4. **RQ4 — Numerical evidence:** How do numerical checks, grouped-fold variability, timing, convergence and regularisation sensitivity support or limit the development-stage conclusions?

**Separation of responsibilities:** This notebook executes selection and development-validation only. Notebook 01 will independently do final model fitting and reused historical 14-profile evaluation *after* this notebook is audited. The earlier inspection of that held-out set must remain disclosed as a limitation.


### Project-brief documentation deliverables

The repository package contains `src/pmsm_project/` with documented, source-derived helper functions, `docs/` with browsable HTML documentation, `tests/` with pytest tests, a test report, a source-provenance log, a licence for this repository's authored code, and the execution README. HTML documentation is **a repository deliverable**, not a claim that notebook Markdown alone satisfies pdoc. On a fresh Colab execution, an end-of-run documentation cell recreates the source modules and attempts `pdoc` generation; if pdoc installation is unavailable, it reports failure rather than silently claiming completion. Notebook 02 still uses its original algorithm cells rather than importing changed implementations.

**Full-workflow limit:** Until `Runtime → Run all` completes, the expensive search outcomes are unknown. The submitted source code and locally generated HTML/tests do not claim the full Bayesian searches have run.

## 0. Environment and immutable-source verification

The following cell preserves the original data-download approach and adds only an optional local CSV override and environment output. It does **not** alter the observations or data transformations. The exact dataset SHA-256 is checked after loading.

In [ ]:
# Reproducible environment and Kaggle dataset acquisition
# Install dependencies on a clean Google Colab runtime.
%pip -q install kagglehub scikit-optimize

import os
from pathlib import Path
import numpy as np
import pandas as pd
import kagglehub

# Working outputs are regenerated from scratch; no previous files are required.
BASE_DIR = Path('/content')
BASE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = str(BASE_DIR / 'bayesian_model_pilots')
SEARCH_DIR = str(BASE_DIR / 'scaled_model_bayesian')
for directory in [OUTPUT_DIR, SEARCH_DIR]:
    Path(directory).mkdir(parents=True, exist_ok=True)

# Public KaggleHub dataset; download/cache is automatic on each fresh runtime.
local_csv_override = os.environ.get('PMSM_LOCAL_CSV', '').strip()
if local_csv_override:
    assert Path(local_csv_override).is_file(), 'PMSM_LOCAL_CSV does not exist'
    path = str(Path(local_csv_override).parent)
else:
    path = kagglehub.dataset_download('wkirgsn/electric-motor-temperature')
print('Kaggle dataset directory:', path)


# Environment/provenance only — no changed model behaviour.
import sys, platform, sklearn, scipy
print('Python:',sys.version.split()[0]); print('Platform:',platform.platform())
print('NumPy:',np.__version__,'Pandas:',pd.__version__,'Scikit-learn:',sklearn.__version__,'SciPy:',scipy.__version__)


## 2.1 Dataset Description & Source

The dataset used in this investigation is the PMSM Electric Motor Temperature Measurement dataset, obtained from Kaggle and associated with Kirchgässner et al. (2021), Data-Driven Permanent Magnet Temperature Estimation in Synchronous Motors With Supervised Machine Learning: A Benchmark. The dataset contains measurements collected from a permanent magnet synchronous motor (PMSM) test bench operating at a sampling frequency of 2 Hz, corresponding to one observation every 0.5 seconds. The downloaded dataset contains 1,330,816 observations, 13 columns and 69 measurement profiles, representing approximately 184.84 hours of recorded motor operation. This is an expanded version of the dataset used in the original paper, which reported approximately 139 hours of measurements. Consequently, the published results will be used as reference benchmarks rather than targets for exact numerical reproduction.


In [ ]:
import pandas as pd
import os

# Locate the CSV file downloaded by KaggleHub
csv_path = local_csv_override if local_csv_override else os.path.join(path, "measures_v2.csv")
assert os.path.isfile(csv_path), f"Dataset not found: {csv_path}"

print("File location:", os.path.abspath(csv_path))
print("File size (MB):", os.path.getsize(csv_path) / 1e6)

df = pd.read_csv(csv_path)

print("Observations:", len(df))
print("Columns:", len(df.columns))
print("Unique profiles:", df["profile_id"].nunique())
print("Recording hours:", len(df) / (2 * 3600))

In [ ]:
# Provenance guard; source measurements must match the project dataset exactly.
import hashlib
DATASET_SHA256_EXPECTED='78f3d150f0f2ad9c5dc7ff24dd12c00d386ad530f48c1589ad24fcd88867d3ad'
with open(csv_path,'rb') as source_file:
    hasher=hashlib.sha256()
    for block in iter(lambda:source_file.read(8*1024*1024),b''):
        hasher.update(block)
DATASET_SHA256_ACTUAL=hasher.hexdigest()
assert DATASET_SHA256_ACTUAL==DATASET_SHA256_EXPECTED, 'Unexpected dataset bytes: stop and investigate.'
assert len(df)==1_330_816 and df.profile_id.nunique()==69
print('PASS: Original dataset SHA-256, rows and profile count match.')

## 2.2 Inputs & Outputs

Each observation contains measurements of the motor's electrical, mechanical and thermal operating conditions. The dataset includes variables such as motor speed, torque, electrical currents, voltages, coolant temperature and ambient temperature, alongside several measured component temperatures. The primary prediction target is the permanent-magnet temperature, represented by the pm column.

The intended computational output is an estimated magnet temperature PM obtained using supervised machine-learning regression. The profile_id column identifies individual motor-recording sessions and is used for data partitioning rather than as a predictive input. The final model input set will be defined using the relevant measured and derived variables described in Table I of Kirchgässner et al. (2021). Temporal features based on exponentially weighted moving averages (EWMA) and standard deviations (EWMS) will subsequently be introduced.


**Historical original notebook figure omitted from this reconstruction:** it is an embedded screenshot, not an experiment or algorithm. The original is preserved unchanged in the source archive.

In [ ]:
profile_summary = (
    df.groupby("profile_id")
    .agg(
        observations=("pm", "size"),
        pm_min=("pm", "min"),
        pm_max=("pm", "max")
    )
    .reset_index()
)

profile_summary["duration_hours"] = (
    profile_summary["observations"] / (2 * 3600)
)

display(profile_summary)

## 2.3 Data Quality Assesment
Initial data inspection was conducted using Python and Pandas in Google Colab. The number of observations, available variables, recording profiles and temperature ranges were examined. The following checks were performed:

- **Missing values:** No missing values were identified across the 13 columns.
- **Duplicate observations:** No fully duplicated rows were identified.
- **Profile continuity:** All 69 measurement profiles appeared as individual continuous blocks within the dataset.
- **Recording duration:** Profile durations were estimated using the documented 2 Hz sampling frequency.
- **Temperature coverage:** Individual profiles exhibited different minimum and maximum permanent-magnet temperatures.



In [ ]:
# Identify consecutive blocks of profile IDs
blocks = df["profile_id"].ne(
    df["profile_id"].shift()
).cumsum()

# Count distinct continuous blocks
n_blocks = blocks.nunique()
n_profiles = df["profile_id"].nunique()

print("Measurement profiles:", n_profiles)
print("Continuous blocks:", n_blocks)

if n_blocks == n_profiles:
    print("PASS: Each profile appears in one continuous block.")
else:
    print("CHECK: Some profiles may appear in multiple blocks.")

# Check missing values and duplicate rows
print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

These results indicate that the dataset is suitable for further preprocessing and supervised regression modelling. However, the preliminary checks do not establish that all measurements are physically plausible or free from sensor errors. Since the dataset does not contain an explicit timestamp column, elapsed time within each profile can be reconstructed from sample position and the documented sampling interval, provided the original ordering is preserved.

## 2.4 Data Splitting Method
An initial 80/20 row-based partition was investigated. However, this approach divided recording profile 65 between the development and final test sets, meaning that observations from the same continuous motor-testing session appeared in both partitions.This was considered inappropriate for the intended evaluation objective: determining whether the estimator can generalise to previously unseen motor operating sessions.


In [ ]:
cutoff = int(len(df) * 0.80)

train = df.iloc[:cutoff]
test = df.iloc[cutoff:]

shared_profiles = (
    set(train["profile_id"])
    & set(test["profile_id"])
)

print("Training rows:", len(train))
print("Testing rows:", len(test))
print("Profiles appearing in both:", shared_profiles)

Consequently, a profile-grouped partitioning strategy was adopted, following the grouping principle applied by Kirchgässner et al. (2021).To maintain temperature-condition coverage, profiles were divided into three groups according to their maximum recorded permanent-magnet temperature. A stratified split was then performed using a fixed random seed of 42.


In [ ]:

import pandas as pd
from sklearn.model_selection import train_test_split

# STEP 4A: Summarise each measurement profile
profile_table = (
    df.groupby("profile_id")
    .agg(
        observations=("pm", "size"),
        pm_min=("pm", "min"),
        pm_max=("pm", "max"),
        pm_mean=("pm", "mean")
    )
    .reset_index()
    .sort_values("profile_id")
    .reset_index(drop=True)
)

# STEP 4B: Stratify profiles by maximum PM temperature
# Three approximately equal-sized temperature groups
profile_table["temperature_group"] = pd.qcut(
    profile_table["pm_max"],
    q=3,
    labels=["Low", "Medium", "High"]
)

# STEP 4C: Split complete profiles, NOT individual rows
train_ids, test_ids = train_test_split(
    profile_table["profile_id"],
    test_size=0.20,
    random_state=42,
    stratify=profile_table["temperature_group"]
)

train_ids = sorted(train_ids.tolist())
test_ids = sorted(test_ids.tolist())

# STEP 4D: Create development and final test datasets
train_df = df[df["profile_id"].isin(train_ids)].copy()
test_df = df[df["profile_id"].isin(test_ids)].copy()

# STEP 4E: Verify no profile overlap
assert set(train_ids).isdisjoint(set(test_ids))
assert len(train_df) + len(test_df) == len(df)

print("Development profiles:", len(train_ids))
print("Test profiles:", len(test_ids))
print("Development observations:", len(train_df))
print("Test observations:", len(test_df))

print("\nActual proportions:")
print(f"Development: {len(train_df)/len(df)*100:.2f}%")
print(f"Final test:  {len(test_df)/len(df)*100:.2f}%")

print("\nFinal test profile IDs:", test_ids)
print("Profile overlap:", set(train_ids) & set(test_ids))



In [ ]:

# Assign split labels to profiles
profile_table["split"] = profile_table["profile_id"].apply(
    lambda pid: "Test" if pid in test_ids else "Development"
)

# Count low/medium/high temperature profiles in each set
print("Temperature-group distribution:")
display(
    pd.crosstab(
        profile_table["split"],
        profile_table["temperature_group"]
    )
)

# Compare measurement coverage
for name, data in [("Development", train_df), ("Test", test_df)]:
    print(f"\n{name} set:")
    print("Profiles:", data["profile_id"].nunique())
    print("PM range (°C):", data["pm"].min(), "to", data["pm"].max())
    print("Motor speed range:", data["motor_speed"].min(),
          "to", data["motor_speed"].max())
    print("Torque range:", data["torque"].min(),
          "to", data["torque"].max())

# Save profile assignments for reproducibility
profile_table.to_csv("profile_split.csv", index=False)



| Property | Development Set | Final Test Set |
|:---|:---:|:---:|
| Measurement Profiles | 55 | 14 |
| Observations | 1,055,198 | 275,618 |
| Dataset Proportion | 79.29% | 20.71% |
| Recording Duration | 146.56 h | 38.28 h |
| Low-temperature profiles | 18 | 5 |
| Medium-temperature profiles | 18 | 5 |
| High-temperature profiles | 19 | 4 |

The absence of overlapping profile identifiers was verified programmatically. The achieved proportions differ slightly from the targeted 80/20 division because complete recording sessions were preserved. The approximately 20% test allocation was selected as a practical holdout size consistent with the evaluation approach demonstrated in the MMA3001 teaching material, while retaining sufficient development observations for model training and cross-validation. It is an adaptation rather than an exact reproduction of the original paper's seven-hour holdout.

The selected test profiles will remain excluded from model fitting and hyperparameter optimisation. Within the development set, stratified grouped three-fold cross-validation will be implemented for model selection and optimisation. The same profile assignments will subsequently be used to evaluate the effects of reducing the estimator's sampling frequency, ensuring consistent comparisons between the original and modified computational approaches.


## 2.5 Scope and Limitations
Initial data inspection was conducted using Python and Pandas in Google Colab. The number of observations, available variables, recording profiles and temperature ranges were examined. The following checks were performed:


# 3 Computational Solution



## 3.1 Temporal Feature Engineering

Following the methodology of Kirchgässner et al. (2021), exponentially weighted moving averages (EWMA) and exponentially weighted moving standard deviations (EWMS) were implemented to incorporate the historical behaviour of motor operating conditions into the machine-learning inputs.

This feature engineering is necessary because permanent-magnet temperature depends on both current and previous motor operation. Conventional regression models, including ordinary least squares (OLS), do not explicitly maintain a history of previous observations. EWMA and EWMS provide historical information through additional predictor variables.

The EWMA was calculated using:

$$
\mu_t = \frac{\sum_{i=0}^{t} w_i x_{t-i}}{\sum_{i=0}^{t} w_i}
$$

where the weighting factor and smoothing parameter are defined as:

$$
w_i = (1-\alpha)^i, \qquad \alpha = \frac{2}{s+1}
$$

Here, $x_t$ represents the sensor measurement at time $t$, $\alpha$ controls the weighting of historical observations, and $s$ represents the smoothing span.

Larger spans preserve the influence of older measurements for longer, while smaller spans respond more rapidly to changes in motor operating conditions.


In [ ]:

import numpy as np
import pandas as pd

def make_paper_inputs(data):
    """Construct the 12 PMSM inputs from Table I of the paper."""

    X = pd.DataFrame(index=data.index)

    # Seven measured inputs
    X["theta_a"] = data["ambient"]
    X["theta_c"] = data["coolant"]
    X["u_d"] = data["u_d"]
    X["u_q"] = data["u_q"]
    X["i_d"] = data["i_d"]
    X["i_q"] = data["i_q"]
    X["n_mech"] = data["motor_speed"]

    # Five derived inputs
    X["u_s"] = np.hypot(X["u_d"], X["u_q"])
    X["i_s"] = np.hypot(X["i_d"], X["i_q"])

    # Apparent electrical power
    X["S_el"] = 1.5 * X["u_s"] * X["i_s"]

    # Angular speed in rad/s
    omega = 2 * np.pi * X["n_mech"] / 60

    X["i_s_omega"] = X["i_s"] * omega
    X["S_el_omega"] = X["S_el"] * omega

    return X



## 3.2 Python Implementation

The computational implementation was developed in Python using Pandas and NumPy.

First, twelve predictor variables were constructed from seven measured quantities and five derived quantities, following Table I of Kirchgässner et al. (2021).

A reusable function `make_temporal_features()` was then developed to generate EWMA and EWMS features for each predictor variable and smoothing span.

Four illustrative spans of 10, 50, 200 and 1,000 samples were initially selected to verify the implementation. These values were selected to demonstrate different smoothing timescales and do not represent optimised hyperparameters.

The resulting feature matrix consisted of:

| Feature category | Number |
|:-----------------|-------:|
| Original measured and derived predictors | 12 |
| EWMA features | 48 |
| EWMS features | 48 |
| **Total predictor variables** | **108** |


In [ ]:

def make_temporal_features(X, profile_ids, spans):
    # Calculate temporal features independently per profile
    features = {}

    for column in X.columns:
        grouped = X[column].groupby(
            profile_ids, sort=False
        )

        for span in spans:
            features[f"{column}_ewma_s{span}"] = (
                grouped.transform(
                    lambda z: z.ewm(
                        span=span, adjust=True
                    ).mean()
                )
            )

            features[f"{column}_ewms_s{span}"] = (
                grouped.transform(
                    lambda z: z.ewm(
                        span=span, adjust=True
                    ).std(bias=True)
                )
            )

    return pd.concat(
        [X, pd.DataFrame(features, index=X.index)],
        axis=1
    )


In [ ]:

# Test 1: Compare EWMA/EWMS against manual calculations

signal = np.array([10.0, 20.0, 30.0])
span = 3
alpha = 2 / (span + 1)

weights = (1 - alpha) ** np.arange(2, -1, -1)

manual_mean = np.sum(weights * signal) / weights.sum()

manual_std = np.sqrt(
    np.sum(weights * (signal - manual_mean)**2)
    / weights.sum()
)

pandas_ewm = pd.Series(signal).ewm(
    span=span, adjust=True
)

np.testing.assert_allclose(
    pandas_ewm.mean().iloc[-1], manual_mean
)

np.testing.assert_allclose(
    pandas_ewm.std(bias=True).iloc[-1],
    manual_std
)

print("PASS: EWMA/EWMS mathematical verification")
print("Manual EWMA:", manual_mean)
print("Manual EWMS:", manual_std)


# Test 2: Check that memory resets between profiles

toy_X = pd.DataFrame({
    "sensor": [10., 20., 30., 100., 100.]
})

toy_ids = pd.Series([1, 1, 1, 2, 2])

toy_features = make_temporal_features(
    toy_X, toy_ids, spans=[3]
)

assert toy_features.loc[3, "sensor_ewma_s3"] == 100.
assert toy_features.loc[3, "sensor_ewms_s3"] == 0.

print("PASS: Profile memory reset")


# Test 3: Verify that future values do not affect past features

modified_X = toy_X.copy()
modified_X.loc[2, "sensor"] = 100000.

modified_features = make_temporal_features(
    modified_X, toy_ids, spans=[3]
)

np.testing.assert_allclose(
    toy_features.loc[:1, "sensor_ewma_s3"],
    modified_features.loc[:1, "sensor_ewma_s3"]
)

print("PASS: No future-data leakage")



## 3.3 Numerical and Real-Data Verification

Verification was performed before proceeding to machine-learning model training.

Initially, EWMA and EWMS calculations were compared against manually evaluated results using a synthetic signal of (10, 20, 30) and a smoothing span of 3. The manually calculated values agreed with the Pandas implementation within numerical tolerance.

Additional tests confirmed that temporal memory restarted at profile boundaries and that changing a future observation did not affect earlier EWMA results.

The implementation was subsequently tested using actual PMSM measurements. The first 300 observations from each of two development profiles (IDs 2 and 3) were selected, representing 600 observations across two independent sessions.

Four illustrative spans were applied to each of the twelve predictors, generating the expected 108-feature matrix.

**Table X. Real-data feature-engineering verification results.**

| Verification criterion | Result |
|:-----------------------|:-------|
| Measurement profiles | 2 |
| Observations | 600 |
| Original predictor variables | 12 |
| Generated predictor variables | 108 |
| Missing feature values | 0 |
| Non-finite feature values | 0 |
| Feature-dimension verification | Passed |
| Profile-initialisation verification | Passed |

The results confirmed that the feature-engineering pipeline produced the expected dimensions and numerically valid outputs for the selected real-data samples.

In [ ]:

# Select two development profiles only
demo_ids = train_ids[:2]

demo_df = (
    train_df[train_df["profile_id"].isin(demo_ids)]
    .groupby("profile_id", sort=False)
    .head(300)
    .copy()
)

# Construct the twelve input variables
X_demo = make_paper_inputs(demo_df)

# Illustrative spans (NOT yet optimised)
example_spans = [10, 50, 200, 1000]

# Generate temporal features
X_temporal = make_temporal_features(
    X_demo,
    demo_df["profile_id"],
    spans=example_spans
)

# Verify feature matrix
assert X_temporal.shape[1] == 108
assert X_temporal.isna().sum().sum() == 0
assert np.isfinite(X_temporal.to_numpy()).all()

print("Profiles:", demo_df["profile_id"].nunique())
print("Original inputs:", X_demo.shape[1])
print("Total features:", X_temporal.shape[1])
print("Missing values:", X_temporal.isna().sum().sum())

display(X_temporal.iloc[:5, :16])


## 3.4 Visual Verification of EWMA Behaviour

The implementation was further assessed by plotting the q-axis motor current against its exponentially weighted moving averages for measurement Profile 2.

Four illustrative spans of 10, 50, 200 and 1,000 samples were evaluated over the first 150 seconds of the recording.

The results demonstrated that smaller spans responded more rapidly to changes in the original current signal, whereas larger spans produced slower and more heavily smoothed responses.

Figure X. Raw q-axis current and corresponding EWMA responses for four smoothing spans in PMSM measurement Profile 2.

The observed behaviour was consistent with the expected mathematical characteristics of exponential smoothing and provided qualitative evidence supporting the implementation.

However, the numerical and visual verification results do not establish whether the selected temporal features improve permanent-magnet temperature prediction. Furthermore, the implementation uses a weighted standard deviation for EWMS, while Equation (4) of Kirchgässner et al. (2021) contains an ambiguity concerning the variance and standard-deviation convention.

The selection of optimal spans and the quantitative assessment of temperature-prediction accuracy will therefore be conducted during subsequent Bayesian optimisation and grouped cross-validation.

In [ ]:

import matplotlib.pyplot as plt

# Examine one measurement profile
pid = demo_ids[0]

mask = demo_df["profile_id"] == pid
time = np.arange(mask.sum()) / 2  # Seconds at 2 Hz

plt.figure(figsize=(11, 5))

plt.plot(
    time,
    X_temporal.loc[mask, "i_q"].to_numpy(),
    label="Raw q-axis current"
)

for span in [10, 50, 200,1000]:
    plt.plot(
        time,
        X_temporal.loc[
            mask, f"i_q_ewma_s{span}"
        ].to_numpy(),
        label=f"EWMA span={span}"
    )

plt.xlabel("Elapsed time (s)")
plt.ylabel("q-axis current (A)")
plt.title(f"EWMA verification — Profile {pid}")
plt.legend()
plt.grid(True)
plt.show()


## 4. Implementation of Stratified Grouped 3-Fold Cross-Validation

## 4.1 Implementation of Stratified Grouped 3-Fold Cross-Validation
Following the feature-engineering verification, stratified grouped three-fold cross-validation was implemented to establish a consistent methodology for evaluating the regression models and their hyperparameters.

The previously established development/test partition was restored using the dataset downloaded through KaggleHub. The development partition consisted of 55 measurement profiles containing 1,055,198 observations, while the final test partition contained 14 profiles with 275,618 observations. The final test partition was reserved for subsequent model evaluation and excluded from cross-validation and hyperparameter optimisation.

A profile-level summary was constructed to calculate the number of observations and minimum and maximum permanent-magnet temperatures for each development profile. The profiles were then classified into three temperature groups: Low, Medium and High using the maximum recorded permanent-magnet temperature as the stratification criterion.

The StratifiedKFold method from Scikit-learn was applied to this profile-level summary using three folds, random shuffling and a fixed random seed of 42. This approach follows the principle of temperature-stratified profile-based validation adopted by Kirchgässner et al. (2021).

Importantly, the folds were constructed from complete measurement profiles rather than individual observations. This prevented measurements belonging to the same recording session from appearing simultaneously in the training and validation subsets of a given cross-validation iteration.

The implementation included assertions to verify the correct development/test partition sizes, complete fold assignment and absence of profile overlap. The resulting fold assignments were saved to profile_cv_assignments.csv for reproducibility.

In [ ]:

import pandas as pd
from sklearn.model_selection import StratifiedKFold

# =====================================================
# STEP 6.1A — Restore the existing development/test split
# =====================================================

# df is already loaded from KaggleHub

test_ids = [
    13, 16, 23, 24, 29, 42, 44,
    47, 53, 56, 62, 67, 71, 78
]

train_df = df[
    ~df["profile_id"].isin(test_ids)
].copy()

test_df = df[
    df["profile_id"].isin(test_ids)
].copy()

# Verify the previously established partition
assert len(df) == 1330816
assert len(train_df) == 1055198
assert len(test_df) == 275618
assert set(train_df["profile_id"]).isdisjoint(
    set(test_df["profile_id"])
)

print("PASS: Development/test split restored")


# =====================================================
# STEP 6.1B — Summarise development profiles
# =====================================================

profile_cv = (
    train_df.groupby("profile_id")["pm"]
    .agg(
        observations="size",
        pm_min="min",
        pm_max="max"
    )
    .reset_index()
    .sort_values("profile_id")
    .reset_index(drop=True)
)

# Create three temperature categories based
# on each profile's maximum PM temperature
profile_cv["temperature_group"] = pd.qcut(
    profile_cv["pm_max"],
    q=3,
    labels=["Low", "Medium", "High"]
)


# =====================================================
# STEP 6.1C — Construct three stratified folds
# =====================================================

skf = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

profile_cv["cv_fold"] = -1

for fold, (_, val_indices) in enumerate(
    skf.split(
        profile_cv[["pm_max"]],
        profile_cv["temperature_group"]
    ),
    start=1
):
    profile_cv.loc[val_indices, "cv_fold"] = fold

print("PASS: Three validation folds created")


# =====================================================
# STEP 6.1D — Verify profile separation
# =====================================================

development_ids = set(profile_cv["profile_id"])

assert len(development_ids) == 55
assert profile_cv["cv_fold"].isin([1, 2, 3]).all()
assert development_ids.isdisjoint(set(test_ids))

for fold in [1, 2, 3]:

    val_ids = set(
        profile_cv.loc[
            profile_cv["cv_fold"] == fold,
            "profile_id"
        ]
    )

    fold_train_ids = development_ids - val_ids

    assert val_ids.isdisjoint(fold_train_ids)
    assert val_ids.isdisjoint(set(test_ids))

    print(f"\nFold {fold}")
    print("Training profiles:", len(fold_train_ids))
    print("Validation profiles:", len(val_ids))

print("\nPASS: No profile leakage detected")

# Save assignments for reproducibility
profile_cv.to_csv(
    "profile_cv_assignments.csv",
    index=False
)

print("Saved: profile_cv_assignments.csv")


## 4.2 Verification of Cross-Validation Fold Distribution



The resulting cross-validation partitions were examined to verify their distribution across the three temperature groups and the total number of observations assigned to each fold.

**Table 4.1. Distribution of measurement profiles across temperature groups.**

| Validation fold | Low | Medium | High | Total profiles |
|:----------------|----:|-------:|-----:|---------------:|
| Fold 1 | 7 | 6 | 6 | 19 |
| Fold 2 | 6 | 5 | 7 | 18 |
| Fold 3 | 6 | 6 | 6 | 18 |
| **Total** | **19** | **17** | **19** | **55** |

The results demonstrate that each validation fold contained a reasonably balanced representation of the three maximum-temperature categories.

**Table 4.2. Validation-fold observation counts and recording durations.**

| Validation fold | Profiles | Observations | Duration (hours) |
|:----------------|---------:|-------------:|-----------------:|
| Fold 1 | 19 | 401,114 | 55.71 |
| Fold 2 | 18 | 362,087 | 50.29 |
| Fold 3 | 18 | 291,997 | 40.56 |
| **Total** | **55** | **1,055,198** | **146.56** |

Although the number of profiles was approximately balanced, the observation counts differed because individual recording sessions had different durations. This was retained to preserve complete profiles and prevent information leakage.

The verification assertions confirmed that all development observations were assigned to exactly one validation fold, with no overlap between training and validation profile IDs. The reserved final test profiles remained excluded.

During subsequent Bayesian optimisation, each proposed model configuration will be evaluated using all three folds. Its objective value will be the arithmetic mean of the three validation mean squared errors:

$$
\mathrm{MSE}_{\mathrm{CV}} = \frac{\mathrm{MSE}_1 + \mathrm{MSE}_2 + \mathrm{MSE}_3}{3}
$$

Using identical, fixed cross-validation assignments for all regression models ensures a consistent basis for comparing their proposed configurations.


In [ ]:

# =====================================================
# STEP 6.1E — Verify temperature-group distribution
# =====================================================

print("Temperature-group distribution:")

display(
    pd.crosstab(
        profile_cv["cv_fold"],
        profile_cv["temperature_group"]
    )
)


# =====================================================
# STEP 6.1F — Verify fold sizes and coverage
# =====================================================

fold_summary = (
    profile_cv.groupby("cv_fold")
    .agg(
        profiles=("profile_id", "size"),
        observations=("observations", "sum"),
        min_pm=("pm_min", "min"),
        max_pm=("pm_max", "max")
    )
)

# The original dataset is sampled at 2 Hz
fold_summary["duration_hours"] = (
    fold_summary["observations"] / 7200
)

print("\nValidation-fold summary:")
display(fold_summary)

# Confirm that every development observation is assigned
assert fold_summary["observations"].sum() == len(train_df)
assert fold_summary["profiles"].sum() == 55

print("\nPASS: All development observations assigned")


## 4.3 Preliminary Verification of Regression Model Pipelines

Before conducting computationally expensive Bayesian optimisation, preliminary regression-training pipelines were implemented to verify compatibility between the engineered temporal features, the regression algorithms and the grouped validation methodology.

Six regression models investigated by Kirchgässner et al. (2021) were considered: Ordinary Least Squares (OLS), Support Vector Regression (SVR), k-Nearest Neighbours (k-NN), Random Forest, Extra Trees and Multi-Layer Perceptron (MLP).

In [ ]:

import numpy as np
import pandas as pd

# =====================================================
# STEP 6.2A — Prepare a small development-data sample
# =====================================================

# These are example spans, not optimised values.
SMOKE_SPANS = [10, 50, 200, 1000]
SAMPLES_PER_PROFILE = 80

def make_pipeline_test_data(data, spans, samples_per_profile=80):
    """
    Create a small dataset to verify ML training pipelines.

    EWMA/EWMS is calculated using the full 2-Hz history
    of each profile before selecting training observations.
    """
    sampled_profiles = []

    for pid, profile in data.groupby(
        "profile_id", sort=False
    ):
        # Construct the 12 original input variables
        X = make_paper_inputs(profile)

        # Calculate temporal features from full history
        X_full = make_temporal_features(
            X, profile["profile_id"], spans
        )

        # Select representative positions across this profile
        n = min(samples_per_profile, len(profile))

        positions = np.linspace(
            0, len(profile) - 1, n, dtype=int
        )

        sample = X_full.iloc[positions].copy()

        # Target is stored separately from predictors
        sample["pm_target"] = (
            profile["pm"].iloc[positions].to_numpy()
        )
        sample["profile_id"] = pid

        sampled_profiles.append(sample)

    return pd.concat(
        sampled_profiles, ignore_index=True
    )


smoke_df = make_pipeline_test_data(
    train_df,
    SMOKE_SPANS,
    samples_per_profile=SAMPLES_PER_PROFILE
)

# Verification
assert smoke_df["profile_id"].nunique() == 55
assert len(smoke_df) == 55 * SAMPLES_PER_PROFILE
assert smoke_df.shape[1] == 110  # 108 + target + profile ID
assert smoke_df.isna().sum().sum() == 0
assert np.isfinite(smoke_df.to_numpy()).all()

print("PASS: Pipeline test dataset prepared")
print("Observations:", len(smoke_df))
print("Predictor features:", smoke_df.shape[1] - 2)
print("Profiles:", smoke_df["profile_id"].nunique())



### 4.3.1 Preparation of the Preliminary Training Dataset

The complete development dataset contained more than one million observations. To reduce computational requirements during initial implementation testing, a smaller dataset was constructed by selecting 80 observations distributed across each of the 55 development profiles.

Importantly, the EWMA/EWMS temporal features were calculated using each profile's complete original 2 Hz measurement history before selecting the observations. This ensured that the selected features retained information from the preceding measurements rather than being calculated from an artificially shortened history.

Four illustrative spans of 10, 50, 200 and 1,000 samples were used, consistent with the earlier feature-engineering verification. These were fixed demonstration settings, not optimised hyperparameters.

**Table 4.3. Preliminary regression dataset characteristics.**

| Characteristic | Result |
|:---------------|-------:|
| Development profiles represented | 55 |
| Selected observations per profile | 80 |
| Total observations | 4,400 |
| Predictor variables | 108 |
| Prediction target | Permanent-magnet temperature (`pm`) |
| Missing feature values | 0 |

The resulting dataset passed the dimensional and numerical-validity checks and was used exclusively for preliminary model-pipeline verification.


In [ ]:

import time
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor
)
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# =====================================================
# STEP 6.2B — Select the first grouped CV fold
# =====================================================

val_ids = set(
    profile_cv.loc[
        profile_cv["cv_fold"] == 1,
        "profile_id"
    ]
)

train_sample = smoke_df[
    ~smoke_df["profile_id"].isin(val_ids)
].copy()

val_sample = smoke_df[
    smoke_df["profile_id"].isin(val_ids)
].copy()

# Separate inputs and target
feature_cols = [
    c for c in smoke_df.columns
    if c not in ["pm_target", "profile_id"]
]

X_train = train_sample[feature_cols]
y_train = train_sample["pm_target"]

X_val = val_sample[feature_cols]
y_val = val_sample["pm_target"]

# Verify no profile overlap
assert set(train_sample["profile_id"]).isdisjoint(
    set(val_sample["profile_id"])
)

assert len(feature_cols) == 108

print("Training profiles:",
      train_sample["profile_id"].nunique())
print("Validation profiles:",
      val_sample["profile_id"].nunique())


# =====================================================
# STEP 6.2C — Define the six regression models
# =====================================================

models = {
    "OLS": LinearRegression(),

    "SVR": SVR(
        kernel="rbf",
        C=1.0,
        epsilon=0.1
    ),

    "k-NN": KNeighborsRegressor(
        n_neighbors=10,
        weights="distance"
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=20,
        max_depth=12,
        random_state=42,
        n_jobs=2
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=20,
        max_depth=12,
        random_state=42,
        n_jobs=2
    ),

    "MLP": MLPRegressor(
        hidden_layer_sizes=(32,),
        max_iter=150,
        random_state=42
    )
}


# =====================================================
# STEP 6.2D — Train, predict and evaluate each model
# =====================================================

results = []

for name, estimator in models.items():

    # StandardScaler is fitted on TRAINING data only.
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", estimator)
    ])

    # Train
    start = time.perf_counter()
    pipeline.fit(X_train, y_train)
    training_time = time.perf_counter() - start

    # Predict on unseen validation profiles
    start = time.perf_counter()
    predictions = pipeline.predict(X_val)
    inference_time = time.perf_counter() - start

    # Calculate prediction errors
    errors = y_val.to_numpy() - predictions

    results.append({
        "Model": name,
        "MSE": mean_squared_error(y_val, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_val, predictions)),
        "MAE": mean_absolute_error(y_val, predictions),
        "R2": r2_score(y_val, predictions),
        "Max Error": np.max(np.abs(errors)),
        "Training Time (s)": training_time,
        "Inference Time (s)": inference_time
    })

    print(f"PASS: {name} trained and evaluated")

# Display comparison
results_df = pd.DataFrame(results)

display(results_df.round(4))

# Save preliminary results
results_df.to_csv(
    "model_pipeline_smoke_test.csv",
    index=False
)

print("\nPASS: All six regression pipelines completed")



## 4.4 Preliminary OLS Error Investigation

Inspection of the initial model-comparison results identified an unusually large maximum absolute prediction error for OLS. A dedicated diagnostic procedure was therefore implemented to determine which observations contributed most strongly to its validation error.

The OLS pipeline was retrained using the same preliminary training and validation partition. A diagnostic table was constructed containing the measurement profile ID, measured permanent-magnet temperature, predicted temperature and absolute prediction error. The observations were then ranked by descending absolute error.

The largest prediction error was identified in measurement profile 65.

**Table 4.4. Largest OLS prediction error.**

| Quantity | Value |
|:---------|------:|
| Measurement profile | 65 |
| Actual permanent-magnet temperature | 26.04 °C |
| OLS-predicted temperature | 239.77 °C |
| Absolute prediction error | 213.73 °C |

The next-largest absolute error was approximately 38.19 °C, considerably smaller than the error observed in profile 65.

This result indicated that the maximum error was dominated by an exceptionally poor prediction rather than being representative of the model's general prediction accuracy. However, the initial diagnostic could not establish whether the discrepancy originated from a measurement anomaly, unusual predictor values or instability in the fitted regression model.

Consequently, further investigation was conducted before considering any modification to the dataset.


In [ ]:

# STEP 6.2E — Investigate OLS prediction outliers

ols_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

ols_pipeline.fit(X_train, y_train)

ols_predictions = ols_pipeline.predict(X_val)

ols_diagnostics = pd.DataFrame({
    "profile_id": val_sample["profile_id"].to_numpy(),
    "actual_pm": y_val.to_numpy(),
    "predicted_pm": ols_predictions
})

ols_diagnostics["absolute_error"] = abs(
    ols_diagnostics["actual_pm"]
    - ols_diagnostics["predicted_pm"]
)

print("Ten largest OLS prediction errors:")

display(
    ols_diagnostics.nlargest(
        10, "absolute_error"
    )
)

print(
    "\nMeasured PM temperature range:",
    y_val.min(), "to", y_val.max()
)

print(
    "Predicted PM temperature range:",
    ols_predictions.min(),
    "to",
    ols_predictions.max()
)

print(
    "Maximum absolute error:",
    ols_diagnostics["absolute_error"].max()
)



## 4.5 Investigation of Profile-Start Prediction Errors

A subsequent diagnostic examined whether the extreme OLS prediction occurred near the beginning of a measurement profile, where the exponentially weighted temporal features contain limited historical information.

Each observation in the validation diagnostic table was assigned a position within its corresponding profile. The largest prediction errors were then examined alongside their relative sample positions.

The investigation confirmed that the 213.73 °C prediction error occurred at position zero—the first recorded observation of profile 65.

At the first measurement of a profile, the exponentially weighted moving average is equal to the initial sensor measurement, while the exponentially weighted standard deviation is zero under the implemented biased weighted-variance convention. Consequently, the temporal features are initially less informative about the motor's preceding operating conditions.

The first five selected observations of profile 65 were inspected to examine how prediction error changed as additional measurement history became available.

**Table 4.5. Initial sampled observations of profile 65.**

| Sample position | Actual PM (°C) | Predicted PM (°C) | Absolute error (°C) |
|:----------------|---------------:|------------------:|--------------------:|
| 0 | 26.04 | 239.77 | 213.73 |
| 1 | 35.64 | 49.01 | 13.36 |
| 2 | 40.72 | 47.19 | 6.46 |
| 3 | 53.52 | 62.60 | 9.08 |
| 4 | 61.69 | 67.20 | 5.51 |

The error decreased substantially after the initial sampled observation, suggesting that the extreme prediction might be associated with a profile-start or cold-start condition.

To investigate whether this pattern extended across the validation set, the mean absolute errors at the first sampled observations of all profiles were compared with those at subsequent sampled observations.

**Table 4.6. Profile-start versus subsequent prediction errors.**

| Observation category | Mean absolute error |
|:---------------------|--------------------:|
| First sampled observations | 18.52 °C |
| Subsequent sampled observations | 6.15 °C |
| First observations excluding profile 65's extreme prediction, for diagnosis only | 7.67 °C |

Although the initial comparison suggested substantially higher errors at profile starts, the sensitivity comparison demonstrated that the extreme profile-65 prediction strongly influenced this result.

Therefore, the findings support investigating EWMA/EWMS initialisation as a possible contributing factor, but do not establish it as the sole cause of the prediction error. Highly correlated regression inputs, unusual operating conditions and coefficient instability remain possible explanations.


In [ ]:

# STEP 6.2F — Investigate OLS errors at profile starts

# Identify the position of each sampled observation
# within its own validation profile.
ols_diagnostics["position_in_profile"] = (
    ols_diagnostics.groupby("profile_id").cumcount()
)

# Inspect the largest prediction errors
display(
    ols_diagnostics.nlargest(
        10, "absolute_error"
    )
)

# Inspect the first five sampled observations
# from the problematic profile
print("\nFirst five sampled observations of profile 65:")

display(
    ols_diagnostics[
        ols_diagnostics["profile_id"] == 65
    ].head(5)
)

# Compare errors at the first sample versus later samples
first_samples = ols_diagnostics[
    ols_diagnostics["position_in_profile"] == 0
]

later_samples = ols_diagnostics[
    ols_diagnostics["position_in_profile"] > 0
]

print("\nMean absolute error at profile starts:",
      first_samples["absolute_error"].mean())

print("Mean absolute error at later observations:",
      later_samples["absolute_error"].mean())



## 4.6 OLS Outlier Influence and Sensitivity Analysis

A further diagnostic was implemented to quantify the influence of the extreme prediction on the overall validation mean squared error.

The squared error of each OLS prediction was calculated, and the largest squared error was expressed as a percentage of the total squared validation error.

The first raw measurements of profile 65 were also inspected. The initial recorded permanent-magnet temperature was approximately 26.04 °C, with coolant temperature around 24.57 °C, ambient temperature around 25.16 °C and motor speed around 3,029 rpm.

The resulting influence analysis is summarised below.

**Table 4.7. Sensitivity of preliminary OLS validation error to the extreme prediction.**

| Diagnostic measure | Result |
|:-------------------|-------:|
| Validation observations | 1,520 |
| Largest absolute prediction error | 213.73 °C |
| Contribution to total squared error | 33.9% |
| MSE using all validation observations | 88.66 °C² |
| MSE excluding the largest error for diagnosis only | 58.64 °C² |

A single observation therefore accounted for approximately one-third of the total squared validation error.

The reduced MSE was calculated only to quantify the influence of the extreme prediction. It was not adopted as a replacement performance metric, and the observation was not removed from the original dataset, the validation partition or the subsequent model-development procedure.

The result demonstrates that aggregate MSE can be highly sensitive to isolated extreme predictions and supports the use of additional metrics, such as MAE and maximum absolute error, when evaluating the temperature estimators.

#### Treatment Decision

No evidence was established that the original temperature measurement was invalid. The anomaly was identified in the model's prediction error, rather than confirmed as a defective data point.

Therefore, the observation in profile 65 was retained without modification. Removing difficult-to-predict measurements solely to improve validation accuracy would undermine the credibility of the performance comparison.

The potential causes of the extreme prediction remain open for further investigation, particularly the effect of temporal-feature initialisation and the sensitivity of the OLS coefficients.


In [ ]:

# STEP 6.2G — Investigate the extreme prediction

# Examine raw measurements at the start of profile 65
print("First five raw measurements of profile 65:")

display(
    df.loc[df["profile_id"] == 65].head(5)
)

# Quantify influence of the largest error
squared_errors = (
    ols_diagnostics["actual_pm"]
    - ols_diagnostics["predicted_pm"]
) ** 2

worst_idx = squared_errors.idxmax()

outlier_contribution = (
    squared_errors.loc[worst_idx]
    / squared_errors.sum()
    * 100
)

print(
    "\nPercentage of total OLS squared error "
    "from the single largest prediction:",
    round(outlier_contribution, 2), "%"
)

# Diagnostic sensitivity calculation only
print(
    "\nValidation MSE with all observations:",
    squared_errors.mean()
)

print(
    "Validation MSE excluding largest error:",
    squared_errors.drop(worst_idx).mean()
)


## 4.7 Preliminary Verification Conclusion and Transition to Bayesian Optimisation

The preliminary implementation successfully established stratified grouped three-fold cross-validation and functioning training and evaluation pipelines for all six regression models.

The diagnostic investigation also identified an extreme OLS prediction error at the beginning of measurement profile 65. Although its location suggests potential sensitivity to the initialisation of temporal features, the cause has not yet been conclusively determined. The observation was therefore retained to preserve the original dataset and provide an unbiased basis for subsequent evaluation.

The next stage will implement Bayesian hyperparameter optimisation using the three established validation folds. This process will search for suitable EWMA/EWMS spans and model-specific hyperparameters by minimising average cross-validation MSE.

In addition to evaluating overall prediction accuracy, the optimised OLS model will be examined to determine whether its performance on profile 65 improves relative to the preliminary configuration. This comparison will be treated as exploratory and interpreted with regard to differences in training-data size and model configuration. Improved average cross-validation accuracy does not necessarily guarantee that the specific extreme prediction will improve.

Following optimisation, the selected models will be retrained using all development profiles and evaluated on the untouched final test set to establish the 2 Hz benchmark for the subsequent sampling-frequency investigation.

## 4.8 Bayesian Hyperparameter Optimisation

### 4.8.1 Initial Bayesian Optimisation Pilot

Gaussian-process Bayesian optimisation searches the four EWMA/EWMS history spans with OLS as an inexpensive reference estimator. The objective is the arithmetic mean of the three profile-disjoint validation-fold mean squared errors:

$$\mathrm{MSE}_{\mathrm{CV}}=\tfrac13\sum_{f=1}^3\mathrm{MSE}_f.$$

For the initial pilot, each of 55 development profiles contributes 80 observations, selected only **after** causal EWMA/EWMS computation over the complete 2-Hz history. Standardisation is fitted inside each training fold. The `gp_minimize` optimiser uses a Gaussian-process surrogate, expected improvement acquisition and eight evaluations. The manually chosen initial configuration `(10, 50, 200, 1000)` is a fixed **control input**, not a copied optimisation winner. Numerical scores and the selected spans must come from this execution's printed results and exported trial CSV, not from the archived run.

In [ ]:

# STEP 6.3A — Bayesian optimisation setup

%pip -q install scikit-optimize

import time
import numpy as np
import pandas as pd

from skopt import gp_minimize
from skopt.space import Integer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Preliminary optimisation settings
SAMPLES_PER_PROFILE = 80
RANDOM_STATE = 42

# Use the exact CV assignments from Step 6.1
fold_lookup = profile_cv.set_index(
    "profile_id"
)["cv_fold"]

assert len(train_df) == 1055198
assert len(fold_lookup) == 55
assert set(fold_lookup.unique()) == {1, 2, 3}


def build_candidate_features(spans):
    """
    Construct features for candidate EWMA/EWMS spans.

    Full original 2-Hz histories are processed before
    selecting 80 observations per development profile.
    """
    feature_arrays = []
    target_arrays = []
    profile_arrays = []

    for pid, profile in train_df.groupby(
        "profile_id", sort=False
    ):

        X_original = make_paper_inputs(profile)

        X_temporal = make_temporal_features(
            X_original,
            profile["profile_id"],
            spans
        )

        n = min(SAMPLES_PER_PROFILE, len(profile))

        positions = np.linspace(
            0, len(profile) - 1, n, dtype=int
        )

        feature_arrays.append(
            X_temporal.iloc[positions].to_numpy(
                dtype=np.float64
            )
        )

        target_arrays.append(
            profile["pm"].iloc[positions].to_numpy()
        )

        profile_arrays.append(
            np.full(n, pid)
        )

    X = np.vstack(feature_arrays)
    y = np.concatenate(target_arrays)
    profile_ids = np.concatenate(profile_arrays)

    assert X.shape[1] == 108
    assert len(X) == 4400
    assert np.isfinite(X).all()

    return X, y, profile_ids


def evaluate_ols_cv(spans):
    """
    Evaluate a four-span configuration with the same
    fixed, grouped, stratified 3-fold CV.

    Return average CV MSE and individual fold scores.
    """
    X, y, profile_ids = build_candidate_features(spans)

    fold_ids = (
        pd.Series(profile_ids)
        .map(fold_lookup)
        .to_numpy(dtype=int)
    )

    fold_mse = []

    for fold in [1, 2, 3]:

        train_mask = fold_ids != fold
        val_mask = fold_ids == fold

        assert train_mask.sum() > 0
        assert val_mask.sum() > 0
        assert set(profile_ids[train_mask]).isdisjoint(
            set(profile_ids[val_mask])
        )

        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("model", LinearRegression())
        ])

        pipeline.fit(
            X[train_mask],
            y[train_mask]
        )

        predictions = pipeline.predict(
            X[val_mask]
        )

        mse = mean_squared_error(
            y[val_mask],
            predictions
        )

        fold_mse.append(float(mse))

    return float(np.mean(fold_mse)), fold_mse


print("PASS: Bayesian optimisation objective defined")


### 4.8.2 Review and Expansion of Search Intervals (Step 6.3B)

Reviewing the pilot revealed that several best-performing span values approached or reached the initial upper search boundaries. The search was therefore expanded to permit longer temporal-memory representations. To guarantee increasing spans, the four original span values were reparameterised in terms of an initial span and three positive increments:

$$
\begin{aligned}
s_1&=a,\\
s_2&=a+d_2,\\
s_3&=a+d_2+d_3,\\
s_4&=a+d_2+d_3+d_4.
\end{aligned}
$$

The expanded parameter intervals were $a\in[5,100]$, $d_2\in[1,400]$, $d_3\in[1,1500]$, and $d_4\in[1,6000]$. These constraints enforce $s_1<s_2<s_3<s_4$ while allowing the optimizer to explore a wider range of historical smoothing scales. The eight initial observations, including their measured CV errors, were supplied to the newly parameterised optimiser. Twenty additional configurations were then evaluated, producing a cumulative search of **28 trials**. In the current Run-All notebook, those trials are generated during the active execution rather than restored from an earlier session.

In [ ]:

# STEP 6.3B — Verify Bayesian objective function

baseline_spans = [10, 50, 200, 1000]

baseline_mse, baseline_folds = evaluate_ols_cv(
    baseline_spans
)

print("Example spans:", baseline_spans)

for fold, mse in enumerate(baseline_folds, start=1):
    print(f"Fold {fold} MSE: {mse:.4f} °C²")

print(f"\nAverage CV MSE: {baseline_mse:.4f} °C²")

# Verify against the previously observed result
np.testing.assert_allclose(
    baseline_mse,
    64.6399,
    rtol=1e-3
)

print("PASS: OLS CV objective reproduces baseline")


In [ ]:

# STEP 6.3C — Gaussian-process Bayesian optimisation

search_space = [
    Integer(5, 30, name="span_1"),
    Integer(31, 150, name="span_2"),
    Integer(151, 700, name="span_3"),
    Integer(701, 3000, name="span_4")
]

trial_records = []


def bayesian_objective(candidate):
    """
    Objective minimised by Gaussian-process
    Bayesian optimisation.
    """
    spans = [int(s) for s in candidate]

    start = time.perf_counter()

    mean_mse, fold_scores = evaluate_ols_cv(spans)

    duration = time.perf_counter() - start

    record = {
        "trial": len(trial_records) + 1,
        "s1": spans[0],
        "s2": spans[1],
        "s3": spans[2],
        "s4": spans[3],
        "fold1_mse": fold_scores[0],
        "fold2_mse": fold_scores[1],
        "fold3_mse": fold_scores[2],
        "mean_cv_mse": mean_mse,
        "runtime_s": duration
    }

    trial_records.append(record)

    print(
        f"Trial {record['trial']}: "
        f"spans={spans}, "
        f"CV MSE={mean_mse:.4f}, "
        f"time={duration:.1f}s",
        flush=True
    )

    return mean_mse


# Small initial search to verify optimisation
result = gp_minimize(
    func=bayesian_objective,
    dimensions=search_space,
    n_calls=8,
    n_initial_points=4,
    acq_func="EI",
    random_state=RANDOM_STATE
)

pd.DataFrame(trial_records).to_csv(
    "bayesian_ols_pilot_trials.csv", index=False
)
print("\nPASS: OLS Bayesian optimisation pilot completed")
print("Best spans:", result.x)
print(f"Best CV MSE: {result.fun:.4f} °C²")

display(
    pd.DataFrame(trial_records)
    .sort_values("mean_cv_mse")
    .round(4)
)


In [ ]:

# STEP 6.3D — Visualise Bayesian search progress

import matplotlib.pyplot as plt

history = pd.DataFrame(trial_records)

history["best_so_far"] = (
    history["mean_cv_mse"].cummin()
)

plt.figure(figsize=(10, 5))

plt.plot(
    history["trial"],
    history["mean_cv_mse"],
    "o-",
    label="Candidate CV MSE"
)

plt.plot(
    history["trial"],
    history["best_so_far"],
    "s-",
    label="Best MSE so far"
)

plt.axhline(
    baseline_mse,
    linestyle="--",
    label="Fixed-span baseline"
)

plt.xlabel("Bayesian optimisation trial")
plt.ylabel("Mean 3-fold CV MSE (°C²)")
plt.title("OLS Bayesian optimisation — Pilot")
plt.legend()
plt.grid(True)
plt.show()


### 4.8.2 Review and Expansion of Search Intervals (Step 6.3B)

Inspection of the initial optimisation results revealed that three of the four selected spans reached their respective upper search limits. This suggested that the initial search intervals might constrain the exploration of longer temporal-memory representations.

Consequently, the search space was expanded and reparameterised using the first span and three positive differences between consecutive spans:

$$
\begin{aligned}
s_1 &= a \\
s_2 &= a+d_2 \\
s_3 &= a+d_2+d_3 \\
s_4 &= a+d_2+d_3+d_4
\end{aligned}
$$

This parameterisation guaranteed strictly increasing span values while permitting a wider range of smoothing timescales.

The expanded intervals were 5–100 for the first span, 1–400 for the second-span increment, 1–1,500 for the third-span increment and 1–6,000 for the fourth-span increment.

The original eight evaluated configurations were reused by supplying their recorded objective values to the newly parameterised Gaussian-process optimiser. Twenty additional configurations were then evaluated, producing 28 evaluations in total. Results were checkpointed after each completed trial.

In [ ]:

# STEP 6.3B.1 — Expanded Bayesian search setup

import os
import time
import numpy as np
import pandas as pd

from skopt import Optimizer
from skopt.space import Integer

PILOT_PATH = "bayesian_ols_pilot_trials.csv"
EXPANDED_PATH = "bayesian_ols_expanded_trials.csv"

# New search space: first span + three positive gaps
expanded_space = [
    Integer(5, 100, name="s1"),
    Integer(1, 400, name="gap_2"),
    Integer(1, 1500, name="gap_3"),
    Integer(1, 6000, name="gap_4")
]

def decode_spans(parameters):
    """Convert first span and gaps into four ordered spans."""
    a, d2, d3, d4 = map(int, parameters)

    return [
        a,
        a + d2,
        a + d2 + d3,
        a + d2 + d3 + d4
    ]


def encode_spans(spans):
    """Convert four ordered spans into the new search space."""
    a, b, c, d = map(int, spans)

    if not a < b < c < d:
        raise ValueError("Spans must be strictly increasing")

    return [a, b-a, c-b, d-c]


# Previous eight trials are generated earlier in THIS Run All execution.
# Deliberately ignore files from old runtimes.
pilot_df = pd.DataFrame(trial_records).copy()
pilot_df.to_csv(PILOT_PATH, index=False)

assert len(pilot_df) == 8

# Confirm all old trials fit in the expanded space
for _, row in pilot_df.iterrows():
    old_spans = [
        int(row["s1"]), int(row["s2"]),
        int(row["s3"]), int(row["s4"])
    ]

    params = encode_spans(old_spans)

    assert all(
        dimension.low <= value <= dimension.high
        for dimension, value in zip(expanded_space, params)
    )

print("PASS: All eight previous trials are reusable")

print("\nPrevious best configuration:")
best_old = pilot_df.loc[
    pilot_df["mean_cv_mse"].idxmin()
]

print("Spans:", [
    int(best_old[f"s{i}"]) for i in range(1, 5)
])
print("CV MSE:", best_old["mean_cv_mse"])


In [ ]:

# STEP 6.3B.2 — Continue Bayesian optimisation

N_ADDITIONAL_TRIALS = 20

# Always perform a fresh expanded search after the freshly generated pilot.
expanded_df = pd.DataFrame(columns=pilot_df.columns)

# Reconstruct the optimiser from recorded results
optimizer = Optimizer(
    dimensions=expanded_space,
    base_estimator="GP",
    n_initial_points=10,
    acq_func="EI",
    random_state=42
)

existing_history = pd.concat(
    [pilot_df, expanded_df],
    ignore_index=True
)

# Reuse all completed evaluations
for _, row in existing_history.iterrows():

    spans = [
        int(row["s1"]), int(row["s2"]),
        int(row["s3"]), int(row["s4"])
    ]

    parameters = encode_spans(spans)

    optimizer.tell(
        parameters,
        float(row["mean_cv_mse"])
    )

print("Existing evaluations:", len(existing_history))

remaining = max(
    0,
    N_ADDITIONAL_TRIALS - len(expanded_df)
)

print("Additional evaluations remaining:", remaining)

# Search for new configurations
for _ in range(remaining):

    parameters = optimizer.ask()
    spans = decode_spans(parameters)

    start = time.perf_counter()

    mean_mse, fold_scores = evaluate_ols_cv(spans)

    duration = time.perf_counter() - start

    optimizer.tell(parameters, mean_mse)

    record = {
        "trial": len(pilot_df) + len(expanded_df) + 1,
        "s1": spans[0],
        "s2": spans[1],
        "s3": spans[2],
        "s4": spans[3],
        "fold1_mse": fold_scores[0],
        "fold2_mse": fold_scores[1],
        "fold3_mse": fold_scores[2],
        "mean_cv_mse": mean_mse,
        "runtime_s": duration
    }

    expanded_df = pd.concat(
        [expanded_df, pd.DataFrame([record])],
        ignore_index=True
    )

    print(
        f"Trial {record['trial']}: "
        f"spans={spans}, "
        f"MSE={mean_mse:.4f}, "
        f"time={duration:.1f}s",
        flush=True
    )

expanded_df.to_csv(EXPANDED_PATH, index=False)
print("\nPASS: Expanded Bayesian pilot completed")

# Combine original and new results
all_trials = pd.concat(
    [pilot_df, expanded_df],
    ignore_index=True
)

best_result = all_trials.loc[
    all_trials["mean_cv_mse"].idxmin()
]

best_spans = [
    int(best_result[f"s{i}"]) for i in range(1, 5)
]

print("\nBest spans:", best_spans)
print(
    "Best 3-fold CV MSE:",
    round(float(best_result["mean_cv_mse"]), 4)
)

display(
    all_trials.sort_values("mean_cv_mse")
    .head(10)
    .round(4)
)


In [ ]:

# STEP 6.3B.3 — Compare original and expanded search

import matplotlib.pyplot as plt

history = all_trials.copy()

history["best_so_far"] = (
    history["mean_cv_mse"].cummin()
)

plt.figure(figsize=(11, 5))

plt.plot(
    history["trial"],
    history["mean_cv_mse"],
    "o-",
    label="Candidate CV MSE",
    alpha=0.7
)

plt.plot(
    history["trial"],
    history["best_so_far"],
    "s-",
    label="Best MSE so far"
)

# Original manually selected span configuration
plt.axhline(
    64.6399,
    linestyle="--",
    label="Fixed-span baseline"
)

# Mark where the expanded search begins
plt.axvline(
    8.5,
    linestyle=":",
    color="grey",
    label="Expanded search begins"
)

plt.xlabel("Bayesian optimisation trial")
plt.ylabel("Mean 3-fold CV MSE (°C²)")
plt.title("OLS Bayesian Optimisation — Expanded Pilot")
plt.legend()
plt.grid(True)
plt.show()

print("Original pilot best MSE:",
      round(pilot_df["mean_cv_mse"].min(), 4))

print("Expanded search best MSE:",
      round(history["mean_cv_mse"].min(), 4))

print("Total evaluations:", len(history))


> **Archived interpretation only.** Numbers in the following source paragraph/table were reported by an earlier run; compare against the freshly executed cells and saved trial results. This is not a new experimental result.

### 4.8.3 Results of the Expanded OLS Optimisation

The lowest MSE in the expanded search occurred at **Trial 14**.

| Parameter | Initial eight-trial best | Expanded 28-trial best |
|---|---:|---:|
| Span 1 | 30 | 89 |
| Span 2 | 150 | 265 |
| Span 3 | 652 | 622 |
| Span 4 | 3000 | 3829 |
| Mean three-fold CV MSE (°C²) | 45.6467 | **45.1303** |

The expanded optimum produced fold-specific MSEs of approximately **64.9351, 36.5415 and 33.9143 °C²**. It reduced MSE by another **1.13%** relative to the eight-trial optimum and by **30.18%** relative to the original manually selected spans. The optimisation-history plot showed substantial initial improvement followed by smaller gains in the expanded stage. These are observed improvements within a limited, reduced-data search; they do not demonstrate a global optimum.

### 4.8.4 Interpretation and Limitations of the OLS Pilot

The OLS experiments showed that the choice of EWMA/EWMS spans materially affects the ability of a regression estimator to reproduce permanent-magnet temperature. Expanding search intervals was justified because an optimum at a search boundary may indicate that the candidate space is too restrictive. However, the pilot evaluated a selected subset of observations rather than all original measurements. Its objective values should not be directly compared with full-data validation scores as if the evaluation populations were identical.

The earlier preliminary OLS diagnostic also identified a severe prediction error near the beginning of profile 65. The span-search results alone do not establish that the anomaly was resolved: that would require a matched, profile-specific comparison using the optimised estimator. The 14 final test profiles remained excluded from optimisation throughout this stage.

### 4.8.5 Conclusion

The OLS pilot and 20 additional Gaussian-process evaluations establish a reproducible development-only search for causal smoothing spans. Interpret the **new** per-fold MSE, best-span configuration, runtime and convergence evidence after the cells execute. Do not substitute the earlier archived outcome; eight initial plus twenty expanded evaluations do not establish global optimality.

### 4.8.3 Bayesian Optimisation of Alternative Regression Models

### Original regression models: mathematical and algorithmic explanation

The original Bayesian optimiser (source cells 60–63) jointly searches **four chronological smoothing spans and model-specific parameters**. Later, the larger-sample searches reuse the same model factories but a different *historically specified* per-profile training budget. These explanations map directly to the original code without modifying it.

**Ordinary least squares (OLS).** Given the design matrix with intercept \(\tilde X=[1,X]\), the coefficients minimise \(\|y-\tilde X\beta\|_2^2\). `LinearRegression` is the pilot reference; the original full-development implementation uses QR compression of profile-wise augmented matrices to evaluate the same least-squares objective without retaining a full million-row feature matrix. Four EWMA/EWMS spans are selected by a Gaussian-process Bayesian search.

**k-nearest neighbours (k-NN).** The prediction for an input \(x\) aggregates targets from its \(k\) closest standardised training vectors: \(\hat y(x)=\sum_{i\in\mathcal N_k(x)}w_i y_i/\sum_{i\in\mathcal N_k(x)}w_i\). The original integer domain is \(1\le k\le512\), with uniform or distance weighting. Cheap fitting does not imply cheap full-data inference.

**Random Forest.** A forest averages \(B\) regression trees, \(\hat y(x)=B^{-1}\sum_{b=1}^{B}T_b(x)\). Each tree recursively partitions the feature space, while the ensemble reduces variance. The historical search independently varies tree count, depth, minimum split, minimum leaf size and bootstrap on/off; split-level feature subsampling is fixed at `max_features='sqrt'`.

**Extra Trees.** Extremely Randomized Trees also averages tree predictions, but randomises split thresholds more aggressively. The original search explores the same five tree controls; any comparison therefore needs to consider accuracy, training time and differences in randomness together.

**Multilayer perceptron (MLP).** A network composes affine maps and nonlinearities, \(h_{\ell+1}=\operatorname{ReLU}(W_\ell h_\ell+b_\ell)\), followed by a scalar regression output. The Bayesian variables are layer count, number of units, L2 coefficient \(\alpha\) and initial Adam learning rate. Original larger-budget validation additionally enables early stopping *within each training fold*, an explicit historical stage-specific decision.

**Exact RBF support vector regression (SVR).** The nonlinear kernel \(K(x,z)=\exp[-\gamma\|x-z\|_2^2]\) is used with epsilon-insensitive regression loss. The original search varies \(C\), \(\epsilon\) and the temporal spans while retaining `gamma='scale'`. Kernel computation and fit cost increase strongly with training-set size.

**Grouped validation objective.** For each candidate, smoothing is computed causally along the full 2 Hz history of each complete profile, with deterministic positions sampled afterward. Fold-specific scalers are fitted exclusively on training profiles. The optimiser minimises arithmetic mean held-out-group MSE, \(J(\theta)=\frac13\sum_{f=1}^3\operatorname{MSE}_f(\theta)\), in °C². There is no new penalty or substitute objective.

In [ ]:

# STEP 6.3C.1 — Search spaces for five regression models

import os
import json
import time
import warnings
import numpy as np
import pandas as pd

from skopt import Optimizer
from skopt.space import Integer, Real, Categorical

from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor
)
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning


# Same expanded span search space as our OLS pilot
SPAN_SPACE = [
    Integer(5, 100, name="s1"),
    Integer(1, 400, name="gap_2"),
    Integer(1, 1500, name="gap_3"),
    Integer(1, 6000, name="gap_4")
]


# Model-specific hyperparameter spaces
MODEL_SPACES = {

    "SVR": [
        Real(1e-3, 10, prior="log-uniform", name="C"),
        Real(1e-2, 0.3, prior="log-uniform", name="epsilon")
    ],

    "k-NN": [
        Integer(1, 512, name="n_neighbors"),
        Categorical(
            ["uniform", "distance"],
            name="weights"
        )
    ],

    "Random Forest": [
        Integer(20, 100, name="n_estimators"),
        Integer(5, 30, name="max_depth"),
        Integer(2, 20, name="min_samples_split"),
        Integer(1, 10, name="min_samples_leaf"),
        Categorical(["yes", "no"], name="bootstrap")
    ],

    "Extra Trees": [
        Integer(20, 100, name="n_estimators"),
        Integer(5, 30, name="max_depth"),
        Integer(2, 20, name="min_samples_split"),
        Integer(1, 10, name="min_samples_leaf"),
        Categorical(["yes", "no"], name="bootstrap")
    ],

    "MLP": [
        Integer(1, 3, name="n_layers"),
        Integer(4, 32, name="n_units"),
        Real(1e-7, 1e-2, prior="log-uniform", name="alpha"),
        Real(
            1e-4, 1e-2,
            prior="log-uniform",
            name="learning_rate_init"
        )
    ]
}


def decode_spans(parameters):
    """Convert first span and positive gaps to four spans."""
    a, d2, d3, d4 = map(int, parameters)

    return [
        a,
        a + d2,
        a + d2 + d3,
        a + d2 + d3 + d4
    ]


# Confirm that our earlier functions and CV data exist
assert callable(build_candidate_features)
assert len(profile_cv) == 55
assert len(train_df) == 1055198

assert set(profile_cv["profile_id"]) == set(
    train_df["profile_id"]
)

assert set(profile_cv["cv_fold"]) == {1, 2, 3}

print("PASS: Five model search spaces defined")
print("Models:", list(MODEL_SPACES))


In [ ]:

# STEP 6.3C.2 — Regression models and CV objective

def make_regressor(model_name, hp):
    """Construct a regression model from its hyperparameters."""

    if model_name == "SVR":
        return SVR(
            kernel="rbf",
            C=float(hp["C"]),
            epsilon=float(hp["epsilon"]),
            gamma="scale"
        )

    if model_name == "k-NN":
        return KNeighborsRegressor(
            n_neighbors=int(hp["n_neighbors"]),
            weights=hp["weights"],
            n_jobs=2
        )

    if model_name in ["Random Forest", "Extra Trees"]:

        estimator_class = (
            RandomForestRegressor
            if model_name == "Random Forest"
            else ExtraTreesRegressor
        )

        return estimator_class(
            n_estimators=int(hp["n_estimators"]),
            max_depth=int(hp["max_depth"]),
            min_samples_split=int(hp["min_samples_split"]),
            min_samples_leaf=int(hp["min_samples_leaf"]),
            bootstrap=(hp["bootstrap"] == "yes"),
            max_features="sqrt",
            random_state=42,
            n_jobs=2
        )

    if model_name == "MLP":

        n_layers = int(hp["n_layers"])
        n_units = int(hp["n_units"])

        return MLPRegressor(
            hidden_layer_sizes=(n_units,) * n_layers,
            activation="relu",
            solver="adam",
            alpha=float(hp["alpha"]),
            learning_rate_init=float(
                hp["learning_rate_init"]
            ),
            batch_size=64,
            max_iter=200,
            random_state=42,
            early_stopping=False
        )

    raise ValueError(f"Unknown model: {model_name}")


def evaluate_model_cv(model_name, parameters):
    """
    Evaluate model hyperparameters on the fixed
    stratified grouped 3-fold cross-validation split.
    """
    spans = decode_spans(parameters[:4])

    # Translate candidate values into named hyperparameters
    hp = {
        dimension.name: value
        for dimension, value in zip(
            MODEL_SPACES[model_name],
            parameters[4:]
        )
    }

    # Recalculate EWMA/EWMS using complete profile histories,
    # then select the same 80 observations per profile.
    X, y, profile_ids = build_candidate_features(spans)

    fold_lookup = profile_cv.set_index(
        "profile_id"
    )["cv_fold"]

    fold_ids = (
        pd.Series(profile_ids)
        .map(fold_lookup)
        .to_numpy()
    )

    assert not pd.isna(fold_ids).any()

    fold_scores = []
    convergence_warnings = 0

    for fold in [1, 2, 3]:

        training_mask = fold_ids != fold
        validation_mask = fold_ids == fold

        assert set(
            profile_ids[training_mask]
        ).isdisjoint(
            set(profile_ids[validation_mask])
        )

        estimator = make_regressor(model_name, hp)

        # Feature scaling on training fold only
        feature_pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("model", estimator)
        ])

        # Target scaling also fitted on training fold only
        pipeline = TransformedTargetRegressor(
            regressor=feature_pipeline,
            transformer=StandardScaler()
        )

        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)

            pipeline.fit(
                X[training_mask],
                y[training_mask]
            )

        convergence_warnings += sum(
            issubclass(w.category, ConvergenceWarning)
            for w in caught
        )

        # Predictions are returned in the original °C units
        predictions = pipeline.predict(
            X[validation_mask]
        )

        if not np.isfinite(predictions).all():
            raise ValueError(
                f"{model_name} produced invalid predictions"
            )

        mse = mean_squared_error(
            y[validation_mask],
            predictions
        )

        fold_scores.append(float(mse))

    return {
        "spans": spans,
        "hyperparameters": hp,
        "fold_scores": fold_scores,
        "mean_cv_mse": float(np.mean(fold_scores)),
        "convergence_warnings": convergence_warnings
    }


print("PASS: Five-model CV objective defined")


In [ ]:

# ============================================================
# STEP 6.3C.3 — Automated Bayesian Optimisation
# All Five Regression Models (No Checkpointing)
# ============================================================

import os
import json
import time
import numpy as np
import pandas as pd

from skopt import Optimizer

# ============================================================
# 1. Configuration
# ============================================================

OUTPUT_DIR = str(BASE_DIR / "bayesian_model_pilots")
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_TRIALS = 8

MODELS = [
    "SVR",
    "k-NN",
    "Random Forest",
    "Extra Trees",
    "MLP"
]

# Convert NumPy scalar values to standard Python types
def make_json_safe(values):
    return [
        v.item() if isinstance(v, np.generic) else v
        for v in values
    ]


# ============================================================
# 2. Bayesian Optimisation Function
# ============================================================

def run_model_pilot(model_name, n_trials=8):
    """
    Run Bayesian optimisation from scratch for one model.

    No checkpoint restoration or intermediate saving.
    Results are saved only after all trials finish.
    """

    search_space = SPAN_SPACE + MODEL_SPACES[model_name]

    optimizer = Optimizer(
        dimensions=search_space,
        base_estimator="GP",
        acq_func="EI",
        acq_optimizer="sampling",
        n_initial_points=5,
        random_state=42
    )

    records = []

    print(f"\nStarting optimisation: {model_name}")

    for trial_number in range(1, n_trials + 1):

        # Bayesian optimisation proposes hyperparameters
        parameters = make_json_safe(
            optimizer.ask()
        )

        start = time.perf_counter()

        # Evaluate the configuration using grouped 3-fold CV
        evaluation = evaluate_model_cv(
            model_name,
            parameters
        )

        duration = time.perf_counter() - start

        # Update Bayesian surrogate model
        optimizer.tell(
            parameters,
            evaluation["mean_cv_mse"]
        )

        record = {
            "trial": trial_number,
            "model": model_name,
            "parameters_json": json.dumps(parameters),
            "hyperparameters_json": json.dumps(
                evaluation["hyperparameters"],
                default=lambda x: x.item()
            ),
            "s1": evaluation["spans"][0],
            "s2": evaluation["spans"][1],
            "s3": evaluation["spans"][2],
            "s4": evaluation["spans"][3],
            "fold1_mse": evaluation["fold_scores"][0],
            "fold2_mse": evaluation["fold_scores"][1],
            "fold3_mse": evaluation["fold_scores"][2],
            "mean_cv_mse": evaluation["mean_cv_mse"],
            "convergence_warnings": evaluation[
                "convergence_warnings"
            ],
            "runtime_s": duration
        }

        records.append(record)

        print(
            f"Trial {trial_number}/{n_trials} | "
            f"CV MSE = {record['mean_cv_mse']:.4f} | "
            f"Runtime = {duration:.1f}s | "
            f"Warnings = {record['convergence_warnings']}",
            flush=True
        )

    # ========================================================
    # 3. Save Results After Model Completion
    # ========================================================

    results = pd.DataFrame(records)

    filename = (
        model_name.lower()
        .replace(" ", "_")
        .replace("-", "")
    )

    save_path = os.path.join(
        OUTPUT_DIR,
        f"{filename}_trials.csv"
    )

    results.to_csv(save_path, index=False)

    best = results.loc[
        results["mean_cv_mse"].idxmin()
    ]

    print(f"\nPASS: {model_name} completed")
    print("Best CV MSE:", round(best["mean_cv_mse"], 4))
    print("Best spans:", [
        int(best[f"s{i}"]) for i in range(1, 5)
    ])
    print("Results saved:", save_path)

    return results


# ============================================================
# 4. Automatically Optimise All Five Models
# ============================================================

all_results = {}
summary = []

overall_start = time.perf_counter()

for i, model_name in enumerate(MODELS, start=1):

    print("\n" + "=" * 60)
    print(f"MODEL {i}/{len(MODELS)}: {model_name}")
    print("=" * 60)

    results = run_model_pilot(
        model_name,
        n_trials=N_TRIALS
    )

    all_results[model_name] = results

    best = results.loc[
        results["mean_cv_mse"].idxmin()
    ]

    summary.append({
        "Model": model_name,
        "Trials": len(results),
        "Best CV MSE": best["mean_cv_mse"],
        "Span 1": int(best["s1"]),
        "Span 2": int(best["s2"]),
        "Span 3": int(best["s3"]),
        "Span 4": int(best["s4"]),
        "Total Runtime (s)": results["runtime_s"].sum(),
        "Convergence Warnings": (
            results["convergence_warnings"].sum()
        )
    })


# ============================================================
# 5. Final Comparison
# ============================================================

pilot_summary = pd.DataFrame(summary)

pilot_summary = pilot_summary.sort_values(
    "Best CV MSE"
).reset_index(drop=True)

display(pilot_summary.round(4))

# Save consolidated summary
pilot_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "five_model_pilot_summary.csv"
    ),
    index=False
)

total_duration = time.perf_counter() - overall_start

print("\n" + "=" * 60)
print("PASS: ALL FIVE BAYESIAN OPTIMISATIONS COMPLETED")
print("=" * 60)

print("Total Bayesian evaluations:", len(MODELS) * N_TRIALS)
print("Total model fits:", len(MODELS) * N_TRIALS * 3)
print("Total runtime (min):", round(total_duration / 60, 2))
print("Results directory:", OUTPUT_DIR)


In [ ]:

# STEP 6.3C.4 — Summarise model-specific Bayesian pilots

summary = []

for model_name in MODEL_SPACES:

    filename = (
        model_name.lower()
        .replace(" ", "_")
        .replace("-", "")
    )

    filepath = os.path.join(
        OUTPUT_DIR,
        f"{filename}_trials.csv"
    )

    if not os.path.exists(filepath):
        print(f"Not completed: {model_name}")
        continue

    trials = pd.read_csv(filepath)

    best = trials.loc[
        trials["mean_cv_mse"].idxmin()
    ]

    summary.append({
        "Model": model_name,
        "Trials": len(trials),
        "Best CV MSE": best["mean_cv_mse"],
        "Span 1": int(best["s1"]),
        "Span 2": int(best["s2"]),
        "Span 3": int(best["s3"]),
        "Span 4": int(best["s4"]),
        "Total Runtime (s)": trials["runtime_s"].sum()
    })

pilot_summary = pd.DataFrame(summary)

if not pilot_summary.empty:
    pilot_summary = pilot_summary.sort_values(
        "Best CV MSE"
    )

    display(pilot_summary.round(4))

    pilot_summary.to_csv(
        os.path.join(
            OUTPUT_DIR, "five_model_pilot_summary.csv"
        ),
        index=False
    )


### 4.8.6 Bayesian Optimisation of Alternative Regression Models (Step 6.3C)

The same Gaussian-process framework is extended to exact RBF-SVR, k-NN, Random Forest, Extra Trees and MLP. For each model, four temporally ordered spans and its algorithm-specific hyperparameters are jointly proposed and evaluated with the fixed three profile-disjoint folds.

The historical pilot executes **eight trials per model**, sampling 80 observations per development profile after computing full causal features. `skopt.Optimizer` uses a Gaussian-process surrogate, expected improvement and a fixed seed; standardisation for predictors and temperature targets is fitted on the training fold. MLP convergence warnings and all three fold errors are explicitly retained. The five current-run pilot trial CSVs, not the earlier archived model-ranking table, determine each new pilot winner.

In [ ]:

# STEP 6.3C.5 — Verify completed Bayesian pilot results

import numpy as np
import pandas as pd
import os

for model_name in MODEL_SPACES:

    filename = (
        model_name.lower()
        .replace(" ", "_")
        .replace("-", "")
    )

    path = os.path.join(
        OUTPUT_DIR,
        f"{filename}_trials.csv"
    )

    trials = pd.read_csv(path)

    # Check trial count and numerical validity
    assert len(trials) == 8

    metrics = trials[
        ["fold1_mse", "fold2_mse",
         "fold3_mse", "mean_cv_mse"]
    ]

    assert np.isfinite(metrics.to_numpy()).all()

    # Verify the mean was calculated correctly
    calculated_mean = trials[
        ["fold1_mse", "fold2_mse", "fold3_mse"]
    ].mean(axis=1)

    np.testing.assert_allclose(
        calculated_mean,
        trials["mean_cv_mse"]
    )

    print(f"\nPASS: {model_name}")
    print("Completed trials:", len(trials))
    print("Best CV MSE:", trials["mean_cv_mse"].min())
    print(
        "Convergence warnings:",
        trials["convergence_warnings"].sum()
    )

print("\nPASS: All five Bayesian pilot results verified")


### 4.8.7 Verification of the Five-Model Pilot
The pilot CSVs were subsequently checked to verify that each model had exactly eight recorded trials; every fold-specific and aggregate CV MSE was finite; and the recorded mean equalled the arithmetic average of the three fold errors. All five models passed these checks. The verification confirmed integrity of the recorded objective values and identified the MLP convergence warnings as the main training diagnostic requiring attention during scaling. This concluded the reduced-data Bayesian pilot stage.

## 4.9 Computational Scalability Assessment

Before attempting complete-development-data optimisation for all algorithms, computational feasibility was investigated. The development partition contained **1,055,198 observations across 55 recording profiles**, each yielding 108 original and temporal predictors. The remaining **275,618 observations from 14 profiles** were reserved for later final testing. SVR was used to establish an initial relationship between training-set size, prediction error, memory and runtime, because its kernel-based fitting cost was expected to increase substantially with sample count.

In the first SVR comparison (Step 6.3D.1), training increased from 2,880 to 5,760 rows, while the validation sample also increased from 1,520 to 3,040 rows. Training time rose from approximately 0.2256 to 0.6086 seconds; the MSE changed from 37.6411 to 34.9600 °C². Because both training and validation observations changed, the apparent improvement could not be attributed only to training-set size.

A revised experiment (Steps 6.3D.2–6.3D.3) therefore held the Fold 1 validation sample fixed at **3,040 rows** and increased training samples per profile to 80, 160, 320 and 640. The resulting MSE values were approximately **38.8348, 34.9600, 31.1479 and 32.4798 °C²**, respectively. Although performance was best at 320 in this particular comparison, independently generated samples were not strictly nested and only one validation fold was examined. A more controlled analysis was consequently required.

In [ ]:

# STEP 6.3D.1 — Preliminary SVR scalability experiment

import os
import json
import time
import warnings
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

MODEL_NAME = "SVR"
SAMPLE_SIZES = [80, 160]

# Load the best previously optimised configuration
trial_path = os.path.join(
    OUTPUT_DIR, "svr_trials.csv"
)

trials = pd.read_csv(trial_path)

best = trials.loc[
    trials["mean_cv_mse"].idxmin()
]

spans = [
    int(best[f"s{i}"])
    for i in range(1, 5)
]

hp = json.loads(best["hyperparameters_json"])

print("Model:", MODEL_NAME)
print("Fixed optimised spans:", spans)
print("Fixed model hyperparameters:", hp)


# Generate temporal features using full profile histories
# and select different numbers of observations afterward.

sample_blocks = {n: [] for n in SAMPLE_SIZES}

for pid, profile in train_df.groupby(
    "profile_id", sort=False
):

    X_original = make_paper_inputs(profile)

    X_full = make_temporal_features(
        X_original,
        profile["profile_id"],
        spans
    )

    for n in SAMPLE_SIZES:

        positions = np.linspace(
            0, len(profile) - 1, n, dtype=int
        )

        sample = X_full.iloc[positions].copy()

        sample["pm_target"] = (
            profile["pm"].iloc[positions].to_numpy()
        )

        sample["profile_id"] = pid

        sample_blocks[n].append(sample)


# Use the original Fold 1 validation profiles
val_ids = set(
    profile_cv.loc[
        profile_cv["cv_fold"] == 1,
        "profile_id"
    ]
)

scaling_results = []

for n in SAMPLE_SIZES:

    sample_df = pd.concat(
        sample_blocks[n],
        ignore_index=True
    )

    validation_mask = sample_df[
        "profile_id"
    ].isin(val_ids)

    training = sample_df.loc[~validation_mask]
    validation = sample_df.loc[validation_mask]

    feature_columns = [
        c for c in sample_df.columns
        if c not in ["pm_target", "profile_id"]
    ]

    X_train = training[feature_columns]
    y_train = training["pm_target"]

    X_val = validation[feature_columns]
    y_val = validation["pm_target"]

    assert len(feature_columns) == 108
    assert len(sample_df) == 55 * n

    assert set(training["profile_id"]).isdisjoint(
        set(validation["profile_id"])
    )

    # Reconstruct the selected SVR configuration
    estimator = make_regressor(MODEL_NAME, hp)

    pipeline = TransformedTargetRegressor(
        regressor=Pipeline([
            ("scaler", StandardScaler()),
            ("model", estimator)
        ]),
        transformer=StandardScaler()
    )

    # Training time
    start = time.perf_counter()
    pipeline.fit(X_train, y_train)
    train_time = time.perf_counter() - start

    # Inference time
    start = time.perf_counter()
    predictions = pipeline.predict(X_val)
    inference_time = time.perf_counter() - start

    mse = mean_squared_error(y_val, predictions)

    # Memory occupied by the numeric input matrices
    matrix_mb = (
        X_train.to_numpy().nbytes
        + X_val.to_numpy().nbytes
    ) / 1e6

    scaling_results.append({
        "Samples/Profile": n,
        "Train Rows": len(X_train),
        "Validation Rows": len(X_val),
        "Training Time (s)": train_time,
        "Inference Time (s)": inference_time,
        "Input Matrix (MB)": matrix_mb,
        "Validation MSE": mse
    })

    print(f"PASS: {n} observations/profile evaluated")

scaling_df = pd.DataFrame(scaling_results)

display(scaling_df.round(4))

scaling_df.to_csv(
    "svr_scalability_pilot.csv",
    index=False
)


In [ ]:

# STEP 6.3D.2 — Controlled SVR scalability comparison
# Keep validation data fixed while increasing training size

from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import time
import numpy as np
import pandas as pd

# Same validation profiles for both experiments
val_ids = set(
    profile_cv.loc[
        profile_cv["cv_fold"] == 1,
        "profile_id"
    ]
)

# Fixed validation dataset: 160 samples per profile
full_160 = pd.concat(
    sample_blocks[160], ignore_index=True
)

fixed_val = full_160[
    full_160["profile_id"].isin(val_ids)
].copy()

feature_cols = [
    c for c in fixed_val.columns
    if c not in ["pm_target", "profile_id"]
]

X_val = fixed_val[feature_cols]
y_val = fixed_val["pm_target"]

assert len(fixed_val) == 19 * 160

controlled_results = []

for n in [80, 160]:

    sample_df = pd.concat(
        sample_blocks[n], ignore_index=True
    )

    # Training data changes, validation data does not
    training = sample_df[
        ~sample_df["profile_id"].isin(val_ids)
    ]

    X_train = training[feature_cols]
    y_train = training["pm_target"]

    assert set(training["profile_id"]).isdisjoint(
        set(fixed_val["profile_id"])
    )

    pipeline = TransformedTargetRegressor(
        regressor=Pipeline([
            ("scaler", StandardScaler()),
            ("model", make_regressor("SVR", hp))
        ]),
        transformer=StandardScaler()
    )

    start = time.perf_counter()
    pipeline.fit(X_train, y_train)
    train_time = time.perf_counter() - start

    start = time.perf_counter()
    predictions = pipeline.predict(X_val)
    inference_time = time.perf_counter() - start

    mse = mean_squared_error(y_val, predictions)

    controlled_results.append({
        "Training Samples/Profile": n,
        "Training Rows": len(X_train),
        "Fixed Validation Rows": len(X_val),
        "Training Time (s)": train_time,
        "Inference Time (s)": inference_time,
        "Validation MSE": mse
    })

controlled_df = pd.DataFrame(controlled_results)

display(controlled_df.round(4))

controlled_df.to_csv(
    "svr_scalability_controlled.csv",
    index=False
)

print("PASS: Fixed-validation scalability comparison")


In [ ]:

# STEP 6.3D.3 — Expand SVR scalability assessment

# Generate larger sampled datasets from the full
# original 2 Hz temporal histories.

for n in [320, 640]:
    sample_blocks[n] = []

for pid, profile in train_df.groupby(
    "profile_id", sort=False
):

    X_original = make_paper_inputs(profile)

    X_full = make_temporal_features(
        X_original,
        profile["profile_id"],
        spans
    )

    for n in [320, 640]:

        positions = np.linspace(
            0, len(profile) - 1,
            n, dtype=int
        )

        sample = X_full.iloc[positions].copy()

        sample["pm_target"] = (
            profile["pm"].iloc[positions].to_numpy()
        )

        sample["profile_id"] = pid

        sample_blocks[n].append(sample)

print("PASS: Larger training samples generated")


In [ ]:

# STEP 6.3D.2 — Second part of doing the Controlled SVR scalability comparison
# but changing from for n in [80, 160]: to for n in [80, 160, 320, 640]:
# Keep validation data fixed while increasing training size

from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import time
import numpy as np
import pandas as pd

# Same validation profiles for both experiments
val_ids = set(
    profile_cv.loc[
        profile_cv["cv_fold"] == 1,
        "profile_id"
    ]
)

# Fixed validation dataset: 160 samples per profile
full_160 = pd.concat(
    sample_blocks[160], ignore_index=True
)

fixed_val = full_160[
    full_160["profile_id"].isin(val_ids)
].copy()

feature_cols = [
    c for c in fixed_val.columns
    if c not in ["pm_target", "profile_id"]
]

X_val = fixed_val[feature_cols]
y_val = fixed_val["pm_target"]

assert len(fixed_val) == 19 * 160

controlled_results = []

for n in [80, 160, 320, 640]:

    sample_df = pd.concat(
        sample_blocks[n], ignore_index=True
    )

    # Training data changes, validation data does not
    training = sample_df[
        ~sample_df["profile_id"].isin(val_ids)
    ]

    X_train = training[feature_cols]
    y_train = training["pm_target"]

    assert set(training["profile_id"]).isdisjoint(
        set(fixed_val["profile_id"])
    )

    pipeline = TransformedTargetRegressor(
        regressor=Pipeline([
            ("scaler", StandardScaler()),
            ("model", make_regressor("SVR", hp))
        ]),
        transformer=StandardScaler()
    )

    start = time.perf_counter()
    pipeline.fit(X_train, y_train)
    train_time = time.perf_counter() - start

    start = time.perf_counter()
    predictions = pipeline.predict(X_val)
    inference_time = time.perf_counter() - start

    mse = mean_squared_error(y_val, predictions)

    controlled_results.append({
        "Training Samples/Profile": n,
        "Training Rows": len(X_train),
        "Fixed Validation Rows": len(X_val),
        "Training Time (s)": train_time,
        "Inference Time (s)": inference_time,
        "Validation MSE": mse
    })

controlled_df = pd.DataFrame(controlled_results)

display(controlled_df.round(4))

controlled_df.to_csv(
    "svr_scalability_controlled.csv",
    index=False
)

print("PASS: Fixed-validation scalability comparison")


### 4.9.1 Construction and Verification of Nested Training Samples (Step 6.3D.4A)

Nested interval-bisection sampling starts with the first and last records and iteratively subdivides the longest remaining interval so that

$$A_{80}\subset A_{160}\subset A_{320}\subset A_{640}.$$

Across all 55 development profiles, these sample levels contain 4,400, 8,800, 17,600 and 35,200 observations. **The Stage 5 SVR spans, penalty and epsilon are now selected from this run's eight-trial SVR Bayesian pilot**, not hardcoded from an archived pilot result. Within the Stage 5 sample-size investigation they remain fixed so changing sample size is the controlled variable. EWMA/EWMS is computed on complete per-profile histories prior to subsampling.

In [ ]:

# STEP 6.3D.4A — Construct nested sample positions

import os
import json
import time
import heapq
import numpy as np
import pandas as pd

SAMPLE_SIZES = [80, 160, 320, 640]
MAX_SAMPLES = max(SAMPLE_SIZES)

# Restore the best SVR pilot configuration
svr_trials = pd.read_csv(
    os.path.join(OUTPUT_DIR, "svr_trials.csv")
)

best_svr = svr_trials.loc[
    svr_trials["mean_cv_mse"].idxmin()
]

svr_spans = [
    int(best_svr[f"s{i}"])
    for i in range(1, 5)
]

svr_hp = json.loads(
    best_svr["hyperparameters_json"]
)


def nested_sample_order(profile_length, max_samples):
    """
    Return observation positions in nested selection order.

    Begin with the first and last observations.
    Repeatedly bisect the largest remaining interval.

    The first 80 selected positions are contained in
    the first 160, 320, and 640 selected positions.
    """
    if profile_length < max_samples:
        raise ValueError("Profile contains too few observations")

    selected = [0, profile_length - 1]

    intervals = [
        (-(profile_length - 1), 0, profile_length - 1)
    ]

    while len(selected) < max_samples:

        _, left, right = heapq.heappop(intervals)

        middle = (left + right) // 2

        if middle in (left, right):
            continue

        selected.append(middle)

        for a, b in [(left, middle), (middle, right)]:
            if b - a > 1:
                heapq.heappush(
                    intervals, (-(b - a), a, b)
                )

    return selected


print("Optimised SVR spans:", svr_spans)
print("Model hyperparameters:", svr_hp)

# Build temporal features from complete recording histories
start_features = time.perf_counter()

nested_blocks = []

for pid, profile in train_df.groupby(
    "profile_id", sort=False
):

    # Preserve original chronological order
    assert profile.index.is_monotonic_increasing

    order = nested_sample_order(
        len(profile), MAX_SAMPLES
    )

    # Selected positions, ordered chronologically
    positions = np.sort(
        np.asarray(order, dtype=int)
    )

    # Rank tells us when each position was added
    rank_lookup = {
        position: rank
        for rank, position in enumerate(order)
    }

    X_original = make_paper_inputs(profile)

    # Use the FULL 2 Hz history before sampling
    X_full = make_temporal_features(
        X_original,
        profile["profile_id"],
        svr_spans
    )

    sample = X_full.iloc[positions].copy()

    sample["pm_target"] = (
        profile["pm"].iloc[positions].to_numpy()
    )

    sample["profile_id"] = pid

    sample["nest_rank"] = [
        rank_lookup[position]
        for position in positions
    ]

    nested_blocks.append(sample)

nested_df = pd.concat(
    nested_blocks, ignore_index=True
)

feature_generation_time = (
    time.perf_counter() - start_features
)

# Verify full sample
assert len(nested_df) == 55 * MAX_SAMPLES
assert nested_df["profile_id"].nunique() == 55
assert nested_df["nest_rank"].between(
    0, MAX_SAMPLES - 1
).all()

FEATURE_COLS = [
    c for c in nested_df.columns
    if c not in [
        "pm_target", "profile_id", "nest_rank"
    ]
]

assert len(FEATURE_COLS) == 108
assert np.isfinite(
    nested_df[FEATURE_COLS + ["pm_target"]].to_numpy()
).all()

# Verify nested membership and observation counts
for n in SAMPLE_SIZES:

    subset = nested_df[
        nested_df["nest_rank"] < n
    ]

    counts = subset.groupby("profile_id").size()

    assert len(subset) == 55 * n
    assert (counts == n).all()

    print(
        f"PASS: {n} observations/profile "
        f"({len(subset):,} observations)"
    )

print("\nPASS: All training samples are nested")
print("Predictor features:", len(FEATURE_COLS))
print(
    "Feature generation time:",
    round(feature_generation_time, 2), "seconds"
)


### 4.9.2 Three-Fold Cross-Validation and Computational Timing (Step 6.3D.4B)

The nested SVR evaluation was extended to all three established grouped folds, retaining the same **640 validation observations per validation profile** at every training size. Consequently, Fold 1 used 12,160 validation rows, while Folds 2 and 3 each used 11,520. Training observations increased with the selected nested subset, but the validation cases remained fixed within each fold. Training and validation profile IDs were checked for non-overlap.

Each of the four sample sizes was evaluated on three folds, with **three repetitions of fitting and timing for each condition**, yielding **36 model fits**. The code recorded fold-level MSE, training time, inference time and number of SVR support vectors. Median training and inference times across repetitions were used to reduce sensitivity to short-term runtime fluctuations. Repetitions of this fixed-seed configuration served mainly to assess timing variation; they were not independent training-profile splits.

In [ ]:

# STEP 6.3D.4B — Nested 3-fold SVR scalability

from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import time
import pandas as pd
import numpy as np

N_REPEATS = 3

RESULT_PATH = "svr_nested_scalability_v1.csv"

# Fresh execution: no restoration of old checkpoint CSVs.
records = []

fold_lookup = profile_cv.set_index(
    "profile_id"
)["cv_fold"]

nested_df["cv_fold"] = (
    nested_df["profile_id"]
    .map(fold_lookup)
    .astype(int)
)

assert nested_df["cv_fold"].isin([1, 2, 3]).all()


for n in SAMPLE_SIZES:

    for fold in [1, 2, 3]:

        # Training data changes with n
        training = nested_df.loc[
            (nested_df["cv_fold"] != fold)
            & (nested_df["nest_rank"] < n)
        ]

        # Validation data ALWAYS uses 640
        # observations per validation profile
        validation = nested_df.loc[
            nested_df["cv_fold"] == fold
        ]

        X_train = training[FEATURE_COLS]
        y_train = training["pm_target"]

        X_val = validation[FEATURE_COLS]
        y_val = validation["pm_target"]

        train_ids = set(training["profile_id"])
        val_ids = set(validation["profile_id"])

        assert train_ids.isdisjoint(val_ids)

        assert len(X_train) == len(train_ids) * n
        assert len(X_val) == len(val_ids) * MAX_SAMPLES

        for repeat in range(1, N_REPEATS + 1):

            estimator = make_regressor(
                "SVR", svr_hp
            )

            # Fit scalers exclusively on training fold
            pipeline = TransformedTargetRegressor(
                regressor=Pipeline([
                    ("scaler", StandardScaler()),
                    ("model", estimator)
                ]),
                transformer=StandardScaler()
            )

            # Time model training
            start = time.perf_counter()

            pipeline.fit(X_train, y_train)

            training_time = time.perf_counter() - start

            # Time prediction on fixed validation data
            start = time.perf_counter()

            predictions = pipeline.predict(X_val)

            inference_time = time.perf_counter() - start

            mse = mean_squared_error(
                y_val, predictions
            )

            fitted_svr = (
                pipeline.regressor_
                .named_steps["model"]
            )

            record = {
                "Samples_per_Profile": n,
                "Fold": fold,
                "Repeat": repeat,
                "Training_Rows": len(X_train),
                "Validation_Rows": len(X_val),
                "Training_Time_s": training_time,
                "Inference_Time_s": inference_time,
                "MSE": float(mse),
                "Support_Vectors": len(
                    fitted_svr.support_
                )
            }

            records.append(record)
            print(
                f"n={n}, fold={fold}, "
                f"repeat={repeat}: "
                f"MSE={mse:.4f}, "
                f"train={training_time:.2f}s",
                flush=True
            )

results_df = pd.DataFrame(records)
results_df.to_csv(RESULT_PATH, index=False)

# Check all 36 requested runs are present
expected_runs = (
    len(SAMPLE_SIZES) * 3 * N_REPEATS
)

assert len(results_df) == expected_runs
assert np.isfinite(
    results_df[
        ["MSE", "Training_Time_s", "Inference_Time_s"]
    ].to_numpy()
).all()

print(
    "\nPASS: All",
    expected_runs,
    "SVR training/evaluation runs completed"
)


> **Archived interpretation only.** Numbers in the following source paragraph/table were reported by an earlier run; compare against the freshly executed cells and saved trial results. This is not a new experimental result.

### 4.9.3 Assessment of Prediction Accuracy and Computational Scalability (Step 6.3D.4C–D)

| Samples/profile | Mean three-fold CV MSE (°C²) | Between-fold MSE SD (°C²) | Total three-fold training time (s) | Mean support vectors |
|---|---:|---:|---:|---:|
| 80 | 36.3772 | 4.6570 | 0.7386 | 667.7 |
| 160 | 36.4411 | 4.4767 | 1.8672 | 952.7 |
| 320 | **36.0818** | 5.1517 | 5.0677 | 1321.7 |
| 640 | 38.3040 | 6.5129 | 13.5167 | 1721.3 |

With the fixed pilot hyperparameters, 320 observations per profile gave the smallest mean CV MSE among the four tested sizes. The move to 640 observations did not improve accuracy, while the summed fold training time increased from approximately 5.07 to 13.52 seconds. The mean number of support vectors also grew from approximately 668 at the smallest size to 1,721 at the largest, suggesting increasing potential inference cost.

Fold-level analysis showed that MSE increased from 320 to 640 observations per profile in **all three folds**. The changes were approximately **+3.1633, +0.3136 and +3.1895 °C²** for Folds 1–3 respectively. Nevertheless, this does not prove that additional data intrinsically worsens SVR. The hyperparameters were optimised for a smaller sample and were held fixed, so they might not be appropriate for the expanded training distributions. The purpose of this experiment was to inform resource allocation, not to present a new learning-curve result as the independent research contribution.

In [ ]:

# STEP 6.3D.4C — Summarise nested scalability results

import pandas as pd
import matplotlib.pyplot as plt

# Average repeated results within each fold
fold_results = (
    results_df.groupby(
        ["Samples_per_Profile", "Fold"],
        as_index=False
    )
    .agg(
        MSE=("MSE", "mean"),
        Training_Rows=("Training_Rows", "first"),
        Validation_Rows=("Validation_Rows", "first"),
        Training_Time_s=("Training_Time_s", "median"),
        Inference_Time_s=("Inference_Time_s", "median"),
        Support_Vectors=("Support_Vectors", "median")
    )
)

print("Results for each validation fold:")
display(fold_results.round(4))


# Aggregate across all three folds
scalability_summary = (
    fold_results.groupby(
        "Samples_per_Profile",
        as_index=False
    )
    .agg(
        Mean_CV_MSE=("MSE", "mean"),
        Fold_MSE_SD=("MSE", "std"),
        Total_3Fold_Training_s=("Training_Time_s", "sum"),
        Total_3Fold_Inference_s=("Inference_Time_s", "sum"),
        Mean_Support_Vectors=("Support_Vectors", "mean")
    )
)

print("\nOverall scalability summary:")
display(scalability_summary.round(4))

scalability_summary.to_csv(
    "svr_nested_scalability_summary.csv",
    index=False
)


# Graph 1 — Accuracy versus training size
plt.figure(figsize=(9, 5))

plt.plot(
    scalability_summary["Samples_per_Profile"],
    scalability_summary["Mean_CV_MSE"],
    marker="o"
)

plt.xlabel("Training observations per profile")
plt.ylabel("Mean 3-fold CV MSE (°C²)")
plt.title("SVR Accuracy versus Nested Training-Set Size")
plt.grid(True)
plt.show()


# Graph 2 — Computational cost
plt.figure(figsize=(9, 5))

plt.plot(
    scalability_summary["Samples_per_Profile"],
    scalability_summary["Total_3Fold_Training_s"],
    marker="o"
)

plt.xlabel("Training observations per profile")
plt.ylabel("Total 3-fold training time (s)")
plt.title("SVR Training Cost versus Dataset Size")
plt.grid(True)
plt.show()

print("PASS: Scalability analysis completed")


In [ ]:

# STEP 6.3D.4D — Inspect fold-level accuracy differences

fold_mse_table = fold_results.pivot(
    index="Samples_per_Profile",
    columns="Fold",
    values="MSE"
)

print("Validation MSE across the three folds:")
display(fold_mse_table.round(4))

print("\nChange in MSE from 320 to 640 samples/profile:")

mse_change = (
    fold_mse_table.loc[640]
    - fold_mse_table.loc[320]
)

display(
    pd.DataFrame({
        "MSE change (°C²)": mse_change
    }).round(4)
)


### 4.9.4 Full-Development-Data OLS Feasibility Assessment (Step 6.3D.5A)

The analysis next returned to OLS to determine whether a full-development-data benchmark was computationally feasible. Directly storing more than one million observations with 108 `float64` predictors would require approximately **0.91 GB for the predictor array alone**, excluding intermediate Pandas arrays and processing overhead. A profile-wise, memory-conscious QR decomposition was therefore implemented to preserve the information needed for linear least-squares fitting and residual calculation.

Each complete recording profile was processed at the original 2 Hz rate. Its 108 temporal and original predictors, an intercept column and the permanent-magnet temperature target were combined into an augmented regression matrix. QR compression was applied in blocks and then combined within the profile, retaining a small triangular factor and summary statistics instead of preserving the complete multi-profile predictor matrix. In each CV fold, the training-profile means and variances were combined to obtain training-only standardisation, and OLS coefficients were solved using a rank-aware least-squares method. Validation residual sums of squares were computed from the QR factors of the held-out development profiles.

A numerical check compared directly calculated residual sums of squares with QR-compressed residual sums of squares for a representative profile. The values agreed within numerical precision (**1080.94694** in both calculations), supporting the residual-preservation implementation.

In [ ]:

# STEP 6.3D.5A — Memory-efficient full-data OLS

import time
import numpy as np
import pandas as pd

from scipy.linalg import qr, lstsq

# We retain all 55 original development profiles.
assert len(train_df) == 1055198
assert train_df["profile_id"].nunique() == 55

# Original grouped CV assignments from Step 6.1
cv_lookup = (
    profile_cv.set_index("profile_id")["cv_fold"]
    .astype(int)
    .to_dict()
)

assert set(cv_lookup.values()) == {1, 2, 3}


def compress_profile_ols(profile, spans, chunk_size=4096):
    """
    Compress one full recording profile using QR.

    Parameters
    ----------
    profile : pandas.DataFrame
        Original 2 Hz measurements from one recording profile.
    spans : list of int
        Four EWMA/EWMS smoothing spans.
    chunk_size : int
        Maximum observations processed per QR block.

    Returns
    -------
    dict
        Profile size, predictor statistics and compressed R.
    """
    # Build the 12 original and derived input variables
    X_original = make_paper_inputs(profile)

    # Build all 108 predictors from the FULL time history
    X_temporal = make_temporal_features(
        X_original,
        profile["profile_id"],
        spans
    )

    X = X_temporal.to_numpy(dtype=np.float64)
    y = profile["pm"].to_numpy(dtype=np.float64)

    n, p = X.shape

    assert p == 108
    assert np.isfinite(X).all()
    assert np.isfinite(y).all()

    # Per-profile statistics for fold-specific scaling
    feature_mean = X.mean(axis=0)
    feature_var = X.var(axis=0)

    # Augmented matrix: [intercept, predictors, target]
    n_columns = p + 2
    R_profile = None

    for start in range(0, n, chunk_size):

        stop = min(start + chunk_size, n)
        size = stop - start

        block = np.empty(
            (size, n_columns),
            dtype=np.float64
        )

        block[:, 0] = 1.0
        block[:, 1:-1] = X[start:stop]
        block[:, -1] = y[start:stop]

        # Keep the compressed triangular factor.
        R_block = qr(
            block,
            mode="r",
            check_finite=False
        )[0][:n_columns, :]

        if R_profile is None:
            R_profile = R_block
        else:
            # Combine QR factors without storing old blocks
            combined = np.vstack([
                R_profile, R_block
            ])

            R_profile = qr(
                combined,
                mode="r",
                check_finite=False
            )[0][:n_columns, :]

    return {
        "n": n,
        "mean": feature_mean,
        "var": feature_var,
        "R": R_profile
    }


def evaluate_full_ols_cv(spans):
    """
    Evaluate an OLS configuration on all development rows.

    Uses profile-wise QR compression and training-fold-only
    standardisation. No test profiles are accessed.
    """
    start_total = time.perf_counter()

    profile_stats = {}

    # =================================================
    # A — Construct and compress all 55 profiles
    # =================================================

    for count, (pid, profile) in enumerate(
        train_df.groupby("profile_id", sort=False),
        start=1
    ):

        pid = int(pid)

        profile_stats[pid] = compress_profile_ols(
            profile,
            spans
        )

        if count % 10 == 0 or count == 55:
            print(
                f"Processed {count}/55 profiles",
                flush=True
            )

    feature_time = time.perf_counter() - start_total

    # =================================================
    # B — Evaluate the same three grouped CV folds
    # =================================================

    start_cv = time.perf_counter()

    fold_results = []

    for fold in [1, 2, 3]:

        training_ids = [
            pid for pid in profile_stats
            if cv_lookup[pid] != fold
        ]

        validation_ids = [
            pid for pid in profile_stats
            if cv_lookup[pid] == fold
        ]

        assert set(training_ids).isdisjoint(
            set(validation_ids)
        )

        # Calculate training-fold-only feature statistics
        n_train = sum(
            profile_stats[pid]["n"]
            for pid in training_ids
        )

        mu = sum(
            profile_stats[pid]["n"]
            * profile_stats[pid]["mean"]
            for pid in training_ids
        ) / n_train

        variance = sum(
            profile_stats[pid]["n"] * (
                profile_stats[pid]["var"]
                + (
                    profile_stats[pid]["mean"] - mu
                ) ** 2
            )
            for pid in training_ids
        ) / n_train

        scale = np.sqrt(np.maximum(variance, 0))

        # Match StandardScaler behaviour for zero variance
        scale[scale == 0] = 1.0

        # Combine compressed training matrices
        R_train = np.vstack([
            profile_stats[pid]["R"]
            for pid in training_ids
        ])

        # Apply training-fold standardisation algebraically
        R_scaled = R_train.copy()

        R_scaled[:, 1:-1] = (
            R_train[:, 1:-1]
            - R_train[:, [0]] * mu
        ) / scale

        # Fit OLS by rank-aware least squares
        coefficients, _, rank, singular_values = lstsq(
            R_scaled[:, :-1],
            R_scaled[:, -1],
            lapack_driver="gelsd"
        )

        # Convert fitted coefficients to raw feature units
        beta = coefficients[1:] / scale

        intercept = (
            coefficients[0]
            - np.dot(mu, beta)
        )

        # Residual coefficient vector in original units
        residual_vector = np.concatenate([
            [intercept],
            beta,
            [-1.0]
        ])

        # Compute validation SSE from compressed R factors
        n_validation = 0
        validation_sse = 0.0

        for pid in validation_ids:

            R = profile_stats[pid]["R"]

            residuals_compressed = (
                R @ residual_vector
            )

            validation_sse += float(
                np.dot(
                    residuals_compressed,
                    residuals_compressed
                )
            )

            n_validation += profile_stats[pid]["n"]

        mse = validation_sse / n_validation

        assert np.isfinite(mse)
        assert mse >= 0

        fold_results.append({
            "Fold": fold,
            "Training_Rows": n_train,
            "Validation_Rows": n_validation,
            "MSE": mse,
            "Numerical_Rank": rank
        })

    cv_time = time.perf_counter() - start_cv

    results = pd.DataFrame(fold_results)

    mean_cv_mse = results["MSE"].mean()

    return {
        "spans": list(map(int, spans)),
        "fold_results": results,
        "mean_cv_mse": float(mean_cv_mse),
        "feature_time_s": feature_time,
        "cv_time_s": cv_time,
        "total_time_s": time.perf_counter() - start_total
    }


print("PASS: Full-data OLS evaluation function defined")


### 4.9.5 Initial Full-Data OLS Evaluation and Computational Timing (Step 6.3D.5B)

The starting temporal spans are automatically taken from the **current run's best of 28 reduced-sample OLS Bayesian trials**. Using these newly selected spans, the original QR-compressed least-squares procedure evaluates **all 1,055,198 development observations**, with fold-specific fitting and validation respecting the previously locked profile assignments.

The code records each fold's number of observations, numerical rank, validation MSE, feature generation time, linear solve time and total time. Interpret the new result printed below; the historical table and its timing values are deliberately **not reproduced as if they were fresh measurements**.

In [ ]:

# STEP 6.3D.5 — Verify QR compression numerically

import numpy as np

# Use a small subset of an existing development profile
pid = int(profile_cv["profile_id"].iloc[0])

demo = train_df.loc[
    train_df["profile_id"] == pid
].head(300).copy()

spans = list(map(int, best_spans))

X_demo = make_temporal_features(
    make_paper_inputs(demo),
    demo["profile_id"],
    spans
).to_numpy(dtype=np.float64)

y_demo = demo["pm"].to_numpy(dtype=np.float64)

compressed = compress_profile_ols(
    demo, spans, chunk_size=128
)

# A deliberately chosen example regression equation
w = np.zeros(110)
w[0] = 25.0
w[1] = 0.01
w[-1] = -1.0

# Calculate the residual sum of squares directly
direct_residuals = (
    w[0] + 0.01 * X_demo[:, 0] - y_demo
)

direct_sse = np.sum(direct_residuals**2)

# Calculate equivalent residuals from QR factor
compressed_sse = np.sum(
    (compressed["R"] @ w)**2
)

np.testing.assert_allclose(
    direct_sse,
    compressed_sse,
    rtol=1e-8
)

print("PASS: QR residual calculation verified")
print("Direct SSE:", direct_sse)
print("Compressed SSE:", compressed_sse)


In [ ]:

# STEP 6.3D.5B — First full-data OLS feasibility run

import time
import pandas as pd

# Best spans from our expanded 28-trial pilot
# Use the winner of this execution's 8+20 pilot/expanded search.
# The original OLS feasibility method is unchanged; only its seed provenance is fixed.
assert len(all_trials) == 28
candidate_spans = list(map(int, best_spans))

print("Evaluating full-data OLS configuration")
print("Spans:", candidate_spans)
print("Development observations:", len(train_df))

full_ols_result = evaluate_full_ols_cv(
    candidate_spans
)

print("\nPASS: Full-data OLS evaluation completed")

print("\nIndividual fold results:")
display(
    full_ols_result["fold_results"].round(4)
)

print(
    "\nMean 3-fold CV MSE:",
    round(full_ols_result["mean_cv_mse"], 4),
    "°C²"
)

print(
    "Feature generation and compression:",
    round(full_ols_result["feature_time_s"], 2),
    "seconds"
)

print(
    "Cross-validation fitting and evaluation:",
    round(full_ols_result["cv_time_s"], 2),
    "seconds"
)

print(
    "Total runtime:",
    round(full_ols_result["total_time_s"], 2),
    "seconds"
)

# Save results
full_ols_result["fold_results"].to_csv(
    "ols_full_data_first_trial_folds.csv",
    index=False
)

full_trial_summary = pd.DataFrame([{
    "s1": candidate_spans[0],
    "s2": candidate_spans[1],
    "s3": candidate_spans[2],
    "s4": candidate_spans[3],
    "mean_cv_mse": full_ols_result["mean_cv_mse"],
    "feature_time_s": full_ols_result["feature_time_s"],
    "cv_time_s": full_ols_result["cv_time_s"],
    "total_time_s": full_ols_result["total_time_s"]
}])

full_trial_summary.to_csv(
    "ols_full_data_first_trial_summary.csv",
    index=False
)


### 4.9.6 Numerical Equivalence Between QR-Based OLS and Scikit-learn (Step 6.3D.6A)

A further check examined whether QR-based regression produced the same fitted predictions as Scikit-learn's `LinearRegression`, beyond the earlier residual-identity test. The comparison used the same 4,400-observation pilot dataset and Fold 1 split for both implementations. Feature standardisation was reproduced from training-profile statistics, and validation predictions were calculated using the same observations.

The maximum difference between the two sets of predictions was approximately **$1.17\times10^{-12}$ °C**. Both methods returned a validation MSE of approximately **88.6562614 °C²**, and the QR regression had numerical rank 109. The numerical assertions passed. This verified the compressed fitting and prediction procedure on the selected verification sample before its use in repeated full-data Bayesian evaluations.

In [ ]:

# STEP 6.3D.6A — Verify QR-based OLS numerically

import numpy as np
import pandas as pd

from scipy.linalg import qr, lstsq
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# Use the existing 4,400-observation verification dataset.
# It is based on full profile histories before subsampling.
assert len(smoke_df) == 4400
assert smoke_df["profile_id"].nunique() == 55

feature_cols = [
    c for c in smoke_df.columns
    if c not in ["pm_target", "profile_id"]
]

assert len(feature_cols) == 108

val_ids = set(
    profile_cv.loc[
        profile_cv["cv_fold"] == 1,
        "profile_id"
    ]
)

training = smoke_df.loc[
    ~smoke_df["profile_id"].isin(val_ids)
]

validation = smoke_df.loc[
    smoke_df["profile_id"].isin(val_ids)
]

X_train = training[feature_cols].to_numpy()
y_train = training["pm_target"].to_numpy()

X_val = validation[feature_cols].to_numpy()
y_val = validation["pm_target"].to_numpy()

assert set(training["profile_id"]).isdisjoint(
    set(validation["profile_id"])
)

# A — Reference implementation using Scikit-learn
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

reference_model = LinearRegression()
reference_model.fit(X_train_scaled, y_train)

reference_predictions = reference_model.predict(
    X_val_scaled
)

reference_mse = mean_squared_error(
    y_val, reference_predictions
)


# B — Compress training profiles independently with QR
R_blocks = []
profile_statistics = []

for pid, group in training.groupby(
    "profile_id", sort=False
):

    Xp = group[feature_cols].to_numpy()
    yp = group["pm_target"].to_numpy()

    augmented = np.column_stack([
        np.ones(len(Xp)),
        Xp,
        yp
    ])

    R = qr(
        augmented,
        mode="r",
        check_finite=False
    )[0]

    R_blocks.append(R)

    profile_statistics.append({
        "n": len(Xp),
        "mean": Xp.mean(axis=0),
        "var": Xp.var(axis=0)
    })


# C — Reconstruct training-fold standardisation
n_total = sum(
    item["n"] for item in profile_statistics
)

mu = sum(
    item["n"] * item["mean"]
    for item in profile_statistics
) / n_total

variance = sum(
    item["n"] * (
        item["var"] + (item["mean"] - mu)**2
    )
    for item in profile_statistics
) / n_total

scale = np.sqrt(np.maximum(variance, 0))
scale[scale == 0] = 1.0

np.testing.assert_allclose(
    mu, scaler.mean_, rtol=1e-9, atol=1e-9
)

np.testing.assert_allclose(
    scale, scaler.scale_, rtol=1e-9, atol=1e-9
)


# D — Fit OLS using the compressed matrices
R_train = np.vstack(R_blocks)

R_scaled = R_train.copy()

R_scaled[:, 1:-1] = (
    R_train[:, 1:-1]
    - R_train[:, [0]] * mu
) / scale

coefficients, _, rank, _ = lstsq(
    R_scaled[:, :-1],
    R_scaled[:, -1],
    lapack_driver="gelsd"
)

beta = coefficients[1:] / scale

intercept = (
    coefficients[0] - np.dot(mu, beta)
)

qr_predictions = X_val @ beta + intercept

qr_mse = mean_squared_error(
    y_val, qr_predictions
)


# E — Compare predictions and validation errors
max_difference = np.max(
    np.abs(reference_predictions - qr_predictions)
)

np.testing.assert_allclose(
    qr_predictions,
    reference_predictions,
    rtol=1e-8,
    atol=1e-6
)

np.testing.assert_allclose(
    qr_mse,
    reference_mse,
    rtol=1e-8
)

print("PASS: QR and Scikit-learn agree")
print("Numerical rank:", rank)
print("Maximum prediction difference:", max_difference)
print("Scikit-learn MSE:", reference_mse)
print("QR-based OLS MSE:", qr_mse)


### 4.9.7 Configuration of the Full-Data Bayesian Search (Step 6.3D.6B)

The original 20-evaluation full-development-data OLS Gaussian-process search is preserved. **Trial 1 re-evaluates the freshly selected reduced-sample OLS pilot winner** on the full development dataset. The remaining 19 trial configurations are proposed using expected improvement; the OLS parameter domain, grouped three-fold objective and random seed remain the same. No historical seed spans or old trial scores are imported.

In [ ]:
# STEP 6.3D.6B — Fresh full-data Bayesian optimisation configuration
# No Google Drive mount, checkpoint restore or prior CSV required.
from skopt import Optimizer
from skopt.space import Integer
import numpy as np
import pandas as pd
import json
import os

OLS_RESULTS_DIR = str(BASE_DIR / "ols_full_bayesian")
os.makedirs(OLS_RESULTS_DIR, exist_ok=True)
RESULTS_PATH = os.path.join(OLS_RESULTS_DIR, "ols_full_bayesian_trials.csv")
TARGET_TRIALS = 20  # includes the earlier feasibility evaluation

search_space = [
    Integer(5, 100, name="s1"),
    Integer(1, 400, name="gap_2"),
    Integer(1, 1500, name="gap_3"),
    Integer(1, 6000, name="gap_4")
]

SEED_SPANS = list(map(int, candidate_spans))
assert decode_spans(encode_spans(SEED_SPANS)) == SEED_SPANS

print("PASS: Fresh full-data Bayesian search configured")
print("Full-data OLS evaluations (including seed):", TARGET_TRIALS)


### 4.9.8 Full-Data Bayesian Hyperparameter Optimisation (Step 6.3D.6C)

Each of the original 20 evaluations constructs profile-causal temporal features, compresses development-profile feature/target matrices by QR and evaluates the candidate with training-fold-only scaling across three group-disjoint folds. After each evaluation, the Gaussian-process surrogate is updated using the observed mean validation MSE.

The cell below prints and saves this run's trial-level spans, fold MSE, runtime and numerical rank; **neither the trial index of the winner nor its score is assumed in advance**.

In [ ]:
# STEP 6.3D.6C — Full-data OLS Bayesian optimisation, from scratch
# Relies only on the current Run All execution's full_ols_result.
import json
import time
import numpy as np
import pandas as pd
from skopt import Optimizer

assert full_ols_result["spans"] == SEED_SPANS
assert len(full_ols_result["fold_results"]) == 3

ols_optimizer = Optimizer(
    dimensions=search_space,
    base_estimator="GP",
    n_initial_points=6,
    acq_func="EI",
    acq_optimizer="sampling",
    random_state=42
)
seed_parameters = encode_spans(SEED_SPANS)
ols_optimizer.tell(seed_parameters, float(full_ols_result["mean_cv_mse"]))

seed_folds = full_ols_result["fold_results"].sort_values("Fold")
records = [{
    "trial": 1,
    "source": "full-data feasibility run",
    "s1": SEED_SPANS[0],
    "s2": SEED_SPANS[1],
    "s3": SEED_SPANS[2],
    "s4": SEED_SPANS[3],
    "parameters_json": json.dumps(seed_parameters),
    "fold1_mse": float(seed_folds.iloc[0]["MSE"]),
    "fold2_mse": float(seed_folds.iloc[1]["MSE"]),
    "fold3_mse": float(seed_folds.iloc[2]["MSE"]),
    "mean_cv_mse": float(full_ols_result["mean_cv_mse"]),
    "min_numerical_rank": int(seed_folds["Numerical_Rank"].min()),
    "feature_time_s": float(full_ols_result["feature_time_s"]),
    "cv_time_s": float(full_ols_result["cv_time_s"]),
    "total_time_s": float(full_ols_result["total_time_s"])
}]

# Evaluations 2...20 have brand-new proposals and scores.
for trial_number in range(2, TARGET_TRIALS + 1):
    parameters = [int(v) for v in ols_optimizer.ask()]
    spans = decode_spans(parameters)

    print(f"OLS full-data trial {trial_number}/{TARGET_TRIALS}: {spans}", flush=True)
    evaluation = evaluate_full_ols_cv(spans)
    score = float(evaluation["mean_cv_mse"])
    assert np.isfinite(score)
    ols_optimizer.tell(parameters, score)
    f = evaluation["fold_results"].sort_values("Fold")
    record = {
        "trial": trial_number,
        "source": "Bayesian proposal",
        "s1": spans[0], "s2": spans[1], "s3": spans[2], "s4": spans[3],
        "parameters_json": json.dumps(parameters),
        "fold1_mse": float(f.iloc[0]["MSE"]),
        "fold2_mse": float(f.iloc[1]["MSE"]),
        "fold3_mse": float(f.iloc[2]["MSE"]),
        "mean_cv_mse": score,
        "min_numerical_rank": int(f["Numerical_Rank"].min()),
        "feature_time_s": evaluation["feature_time_s"],
        "cv_time_s": evaluation["cv_time_s"],
        "total_time_s": evaluation["total_time_s"]
    }
    records.append(record)
    print(f"CV MSE {score:.4f}; best {min(r['mean_cv_mse'] for r in records):.4f}", flush=True)

full_trials = pd.DataFrame(records)
assert len(full_trials) == TARGET_TRIALS
full_trials.to_csv(RESULTS_PATH, index=False)
best_trial = full_trials.loc[full_trials["mean_cv_mse"].idxmin()]
best_spans_full = [int(best_trial[f"s{i}"]) for i in range(1, 5)]
print("PASS: OLS full-data Bayesian optimisation complete")
print("Best spans:", best_spans_full)
print("Best mean CV MSE:", best_trial["mean_cv_mse"])
display(full_trials.sort_values("mean_cv_mse").head(10).round(4))


> **Archived interpretation only.** Numbers in the following source paragraph/table were reported by an earlier run; compare against the freshly executed cells and saved trial results. This is not a new experimental result.

### 4.9.9 Evaluation of Full-Data Bayesian Optimisation Performance (Step 6.3D.6D)

Analyse the freshly executed full-data OLS trial table below by comparing the first full-development trial with the newly selected minimum-CV-MSE configuration. Report fold-level performance and total computation time, and distinguish temporal-feature generation cost from the QR-compressed linear solve. A 20-trial finite Bayesian search is a computationally limited development investigation, not proof of a global optimum. No historical MSE or selected span values are inserted into this section.

In [ ]:

# STEP 6.3D.6D — Analyse full-data Bayesian results

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Load directly from the persistent CSV
history = pd.read_csv(RESULTS_PATH)

history = history.sort_values("trial").copy()

history["best_so_far"] = (
    history["mean_cv_mse"].cummin()
)

# Plot Bayesian optimisation history
plt.figure(figsize=(11, 5))

plt.plot(
    history["trial"],
    history["mean_cv_mse"],
    "o-",
    label="Candidate 3-fold CV MSE"
)

plt.plot(
    history["trial"],
    history["best_so_far"],
    "s-",
    label="Best CV MSE so far"
)

plt.axhline(
    history.iloc[0]["mean_cv_mse"],
    linestyle="--",
    label="Initial full-data configuration"
)

plt.xlabel("Full-data Bayesian optimisation trial")
plt.ylabel("Average 3-fold CV MSE (°C²)")
plt.title("Full-Data OLS Bayesian Optimisation")
plt.legend()
plt.grid(True)
plt.show()


# Compare initial and final configurations
best = history.loc[
    history["mean_cv_mse"].idxmin()
]

initial_mse = float(
    history.iloc[0]["mean_cv_mse"]
)

best_mse = float(best["mean_cv_mse"])

improvement = (
    (initial_mse - best_mse) / initial_mse
) * 100

comparison = pd.DataFrame([
    {
        "Configuration": "Initial",
        "s1": int(history.iloc[0]["s1"]),
        "s2": int(history.iloc[0]["s2"]),
        "s3": int(history.iloc[0]["s3"]),
        "s4": int(history.iloc[0]["s4"]),
        "CV MSE": initial_mse
    },
    {
        "Configuration": "Best full-data",
        "s1": int(best["s1"]),
        "s2": int(best["s2"]),
        "s3": int(best["s3"]),
        "s4": int(best["s4"]),
        "CV MSE": best_mse
    }
])

display(comparison.round(4))

print(f"Relative MSE improvement: {improvement:.2f}%")

print(
    "Mean runtime per evaluation:",
    round(history["total_time_s"].mean(), 2),
    "seconds"
)

print(
    "Total recorded evaluation runtime:",
    round(history["total_time_s"].sum() / 60, 2),
    "minutes"
)

print(
    "Minimum numerical rank:",
    history["min_numerical_rank"].min()
)


### 4.9.10 Model-Specific Computational Budgets and Larger-Scale Validation (Step 6.3D.7A)

For the five remaining regression models, full-million-row Bayesian optimisation had not been demonstrated feasible. A larger but resource-limited sampling strategy was therefore adopted. The fixed SVR scalability tests motivated using **320 training observations per profile** for SVR, while **640 training observations per profile** were selected for k-NN, Random Forest, Extra Trees and MLP. Every algorithm was evaluated on the same **640 selected observations per validation profile** across the original three grouped folds, with all temporal features calculated from the underlying complete 2 Hz recording histories before sampling.

This produced a common validation design but different training budgets: across all 55 profiles, the training sampling pools contained 17,600 selected observations for SVR and 35,200 for the other four models, before splitting each fold into training and validation profiles. Standardisation remained training-fold-only. For MLP, the training procedure was changed from the early pilot by increasing `max_iter` from 200 to 400 and enabling early stopping with a 10% internal training-fold validation fraction. These changes aimed to address earlier convergence warnings, but they also mean the larger-scale MLP setup is not strictly identical to the pilot setup.

In [ ]:

# STEP 6.3D.7A — Larger-scale model evaluation

import os
import json
import time
import warnings
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

# Model-specific sample sizes
TRAIN_SAMPLES = {
    "SVR": 320,
    "k-NN": 640,
    "Random Forest": 640,
    "Extra Trees": 640,
    "MLP": 640
}

VALIDATION_SAMPLES = 640

# Existing development-profile fold assignments
fold_lookup = (
    profile_cv.set_index("profile_id")["cv_fold"]
    .astype(int)
    .to_dict()
)

assert len(train_df) == 1055198
assert len(fold_lookup) == 55

for function in [
    make_paper_inputs,
    make_temporal_features,
    make_regressor,
    nested_sample_order
]:
    assert callable(function)


def evaluate_larger_model_cv(model_name, parameters):
    """
    Evaluate one Bayesian candidate using:
    - Full original 2-Hz temporal histories
    - Nested training samples
    - Fixed 640-sample/profile validation
    - Three profile-grouped CV folds
    """

    start_total = time.perf_counter()

    n_train = TRAIN_SAMPLES[model_name]
    n_val = VALIDATION_SAMPLES
    n_select = max(n_train, n_val)

    spans = decode_spans(parameters[:4])

    hp = {
        dimension.name: value
        for dimension, value in zip(
            MODEL_SPACES[model_name],
            parameters[4:]
        )
    }

    # ==========================================
    # A — Build the temporal features
    # ==========================================

    feature_arrays = []
    target_arrays = []
    profile_arrays = []
    rank_arrays = []

    for pid, profile in train_df.groupby(
        "profile_id", sort=False
    ):

        # Nested sample order is independent of
        # the particular smoothing spans.
        order = nested_sample_order(
            len(profile), n_select
        )

        positions = np.sort(
            np.asarray(order, dtype=int)
        )

        rank_lookup = {
            int(position): rank
            for rank, position in enumerate(order)
        }

        ranks = np.asarray([
            rank_lookup[int(position)]
            for position in positions
        ])

        X_original = make_paper_inputs(profile)

        # Calculate EWMA/EWMS using all 2-Hz observations
        X_full = make_temporal_features(
            X_original,
            profile["profile_id"],
            spans
        )

        X_selected = X_full.iloc[
            positions
        ].to_numpy(dtype=np.float64)

        y_selected = profile["pm"].iloc[
            positions
        ].to_numpy(dtype=np.float64)

        feature_arrays.append(X_selected)
        target_arrays.append(y_selected)

        profile_arrays.append(
            np.full(len(positions), int(pid))
        )

        rank_arrays.append(ranks)

    X = np.vstack(feature_arrays)
    y = np.concatenate(target_arrays)
    profile_ids = np.concatenate(profile_arrays)
    sample_ranks = np.concatenate(rank_arrays)

    feature_time = time.perf_counter() - start_total

    assert X.shape[1] == 108
    assert np.isfinite(X).all()
    assert np.isfinite(y).all()

    fold_ids = pd.Series(
        profile_ids
    ).map(fold_lookup).to_numpy(dtype=int)

    # ==========================================
    # B — Train and evaluate on all three folds
    # ==========================================

    fold_scores = []
    convergence_warnings = 0
    training_time = 0.0
    inference_time = 0.0

    for fold in [1, 2, 3]:

        train_mask = (
            (fold_ids != fold)
            & (sample_ranks < n_train)
        )

        val_mask = (
            (fold_ids == fold)
            & (sample_ranks < n_val)
        )

        assert set(
            profile_ids[train_mask]
        ).isdisjoint(
            set(profile_ids[val_mask])
        )

        estimator = make_regressor(
            model_name, hp
        )

        # Revised MLP training procedure:
        # early stopping uses training-fold data only.
        if model_name == "MLP":
            estimator.max_iter = 400
            estimator.early_stopping = True
            estimator.validation_fraction = 0.10
            estimator.n_iter_no_change = 15

        pipeline = TransformedTargetRegressor(
            regressor=Pipeline([
                ("scaler", StandardScaler()),
                ("model", estimator)
            ]),
            transformer=StandardScaler()
        )

        start_fit = time.perf_counter()

        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter(
                "always", ConvergenceWarning
            )

            pipeline.fit(
                X[train_mask],
                y[train_mask]
            )

        training_time += (
            time.perf_counter() - start_fit
        )

        convergence_warnings += sum(
            issubclass(w.category, ConvergenceWarning)
            for w in caught
        )

        start_predict = time.perf_counter()

        predictions = pipeline.predict(
            X[val_mask]
        )

        inference_time += (
            time.perf_counter() - start_predict
        )

        assert np.isfinite(predictions).all()

        fold_scores.append(
            float(mean_squared_error(
                y[val_mask],
                predictions
            ))
        )

    mean_mse = float(np.mean(fold_scores))

    return {
        "spans": spans,
        "hyperparameters": hp,
        "fold_scores": fold_scores,
        "mean_cv_mse": mean_mse,
        "feature_time_s": feature_time,
        "training_time_s": training_time,
        "inference_time_s": inference_time,
        "total_time_s": time.perf_counter() - start_total,
        "convergence_warnings": convergence_warnings
    }


print("PASS: Larger-scale CV objective defined")


### 4.9.11 Larger-Scale Bayesian Hyperparameter Search (Step 6.3D.7B)

An automated larger-scale Bayesian search was performed sequentially for SVR, k-NN, Random Forest, Extra Trees and MLP. **Twelve configurations per model** were evaluated: Trial 1 reevaluated that model's best eight-trial pilot hyperparameters under the *new* sampling and validation protocol, and the following 11 configurations were generated by Gaussian-process Bayesian optimisation. Previous pilot MSE values were deliberately not reused as objective scores. The workflow therefore performed **60 three-fold evaluations or 180 fold-level model fits** and saved each model's result table in `scaled_model_bayesian/`.

The search executed automatically without needing to manually choose or rerun each model. All five searches completed, and their output tables were checked for finite, internally consistent fold-mean scores. **No convergence warnings were recorded** during the larger-scale runs, including for the revised MLP implementation. The measured runtime varied strongly among algorithms, reflecting differences in model complexity, selected hyperparameters and training sample size.

In [ ]:
# STEP 6.3D.7B — Automatic large-sample Bayesian optimisation (five models)
# Fresh Run All: start from the freshly produced 6.3C pilot CSVs.
# No checkpoint files, no Google Drive login, no per-model manual action.
import os
import json
import time
import numpy as np
import pandas as pd
from skopt import Optimizer

SEARCH_DIR = str(BASE_DIR / "scaled_model_bayesian")
os.makedirs(SEARCH_DIR, exist_ok=True)
MODEL_ORDER = ["SVR", "k-NN", "Random Forest", "Extra Trees", "MLP"]
SCALED_TRIALS = 12  # each includes a newly scored pilot seed


def json_safe(values):
    """Return native Python values accepted by JSON and scikit-optimize."""
    return [v.item() if isinstance(v, np.generic) else v for v in values]


def run_scaled_bayesian(model_name, n_trials=SCALED_TRIALS):
    """Optimise one model with three profile-grouped CV folds from scratch.

    Previous pilot hyperparameters are used only as a starting configuration;
    their old pilot MSE is *not* reused. Saves a CSV on completion.
    """
    assert model_name in MODEL_ORDER
    basename = model_name.lower().replace(" ", "_").replace("-", "")
    old_path = os.path.join(OUTPUT_DIR, f"{basename}_trials.csv")
    previous = pd.read_csv(old_path)
    assert len(previous) == 8, f"Missing pilot results for {model_name}"
    pilot_best = previous.loc[previous["mean_cv_mse"].idxmin()]
    seed = json_safe(json.loads(pilot_best["parameters_json"]))

    model_optimizer = Optimizer(
        dimensions=SPAN_SPACE + MODEL_SPACES[model_name],
        base_estimator="GP", acq_func="EI", acq_optimizer="sampling",
        n_initial_points=5, random_state=42
    )
    trials = []
    for number in range(1, n_trials + 1):
        parameters = seed if number == 1 else json_safe(model_optimizer.ask())
        start = time.perf_counter()
        evaluation = evaluate_larger_model_cv(model_name, parameters)
        score = float(evaluation["mean_cv_mse"])
        assert np.isfinite(score)
        model_optimizer.tell(parameters, score)

        record = {
            "trial": number,
            "source": "Reevaluated pilot seed" if number == 1 else "Bayesian proposal",
            "parameters_json": json.dumps(parameters),
            "hyperparameters_json": json.dumps(evaluation["hyperparameters"],
                                                default=lambda x: x.item()),
            "s1": evaluation["spans"][0], "s2": evaluation["spans"][1],
            "s3": evaluation["spans"][2], "s4": evaluation["spans"][3],
            "fold1_mse": evaluation["fold_scores"][0],
            "fold2_mse": evaluation["fold_scores"][1],
            "fold3_mse": evaluation["fold_scores"][2],
            "mean_cv_mse": score,
            "feature_time_s": evaluation["feature_time_s"],
            "training_time_s": evaluation["training_time_s"],
            "inference_time_s": evaluation["inference_time_s"],
            "total_time_s": evaluation["total_time_s"],
            "convergence_warnings": evaluation["convergence_warnings"]
        }
        trials.append(record)
        print(f"{model_name} — {number}/{n_trials}, CV MSE={score:.4f}, "
              f"runtime={record['total_time_s']:.1f}s", flush=True)

    result = pd.DataFrame(trials)
    path = os.path.join(SEARCH_DIR, f"{basename}_trials.csv")
    result.to_csv(path, index=False)
    return result

# Completely unattended sequential execution
scaled_results = {}
for number, model_name in enumerate(MODEL_ORDER, 1):
    print(f"\n{'='*50}\nMODEL {number}/5 — {model_name}\n{'='*50}", flush=True)
    scaled_results[model_name] = run_scaled_bayesian(model_name, SCALED_TRIALS)
    print(f"PASS: {model_name} large-sample optimisation complete", flush=True)

print("PASS: All five larger-scale searches completed")


**Reproducibility note.** Numerical results and interpretations written earlier in this notebook describe a previous completed run. When using **Run all**, rely on newly generated output tables for the current measurements. Differences in library versions, timing, and fresh Bayesian proposals may change the numerical results. The notebook stops at Step 6.3D.7; final full-development retraining (Step 6.4) and untouched test-set evaluation (Step 7) are not yet implemented.

> **Archived interpretation only.** Numbers in the following source paragraph/table were reported by an earlier run; compare against the freshly executed cells and saved trial results. This is not a new experimental result.

### 4.9.12 Comparison of Larger-Scale Bayesian Optimisation Results (Step 6.3D.7C)

The preserved historical procedure executes **twelve larger-sample Gaussian-process Bayesian evaluations per model**, for exact RBF-SVR, k-NN, Random Forest, Extra Trees and MLP. Each search starts by **re-evaluating its fresh pilot winner** with the larger-sample objective; only the parameter vector, not the previous pilot MSE, is used.

Original training budgets are **320 observations/profile for SVR** and **640 observations/profile for the other four models**; validation uses 640 observations/profile for every model. The summary below must be read from this execution's new `scaled_results`, including each model's selected spans, hyperparameters, error, runtime and convergence warnings. These sampled-search scores are not directly equivalent to the full-development OLS score; no six-model final-test ranking is asserted here.

In [ ]:
# STEP 6.3D.7C — Automatically consolidate completed model results
rows = []
for model_name in MODEL_ORDER:
    trials = scaled_results[model_name]
    assert len(trials) == SCALED_TRIALS
    folds = trials[["fold1_mse", "fold2_mse", "fold3_mse"]]
    np.testing.assert_allclose(folds.mean(axis=1), trials["mean_cv_mse"])
    b = trials.loc[trials["mean_cv_mse"].idxmin()]
    rows.append({
        "Model": model_name,
        "Training samples/profile": TRAIN_SAMPLES[model_name],
        "Evaluations": len(trials),
        "Best CV MSE": b["mean_cv_mse"],
        "Span 1": int(b["s1"]), "Span 2": int(b["s2"]),
        "Span 3": int(b["s3"]), "Span 4": int(b["s4"]),
        "Total runtime (min)": trials["total_time_s"].sum() / 60,
        "Convergence warnings": int(trials["convergence_warnings"].sum())
    })
scaled_summary = pd.DataFrame(rows).sort_values("Best CV MSE")
scaled_summary.to_csv(os.path.join(SEARCH_DIR, "larger_scale_model_summary.csv"), index=False)
print("PASS: all five model searches and consistency checks completed")
display(scaled_summary.round(4))
print("Notes: OLS used all development observations; the other five models used")
print("nested subsamples. These CV results are NOT the final held-out 2 Hz test benchmark.")


### 4.9.13 Full-Development-Data Training Feasibility Assessment (Step 6.3D.8)
Following completion of the five larger-sample Bayesian searches, the selected model-specific EWMA/EWMS spans and regression hyperparameters were fixed. The purpose of this stage is computational feasibility, not further optimisation or the final comparison of prediction accuracy.
The same 55 development profiles and existing stratified grouped CV assignment were retained. For this preliminary scalability assessment, profiles assigned to Fold 1 were reserved for validation and the remaining 36 profiles were used for fitting. The validation subset comprised the same 64 nested observations from each of the 19 Fold-1 recording sessions (1,216 observations) at every training size. This intentionally differs from the previous 640-observation-per-profile three-fold optimisation objective and therefore produces diagnostic, not directly comparable, MSE values.
Temporal features were always calculated using each original complete 2 Hz recording history before the nested selection of observations. The training sample sizes increased from 320 to 1,280 samples per profile for SVR, and from 640 to 2,048 for the other four estimators. The selected MLP stopping procedure and training-only feature and target scaling were retained.
Fitting time, prediction time, approximate process peak resident memory, retained estimator-array size, and convergence warnings were recorded. A conservative runtime guard prevents expansion when the preceding fit exceeds 90 seconds. All 14 final-test profiles remain excluded. This experiment does not fit the five models on all 1,055,198 development observations, and no full-data feasibility claim will be made from these partial-size results alone.

In [ ]:
# STEP 6.3D.8A — Automatically profile final-model training feasibility
# Intended to run after Step 6.3D.7C in the Run-All notebook.
# No checkpoints are loaded and the final 14 test profiles are untouched.

import os
import json
import gc
import time
import threading
import warnings
import psutil
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

FEASIBILITY_DIR = str(BASE_DIR / 'training_feasibility')
os.makedirs(FEASIBILITY_DIR, exist_ok=True)

# SVR is initially limited because of kernel complexity.
# The other estimators are assessed using progressively larger samples.
FEASIBILITY_SIZES = {
    'SVR': [320, 640, 1280],
    'k-NN': [640, 1280, 2048],
    'Random Forest': [640, 1280, 2048],
    'Extra Trees': [640, 1280, 2048],
    'MLP': [640, 1280, 2048]
}
VAL_FOLD = 1                     # One existing development fold
VAL_SAMPLES_PER_PROFILE = 64     # Identical validation observations at each size
NEXT_SIZE_FIT_LIMIT_S = 90.0     # Avoid larger fits after a slow result

assert len(train_df) == 1055198
assert train_df['profile_id'].nunique() == 55
assert set(train_df.profile_id).isdisjoint(set(test_df.profile_id))
assert set(FEASIBILITY_SIZES) == set(MODEL_ORDER)
assert min(train_df.groupby('profile_id').size()) >= 2048

FEASIBILITY_FOLDS = profile_cv.set_index('profile_id')['cv_fold'].astype(int).to_dict()
assert set(FEASIBILITY_FOLDS.values()) == {1, 2, 3}

# Read best parameters directly from this notebook's completed 6.3D.7 results.
# This intentionally does not restore any checkpoint or depend on a past session.
def best_scaled_configuration(name):
    source = scaled_results[name]
    winner = source.loc[source['mean_cv_mse'].idxmin()]
    spans = [int(winner[f's{i}']) for i in range(1, 5)]
    hp = json.loads(winner['hyperparameters_json'])
    return spans, hp


class PeakProcessRAM:
    """Approximate process peak RSS (MiB) during one fit/predict cycle."""
    def __init__(self):
        self.process = psutil.Process()
        self.peak_mib = 0.0
        self.stop_event = threading.Event()

    def _sample(self):
        while not self.stop_event.wait(0.05):
            self.peak_mib = max(
                self.peak_mib,
                self.process.memory_info().rss / (1024 ** 2)
            )

    def __enter__(self):
        self.initial_mib = self.process.memory_info().rss / (1024 ** 2)
        self.peak_mib = self.initial_mib
        self.worker = threading.Thread(target=self._sample, daemon=True)
        self.worker.start()
        return self

    def __exit__(self, *args):
        self.stop_event.set()
        self.worker.join()
        self.peak_mib = max(
            self.peak_mib,
            self.process.memory_info().rss / (1024 ** 2)
        )


def approximate_fitted_model_mib(estimator):
    """Approximate stored model arrays, excluding Python object overhead."""
    arrays = []
    if hasattr(estimator, 'support_vectors_'):       # SVR
        arrays = [estimator.support_vectors_, estimator.dual_coef_]
    elif hasattr(estimator, '_fit_X'):                # k-NN
        arrays = [estimator._fit_X, estimator._y]
    elif hasattr(estimator, 'estimators_'):           # Tree ensembles
        for tree in estimator.estimators_:
            t = tree.tree_
            arrays.extend([t.children_left, t.children_right,
                           t.feature, t.threshold, t.value])
    elif hasattr(estimator, 'coefs_'):                # MLP
        arrays = list(estimator.coefs_) + list(estimator.intercepts_)
    return sum(np.asarray(a).nbytes for a in arrays) / (1024 ** 2)


def build_feasibility_arrays(model_name, spans, max_samples):
    """Compute full-history temporal features; retain only nested samples."""
    tic = time.perf_counter()
    x_chunks, y_chunks, id_chunks, rank_chunks = [], [], [], []

    for pid, profile in train_df.groupby('profile_id', sort=False):
        order = nested_sample_order(len(profile), max_samples)
        positions = np.sort(np.asarray(order, dtype=int))
        position_rank = {int(pos): rank for rank, pos in enumerate(order)}

        # Features are derived from the COMPLETE 2-Hz input history first.
        base = make_paper_inputs(profile)
        temporal = make_temporal_features(base, profile['profile_id'], spans)
        x_chunks.append(temporal.iloc[positions].to_numpy(dtype=np.float64))
        y_chunks.append(profile['pm'].iloc[positions].to_numpy(dtype=np.float64))
        id_chunks.append(np.full(len(positions), int(pid), dtype=np.int32))
        rank_chunks.append(np.array(
            [position_rank[int(pos)] for pos in positions], dtype=np.int32
        ))
        del temporal, base

    X = np.vstack(x_chunks)
    y = np.concatenate(y_chunks)
    pids = np.concatenate(id_chunks)
    ranks = np.concatenate(rank_chunks)
    assert X.shape == (55 * max_samples, 108)
    assert np.isfinite(X).all() and np.isfinite(y).all()
    folds = np.array([FEASIBILITY_FOLDS[int(pid)] for pid in pids], dtype=np.int8)
    return X, y, pids, ranks, folds, time.perf_counter() - tic


feasibility_records = []
feasibility_errors = []

for model_number, model_name in enumerate(MODEL_ORDER, start=1):
    sizes = FEASIBILITY_SIZES[model_name]
    spans, hp = best_scaled_configuration(model_name)
    print(f'\nMODEL {model_number}/5: {model_name} | spans={spans}', flush=True)
    X, y, pids, ranks, folds, feature_seconds = build_feasibility_arrays(
        model_name, spans, max(sizes)
    )
    val_mask = (folds == VAL_FOLD) & (ranks < VAL_SAMPLES_PER_PROFILE)
    assert val_mask.sum() == 19 * VAL_SAMPLES_PER_PROFILE
    X_val, y_val = X[val_mask], y[val_mask]
    prev_fit_seconds = None

    for n in sizes:
        if prev_fit_seconds is not None and prev_fit_seconds > NEXT_SIZE_FIT_LIMIT_S:
            print(f'  SKIP {n}/profile: previous fit exceeded the time limit')
            break

        train_mask = (folds != VAL_FOLD) & (ranks < n)
        train_ids = set(pids[train_mask])
        val_ids = set(pids[val_mask])
        assert train_ids.isdisjoint(val_ids)
        assert train_mask.sum() == 36 * n

        X_train, y_train = X[train_mask], y[train_mask]
        estimator = make_regressor(model_name, hp)
        if model_name == 'MLP':
            estimator.max_iter = 400
            estimator.early_stopping = True
            estimator.validation_fraction = 0.10
            estimator.n_iter_no_change = 15

        pipeline = TransformedTargetRegressor(
            regressor=Pipeline([
                ('scaler', StandardScaler()),
                ('model', estimator)
            ]),
            transformer=StandardScaler()
        )

        try:
            with PeakProcessRAM() as ram:
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter('always', ConvergenceWarning)
                    tic = time.perf_counter()
                    pipeline.fit(X_train, y_train)
                    fit_seconds = time.perf_counter() - tic
                    tic = time.perf_counter()
                    predictions = pipeline.predict(X_val)
                    predict_seconds = time.perf_counter() - tic

            assert np.isfinite(predictions).all()
            fitted = pipeline.regressor_.named_steps['model']
            warning_count = sum(issubclass(w.category, ConvergenceWarning)
                                for w in caught)
            mse = float(mean_squared_error(y_val, predictions))
            model_mib = approximate_fitted_model_mib(fitted)
            support_vectors = len(fitted.support_) if model_name == 'SVR' else np.nan

            feasibility_records.append({
                'Model': model_name,
                'Samples_per_profile': n,
                'Training_rows': len(y_train),
                'Fixed_validation_rows': len(y_val),
                'Feature_build_s': feature_seconds,
                'Fit_s': fit_seconds,
                'Predict_s': predict_seconds,
                'Process_peak_RSS_MiB': ram.peak_mib,
                'Fit_peak_increase_MiB': max(0., ram.peak_mib - ram.initial_mib),
                'Input_arrays_MiB': (X_train.nbytes + X_val.nbytes) / (1024 ** 2),
                'Estimator_array_MiB': model_mib,
                'Support_vectors': support_vectors,
                'Fold1_MSE_diagnostic': mse,
                'Convergence_warnings': warning_count,
                'Status': 'Completed'
            })
            pd.DataFrame(feasibility_records).to_csv(
                os.path.join(FEASIBILITY_DIR, 'feasibility_measurements.csv'),
                index=False
            )
            prev_fit_seconds = fit_seconds
            print(f'  {n}/profile: train={len(y_train):,}, '
                  f'fit={fit_seconds:.1f}s, predict={predict_seconds:.1f}s, '
                  f'peak RSS={ram.peak_mib:.0f} MiB, MSE={mse:.3f}', flush=True)
        except Exception as error:
            feasibility_errors.append({'Model': model_name,
                                       'Samples_per_profile': n,
                                       'Error': repr(error)})
            print(f'  STOP {model_name} at {n}: {error!r}', flush=True)
            break
        finally:
            del pipeline, estimator, X_train, y_train
            gc.collect()

    del X, y, pids, ranks, folds, X_val, y_val
    gc.collect()

feasibility_df = pd.DataFrame(feasibility_records)
assert not feasibility_df.empty, 'All training feasibility runs failed'
feasibility_df.to_csv(
    os.path.join(FEASIBILITY_DIR, 'feasibility_measurements.csv'), index=False
)
pd.DataFrame(feasibility_errors).to_csv(
    os.path.join(FEASIBILITY_DIR, 'feasibility_errors.csv'), index=False
)
print('\nCompleted feasibility fits:', len(feasibility_df))
print('Errors:', len(feasibility_errors))
display(feasibility_df.round(3))
print('Note: these are one-fold feasibility results, NOT final test-set scores.')


> **Archived interpretation only.** Numbers in the following source paragraph/table were reported by an earlier run; compare against the freshly executed cells and saved trial results. This is not a new experimental result.

### 4.9.14 Results and Limitations of Training-Feasibility Assessment
The following results summarise the largest completed training sample for each estimator and the observed computational scaling. Training and inference timings were collected on one grouped development fold, not on the untouched final test set. Reported peak RSS represents an approximate high-water measurement of the Python process and includes the runtime's existing allocations; it is not an isolated measurement of model memory. Model-array sizes omit Python-object overhead.
A computational budget for final development-set fitting will be selected only after reviewing these results. Extrapolation to all 1,055,198 development observations is inherently uncertain, particularly for kernel SVR and nearest-neighbour inference. Decisions and any limitations will be documented before Step 6.4.

In [ ]:
# STEP 6.3D.8B — Summarise observed scaling (no further model training)
import os
import matplotlib.pyplot as plt
import pandas as pd

summary = pd.read_csv(
    os.path.join(FEASIBILITY_DIR, 'feasibility_measurements.csv')
)
summary = summary.sort_values(['Model', 'Training_rows'])
assert set(summary['Model']).issubset(set(MODEL_ORDER))

print('Model-specific largest completed sample:')
summary_max = (
    summary.sort_values('Training_rows')
    .groupby('Model', as_index=False)
    .tail(1)
    .copy()
)
summary_max['Fraction_of_full_development'] = (
    summary_max['Training_rows'] / len(train_df)
)
display(summary_max[
    ['Model', 'Samples_per_profile', 'Training_rows',
     'Fraction_of_full_development', 'Fit_s', 'Predict_s',
     'Process_peak_RSS_MiB', 'Convergence_warnings']
].round(3))

# Independent figure for training time
plt.figure(figsize=(9, 5))
for name, group in summary.groupby('Model'):
    plt.plot(group['Training_rows'], group['Fit_s'], marker='o', label=name)
plt.xlabel('Training observations (one development fold)')
plt.ylabel('Fitting time (seconds)')
plt.title('Step 6.3D.8 — Model Training Scalability')
plt.legend()
plt.grid(True)
plt.show()

# Independent figure for approximate process RSS
plt.figure(figsize=(9, 5))
for name, group in summary.groupby('Model'):
    plt.plot(group['Training_rows'], group['Process_peak_RSS_MiB'],
             marker='o', label=name)
plt.xlabel('Training observations (one development fold)')
plt.ylabel('Approximate peak Python-process RSS (MiB)')
plt.title('Step 6.3D.8 — Observed Memory Usage')
plt.legend()
plt.grid(True)
plt.show()

summary_max.to_csv(
    os.path.join(FEASIBILITY_DIR, 'feasibility_largest_completed.csv'),
    index=False
)

print('PASS: Scalability assessment saved')
print('Caution: sampled-data benchmarks do not establish full-data feasibility.')
print('Step 6.4 must separately validate any proposed full-data fit.')


---
# Stage 9 — Original computationally efficient SVR experiments (restored)

The original Bayesian optimiser considered **exact RBF-SVR**, while this independent subsequent stage evaluated **matched RBF kernels** for exact SVR and Nyström, **direct LinearSVR**, regularisation and computational scaling. This stage retains its historical reference spans and original historical reference SVR C/epsilon. **It does not silently adopt a new Bayesian optimum**. All 14 held-out profiles remain excluded.

## Stage 9 adapter: restore only equivalent interfaces

The archival Stage 9 notebook relied on its local `development_df`, `stage5_data`, `fold_train_val`, `vectorised_temporal_sample`, `stage6_feature_names` names. The following adapter maps these names to the already recreated 55 development profiles, adds the original verified Stage 5 data preparation, and checks source-equivalent folds and feature ordering. It does not alter any Stage 9 search algorithm, hyperparameter, size or objective.

In [ ]:
from pathlib import Path
import sklearn, scipy, importlib.metadata as imd, json
# Source code above uses a string output directory. Stage 9 historical cells
# expect pathlib.Path; conversion changes only file-handling behaviour.
OUTPUT_DIR = Path(OUTPUT_DIR)
(OUTPUT_DIR/'figures').mkdir(parents=True, exist_ok=True)
development_df = df.loc[~df.profile_id.isin(test_ids)].copy()
heldout_df = df.loc[df.profile_id.isin(test_ids)].copy()  # split audit only
LOCKED_TEST_IDS = [13,16,23,24,29,42,44,47,53,56,62,67,71,78]
development_ids = sorted(set(development_df.profile_id))
assert sorted(test_ids) == LOCKED_TEST_IDS
assert len(development_df) == 1_055_198
assert len(heldout_df) == 275_618
assert len(profile_cv)==55
fold_train_val = {}
for fold in (1,2,3):
    group_validation = sorted(profile_cv.loc[profile_cv.cv_fold.eq(fold),'profile_id'].astype(int).tolist())
    group_training = sorted(set(development_ids)-set(group_validation))
    assert set(group_training).isdisjoint(group_validation)
    fold_train_val[fold]=(group_training,group_validation)
assert len(fold_train_val[1][0])==36 and len(fold_train_val[1][1])==19
print('PASS: stage 9 is linked to the original 55-profile, 3-fold development split')

### Historical Stage 5 equivalent and independent reference implementation

These next cells are copied from the Stage 5 verification record to establish an independent source-derived 640-observation/per-profile feature table. Stage 9 compares its regenerated sample positions and raw engineered inputs against this table. The original feature equations remain unchanged.

In [ ]:
import os, json, heapq, time, platform
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

STAGE5_DIR = OUTPUT_DIR / 'stage5'
STAGE5_DIR.mkdir(parents=True, exist_ok=True)

STAGE5_SAMPLE_SIZES = (80, 160, 320, 640)  # Nested sample construction and audit
STAGE5_MAX_SAMPLES = max(STAGE5_SAMPLE_SIZES)
STAGE5_EVALUATED_SIZES = (80, 160)  # Fixed verification scope
STAGE5_REPEATS = 1  # One fit per grouped fold (6 fits in total)

# SAME Stage-5 approach: use the winner of this Run All's actual
# eight-evaluation Gaussian-process SVR pilot, never an archived optimum.
assert len(all_results['SVR']) == N_TRIALS == 8
_stage5_best = all_results['SVR'].loc[all_results['SVR']['mean_cv_mse'].idxmin()]
STAGE5_SPANS = tuple(int(_stage5_best[f's{j}']) for j in (1, 2, 3, 4))
_stage5_hps = json.loads(_stage5_best['hyperparameters_json'])
STAGE5_SVR_HP = {
    'C': float(_stage5_hps['C']),
    'epsilon': float(_stage5_hps['epsilon']),
    'gamma': 'scale',
    'kernel': 'rbf'
}
STAGE5_CONFIG_SOURCE = 'current execution: Stage 6.3C SVR Gaussian-process pilot winner'

assert len(STAGE5_SPANS) == 4 and list(STAGE5_SPANS) == sorted(STAGE5_SPANS)
assert set(development_df.profile_id).isdisjoint(test_ids)
assert len(development_df) == 1_055_198 and development_df.profile_id.nunique() == 55
assert STAGE5_REPEATS in (1,3)

print('Fresh pilot-selected SVR spans:', STAGE5_SPANS)
print('Fresh pilot-selected SVR hyperparameters:', STAGE5_SVR_HP)
print('Stage 5 model-fit budgets:', STAGE5_EVALUATED_SIZES)
print('Stage 5 repetitions per size/fold:',STAGE5_REPEATS)
print('No final-test profiles used.')

In [ ]:
def nested_sample_order(profile_length: int, max_samples: int) -> list[int]:
    """Return nested sample positions in selection order (not time order).

    Repeatedly bisects the largest interval, preserving first and last
    positions. For every n in (80,160,320,640), the first n selections
    are a strict subset of the first 2n selections when those exist.
    """
    if not (2 <= max_samples <= profile_length):
        raise ValueError('Require 2 <= max_samples <= profile_length')
    selected = [0, profile_length - 1]
    intervals = [(-(profile_length - 1), 0, profile_length - 1)]
    while len(selected) < max_samples:
        _, left, right = heapq.heappop(intervals)
        midpoint = (left + right) // 2
        if midpoint in (left, right):
            continue
        selected.append(midpoint)
        for a,b in ((left,midpoint),(midpoint,right)):
            if b-a > 1:
                heapq.heappush(intervals, (-(b-a),a,b))
    assert len(selected) == len(set(selected)) == max_samples
    return selected

# Independent low-cost verification before transforming the 1.05M-row dataset.
example_order = nested_sample_order(2500, 640)
for n in STAGE5_SAMPLE_SIZES:
    assert len(set(example_order[:n])) == n
for low,high in zip(STAGE5_SAMPLE_SIZES[:-1],STAGE5_SAMPLE_SIZES[1:]):
    assert set(example_order[:low]).issubset(set(example_order[:high]))
assert example_order[:2] == [0, 2499]
print('PASS: Nesting and boundary tests on an independent 2,500-row example')

In [ ]:
def vectorised_temporal_sample(X_original, positions, spans):
    """Sample causal EWM features with 12-column vectorised Pandas filters.

    Returns exactly the same columns/values as make_temporal_features,
    but avoids looping separately over 96 individual feature series.
    Every EWM is computed over the full original profile *before* sampling.
    """
    sampled_columns = {col: X_original[col].iloc[positions].to_numpy()
                       for col in X_original.columns}
    for s in spans:
        ewm = X_original.ewm(span=int(s), adjust=True)
        historical_mean = ewm.mean().iloc[positions]
        historical_std = ewm.std(bias=True).iloc[positions]
        for col in X_original.columns:
            sampled_columns[f'{col}_ewma_s{s}'] = historical_mean[col].to_numpy()
            sampled_columns[f'{col}_ewms_s{s}'] = historical_std[col].to_numpy()
    return pd.DataFrame(sampled_columns,index=X_original.index[positions])

# Confirm vectorised and original implementations agree on real data.
_check_profile = next(iter(development_df.groupby('profile_id',sort=False)))[1].iloc[:450]
_check_raw = make_paper_inputs(_check_profile)
_check_positions = np.array([0,1,2,6,42,123,449])
_check_orig = make_temporal_features(_check_raw, _check_profile['profile_id'], STAGE5_SPANS)
_check_fast = vectorised_temporal_sample(_check_raw, _check_positions, STAGE5_SPANS)
np.testing.assert_allclose(
    _check_fast[_check_orig.columns].to_numpy(),
    _check_orig.iloc[_check_positions].to_numpy(),
    rtol=1e-9, atol=1e-9
)
print('PASS: Vectorised filters agree with Stage 2 reference on original sensor values')

started_features = time.perf_counter()
nested_parts = []

for profile_count, (profile_id, profile) in enumerate(
    development_df.groupby('profile_id', sort=False), start=1
):
    if not profile.index.is_monotonic_increasing:
        raise ValueError('Profile rows must be ordered chronologically')
    selection_order = nested_sample_order(len(profile), STAGE5_MAX_SAMPLES)
    positions = np.sort(np.asarray(selection_order, dtype=int))
    rank = {pos: i for i, pos in enumerate(selection_order)}

    X_original = make_paper_inputs(profile)
    sample = vectorised_temporal_sample(X_original, positions, STAGE5_SPANS)
    sample['pm_target'] = profile['pm'].iloc[positions].to_numpy(dtype=float)
    sample['profile_id'] = int(profile_id)
    sample['nest_rank'] = [rank[int(pos)] for pos in positions]
    nested_parts.append(sample)
    if profile_count % 10 == 0 or profile_count == 55:
        print(f'Processed {profile_count}/55 profiles',flush=True)

stage5_data = pd.concat(nested_parts, ignore_index=True)
stage5_feature_time_s = time.perf_counter() - started_features
stage5_feature_cols = [c for c in stage5_data.columns
    if c not in ('pm_target','profile_id','nest_rank')]
assert stage5_data.shape[0] == 55 * STAGE5_MAX_SAMPLES == 35_200
assert len(stage5_feature_cols) == 108
assert stage5_data.profile_id.nunique() == 55
assert stage5_data.nest_rank.between(0,STAGE5_MAX_SAMPLES-1).all()
assert np.isfinite(stage5_data[stage5_feature_cols+['pm_target']].to_numpy()).all()
assert set(stage5_data.profile_id).isdisjoint(test_ids)
for n in STAGE5_SAMPLE_SIZES:
    sample_counts = stage5_data.loc[stage5_data.nest_rank < n].groupby('profile_id').size()
    assert len(sample_counts) == 55 and sample_counts.eq(n).all()
    print(f'PASS: {n} per profile -> {55*n:,} selected development rows')
print(f'Feature preparation took {stage5_feature_time_s:.2f} s on this runtime')


In [ ]:
# Stage 5 (reference) and Stage 9 use the same immutable per-profile folds.
stage5_fold_lookup = profile_cv.set_index('profile_id')['cv_fold'].astype(int)
stage5_data['cv_fold'] = stage5_data['profile_id'].map(stage5_fold_lookup)
assert stage5_data.groupby('profile_id').size().eq(640).all()
assert stage5_data['cv_fold'].isin((1,2,3)).all()
assert set(stage5_data.profile_id).isdisjoint(test_ids)
print('PASS: Stage 5 independently reproduced sample dataset with fixed folds')

In [ ]:
def stage6_feature_names(columns, spans):
    """Reproduce the Stage 2 ordering: raw inputs then (EWMA,EWMS) per variable/span."""
    return list(columns) + [
        name for col in columns for span in spans
        for name in (f'{col}_ewma_s{span}', f'{col}_ewms_s{span}')
    ]


def compress_profile_ols(profile: pd.DataFrame, spans, chunk_size=4096):
    """Compress a single complete chronological profile's [1, X, y] using QR.

    Parameters
    ----------
    profile : pandas.DataFrame
        Exactly one recording session, unshuffled, in original sample order.
    spans : sequence[int]
        Exactly four positive, increasing temporal-smoothing spans (in samples).
    chunk_size : int
        Maximum raw observations in each temporary QR block.

    Returns
    -------
    dict
        n, feature_mean, feature_variance, R (augmented QR factor).
    """
    spans = tuple(map(int, spans))
    if len(spans) != 4 or any(a >= b for a, b in zip(spans, spans[1:])):
        raise ValueError('Need four increasing spans')
    if profile.profile_id.nunique() != 1 or not profile.index.is_monotonic_increasing:
        raise ValueError('Require one chronologically ordered recording profile')
    if chunk_size < 2:
        raise ValueError('chunk_size must be >=2')

    raw = make_paper_inputs(profile)
    positions = np.arange(len(raw), dtype=int)
    feature_frame = vectorised_temporal_sample(raw, positions, spans)
    feature_order = stage6_feature_names(raw.columns, spans)
    X = feature_frame.loc[:, feature_order].to_numpy(dtype=np.float64)
    y = profile['pm'].to_numpy(dtype=np.float64)
    n, p = X.shape
    assert p == 108 and n == len(y)
    assert np.isfinite(X).all() and np.isfinite(y).all()

    mu = X.mean(axis=0)
    variance = X.var(axis=0)    # Population variance, same convention as StandardScaler
    R_profile = None
    for start in range(0, n, chunk_size):
        stop = min(start + chunk_size, n)
        block = np.empty((stop-start, p+2), dtype=np.float64)
        block[:, 0] = 1.0
        block[:, 1:-1] = X[start:stop]
        block[:, -1] = y[start:stop]
        R_block = qr(block, mode='r', check_finite=False)[0][:p+2, :]
        if R_profile is None:
            R_profile = R_block
        else:
            R_profile = qr(np.vstack((R_profile, R_block)), mode='r',
                           check_finite=False)[0][:p+2, :]
    return {'n':n, 'mean':mu, 'var':variance, 'R':R_profile}

print('PASS: Profile QR compressor defined.')

### Numerical identity check: original and vectorised causal features

For a real profile, compute the same 108 ordered predictors using both the historical per-variable implementation and the vectorised implementation. Check multiple early and late source positions so that boundary conditions and long smoothing histories are exercised.

In [ ]:
_pid, _p = next(iter(development_df.groupby('profile_id',sort=False)))
_p = _p.iloc[:min(1000,len(_p))]
_raw = make_paper_inputs(_p)
_s = STAGE9_SPANS
_pos = np.array([0,1,2,8,47,299,len(_p)-1],dtype=int)
_ref=make_temporal_features(_raw, _p.profile_id, _s)
_vec=vectorised_temporal_sample(_raw,_pos,_s)
_order=stage6_feature_names(_raw.columns,_s)
assert _ref.shape[1]==_vec.shape[1]==108
np.testing.assert_allclose(_ref.iloc[_pos][_order].to_numpy(),_vec[_order].to_numpy(),rtol=1e-9,atol=1e-9)
print('PASS: 108-feature vectorised/historical equivalence on real motor measurements')

# Chapter 9 — Efficient SVR: accuracy, regularisation and resource trade-offs

**Independent development-only investigation.** This chapter uses the original Stage 9 experimental methods—exact RBF-SVR, Nyström RBF approximation and direct LinearSVR—comparing development-validation MSE, runtime, memory estimates, numerical convergence and sample-size-aware regularisation.

**Fresh-run requirement:** the reference `STAGE9_SPANS`, `STAGE9_SVR_HP['C']` and `STAGE9_SVR_HP['epsilon']` are read from **the 12-trial larger-sample SVR Bayesian search executed earlier in this Run All**, rather than fixed numerical values from the archived Stage 9. The actual new reference is printed by Stage 9A. Changing the reference *values* is required by fresh reconstruction; the matched-kernel formulation, relative candidate multipliers, folds and sample budgets remain original.

The 14 historical held-out profiles are excluded from Stage 9 development experiments. Notebook 01 alone performs full-development training and held-out inference. Exact RBF-SVR, Nyström and LinearSVR may have different accuracy and resource profiles; conclusions must follow this new run's actual outputs.

In [ ]:
# ============================================================
# STAGE 9A — Experimental configuration and invariants
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

# SAME Stage-9 methods, but a genuinely fresh reference from this Run All's
# 12-evaluation, larger-sample Gaussian-process SVR search. No old winners.
assert len(scaled_results['SVR']) == SCALED_TRIALS == 12
_stage9_source = scaled_results['SVR']
_stage9_eligible = _stage9_source.loc[
    _stage9_source['mean_cv_mse'].map(np.isfinite)
].copy()
assert len(_stage9_eligible) == 12, 'Fresh SVR search was incomplete'
_stage9_best = _stage9_eligible.loc[_stage9_eligible['mean_cv_mse'].idxmin()]
STAGE9_SPANS = tuple(int(_stage9_best[f's{j}']) for j in (1, 2, 3, 4))
_stage9_hp = json.loads(_stage9_best['hyperparameters_json'])
STAGE9_SVR_HP = {
    'C': float(_stage9_hp['C']),
    'epsilon': float(_stage9_hp['epsilon']),
    'kernel': 'rbf',
    'gamma': 'scale'
}
assert np.isclose(STAGE9_SVR_HP['C'], _stage9_hp['C'])
assert len(STAGE9_SPANS) == 4 and list(STAGE9_SPANS) == sorted(STAGE9_SPANS)
STAGE9_CONFIG_SOURCE = 'current execution: larger-sample 12-trial GP SVR winner'

STAGE9_SEED = 42

# Initial computational budget
STAGE9_SAMPLES_PER_PROFILE = 640
STAGE9_SCREENING_FOLD = 1

# Output directory
STAGE9_DIR = OUTPUT_DIR / "stage9_efficient_svr"
STAGE9_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Verify the original dataset and partition are unchanged
# ------------------------------------------------------------

assert len(development_df) == 1_055_198
assert len(development_ids) == 55
assert len(test_ids) == 14

assert sorted(test_ids) == LOCKED_TEST_IDS

assert set(development_ids).isdisjoint(set(test_ids))

assert set(profile_cv["profile_id"]) == set(development_ids)

assert set(profile_cv["cv_fold"]) == {1, 2, 3}

# Retrieve original fixed Fold 1 profile allocations
train_ids, val_ids = fold_train_val[STAGE9_SCREENING_FOLD]

assert len(train_ids) == 36
assert len(val_ids) == 19

assert set(train_ids).isdisjoint(val_ids)
assert set(train_ids).isdisjoint(test_ids)
assert set(val_ids).isdisjoint(test_ids)

expected_train_rows = len(train_ids) * STAGE9_SAMPLES_PER_PROFILE
expected_val_rows = len(val_ids) * STAGE9_SAMPLES_PER_PROFILE

# ------------------------------------------------------------
# Print methodological verification
# ------------------------------------------------------------

print("STAGE 9A — CONFIGURATION CHECK")
print("=" * 50)

print("Development profiles:", len(development_ids))
print("Held-out test profiles:", len(test_ids))

print("\nCross-validation:")
print("Screening fold:", STAGE9_SCREENING_FOLD)
print("Training profiles:", len(train_ids))
print("Validation profiles:", len(val_ids))

print("\nPlanned screening observations:")
print("Training:", expected_train_rows)
print("Validation:", expected_val_rows)

print("\nFresh Bayesian-selected reference SVR:")
print("C:", STAGE9_SVR_HP["C"])
print("Epsilon:", STAGE9_SVR_HP["epsilon"])
print("Temporal spans:", STAGE9_SPANS)

print("\nPASS: Stage 9A configuration verified.")

## 9B — Construction of a Common Training and Validation Dataset

From each of the 55 complete chronological development-profile histories, form the same 12 instantaneous engineering predictors and causal EWMA/EWMS features using the **fresh Bayesian-selected Stage 9 spans**, giving 108 predictors. Then select exactly 640 nested observations per profile while preserving raw row locations, nested sampling rank, target values and the locked group-fold assignments.

For Stage 9's Fold 1 screening, the 36 training profiles contribute 23,040 observations and the 19 validation profiles contribute 12,160 observations. Exact RBF-SVR, Nyström and direct LinearSVR receive the same input values and splits. Scaling, random Nyström landmarks and fitting must use training-fold inputs alone; no held-out test records or archived feature matrices are used.

In [ ]:

# ============================================================
# STAGE 9B.1 — Regenerate optimised temporal features
# ============================================================

import time
import numpy as np
import pandas as pd

start_time = time.perf_counter()

STAGE9_MAX_SAMPLES = STAGE9_SAMPLES_PER_PROFILE

# Original development-only fold assignments
stage9_fold_map = (
    profile_cv.set_index("profile_id")["cv_fold"]
    .astype(int)
    .to_dict()
)

stage9_parts = []

# ------------------------------------------------------------
# Process each complete chronological development profile
# ------------------------------------------------------------

for count, (pid, profile) in enumerate(
    development_df.groupby("profile_id", sort=False),
    start=1
):

    # Verify chronological order within the recording
    assert profile.index.is_monotonic_increasing

    # Reuse the validated nested sampling procedure
    selection_order = nested_sample_order(
        len(profile),
        STAGE9_MAX_SAMPLES
    )

    positions = np.sort(
        np.asarray(selection_order, dtype=int)
    )

    # Rank describes the order in which each sample
    # was added by the nested sampling algorithm
    rank_lookup = {
        int(position): rank
        for rank, position in enumerate(selection_order)
    }

    # Step 1: Construct instantaneous engineering predictors
    X_base = make_paper_inputs(profile)

    # Step 2: Generate temporal features from complete history,
    # then retrieve the selected rows
    X_sample = vectorised_temporal_sample(
        X_base,
        positions,
        STAGE9_SPANS
    )

    # Ensure consistent 108-column ordering
    ordered_features = stage6_feature_names(
        X_base.columns,
        STAGE9_SPANS
    )

    X_sample = X_sample.loc[:, ordered_features].copy()

    # Step 3: Attach target and audit metadata
    X_sample["pm_target"] = (
        profile["pm"].iloc[positions].to_numpy(dtype=np.float64)
    )

    X_sample["profile_id"] = int(pid)
    X_sample["raw_position"] = positions
    X_sample["source_row"] = X_sample.index.to_numpy()

    X_sample["nest_rank"] = [
        rank_lookup[int(position)]
        for position in positions
    ]

    X_sample["cv_fold"] = stage9_fold_map[int(pid)]

    stage9_parts.append(X_sample)

    if count % 10 == 0 or count == 55:
        print(f"Processed {count}/55 development profiles")

# ------------------------------------------------------------
# Combine the selected observations
# ------------------------------------------------------------

stage9_data = pd.concat(
    stage9_parts,
    ignore_index=True
)

stage9_feature_cols = stage6_feature_names(
    make_paper_inputs(development_df.iloc[:2]).columns,
    STAGE9_SPANS
)

stage9_feature_time = time.perf_counter() - start_time

print("\nSTAGE 9B — FEATURE CONSTRUCTION")
print("=" * 50)
print("Observations:", f"{len(stage9_data):,}")
print("Profiles:", stage9_data.profile_id.nunique())
print("Predictors:", len(stage9_feature_cols))
print("Temporal spans:", STAGE9_SPANS)
print("Construction time:", round(stage9_feature_time, 2), "s")


In [ ]:

# ============================================================
# STAGE 9B.2 — Numerical and methodological verification
# ============================================================

# 1. Feature dimensions and finite values
assert len(stage9_data) == 55 * 640
assert len(stage9_feature_cols) == 108

assert stage9_feature_cols == list(stage9_data.columns[:108])

assert np.isfinite(
    stage9_data[stage9_feature_cols + ["pm_target"]]
    .to_numpy(dtype=np.float64)
).all()

# 2. No leakage from held-out test profiles
assert set(stage9_data.profile_id).isdisjoint(test_ids)

# 3. Exactly 640 unique sampling ranks per profile
assert stage9_data.groupby("profile_id").size().eq(640).all()

assert (
    stage9_data.groupby("profile_id")
    .nest_rank.nunique()
    .eq(640)
    .all()
)

assert stage9_data["source_row"].is_unique

# 4. Verify original targets and sample positions
source_rows = stage9_data["source_row"].to_numpy(dtype=int)

np.testing.assert_allclose(
    stage9_data["pm_target"].to_numpy(),
    development_df.loc[source_rows, "pm"].to_numpy(),
    rtol=0,
    atol=0
)

# 5. Confirm identical sampling ranks and profile assignments
#    against Version 1.0 Stage 5.
comparison_cols = ["profile_id", "nest_rank", "cv_fold"]

assert stage9_data[comparison_cols].equals(
    stage5_data[comparison_cols]
), "Stage 9 sampling does not match the verified Stage 5."

# Verify the 12 instantaneous predictors are unchanged
base_cols = list(make_paper_inputs(development_df.iloc[:2]).columns)

np.testing.assert_allclose(
    stage9_data[base_cols].to_numpy(),
    stage5_data[base_cols].to_numpy(),
    rtol=0,
    atol=1e-12
)

# 6. Independent first-observation boundary condition
first_rows = stage9_data.groupby("profile_id", sort=False).head(1)

assert first_rows["raw_position"].eq(0).all()

for span in STAGE9_SPANS:
    np.testing.assert_allclose(
        first_rows[f"i_q_ewma_s{span}"],
        first_rows["i_q"],
        rtol=0,
        atol=1e-12
    )
    np.testing.assert_allclose(
        first_rows[f"i_q_ewms_s{span}"],
        0.0,
        rtol=0,
        atol=1e-12
    )

print("PASS: Features, sampled positions and targets verified")

# ------------------------------------------------------------
# 7. Prepare the original fixed Fold 1
# ------------------------------------------------------------

stage9_train_mask = (
    stage9_data["cv_fold"].ne(STAGE9_SCREENING_FOLD)
)

stage9_val_mask = (
    stage9_data["cv_fold"].eq(STAGE9_SCREENING_FOLD)
)

stage9_train = stage9_data.loc[stage9_train_mask]
stage9_val = stage9_data.loc[stage9_val_mask]

# Ensure the intended grouped split
assert stage9_train.profile_id.nunique() == 36
assert stage9_val.profile_id.nunique() == 19

assert len(stage9_train) == 23_040
assert len(stage9_val) == 12_160

assert set(stage9_train.profile_id).isdisjoint(
    set(stage9_val.profile_id)
)

# ------------------------------------------------------------
# 8. Establish common model input arrays
# ------------------------------------------------------------

X9_train = stage9_train[
    stage9_feature_cols
].to_numpy(dtype=np.float64)

y9_train = stage9_train[
    "pm_target"
].to_numpy(dtype=np.float64)

X9_val = stage9_val[
    stage9_feature_cols
].to_numpy(dtype=np.float64)

y9_val = stage9_val[
    "pm_target"
].to_numpy(dtype=np.float64)

assert X9_train.shape == (23_040, 108)
assert X9_val.shape == (12_160, 108)

assert y9_train.shape == (23_040,)
assert y9_val.shape == (12_160,)

print("\nSTAGE 9B — FINAL VERIFICATION")
print("=" * 50)

print("Training:")
print("  Profiles:", stage9_train.profile_id.nunique())
print("  X shape:", X9_train.shape)
print("  y shape:", y9_train.shape)

print("\nValidation:")
print("  Profiles:", stage9_val.profile_id.nunique())
print("  X shape:", X9_val.shape)
print("  y shape:", y9_val.shape)

print("\nPASS: Stage 9B completed.")
print("No estimators fitted; held-out test profiles untouched.")



## 9C — Training-Only Standardisation and Kernel Verification

### 9C.1 Objective

Verify the numerical representation used by exact RBF-SVR
and Nyström-approximated SVR before model fitting.

### Input standardisation

The 108 motor predictors have different physical units
and numerical magnitudes. StandardScaler is fitted only
on the Fold 1 training observations.

The same fitted transformation is applied to validation
observations without updating its parameters.

### Target standardisation

Permanent-magnet temperature is also standardised using
the training-set mean and standard deviation.

Predicted values will be inverse-transformed into °C
before calculating validation errors.

### Common RBF kernel parameter

For Scikit-learn SVR, gamma="scale" corresponds to:

    gamma = 1 / (n_features * X_scaled.var())

We explicitly calculate this parameter from the scaled
training matrix and use it in both exact RBF-SVR and
Nyström approximation.

### Nyström numerical verification

For a small development-training subset, calculate:

1. The exact RBF kernel matrix K.
2. The approximate kernel matrix K_approx = Z @ Z.T.
3. Their relative Frobenius error:

    E_K = ||K - K_approx||_F / ||K||_F

Increasing the number of basis components should generally
improve approximation quality, although independently
selected basis subsets need not produce monotonically
decreasing errors.

When every probe observation is included in the basis,
the approximation should reconstruct the original kernel
up to numerical precision.

No regression model is trained and no held-out test
observations are accessed in this stage.


In [ ]:

# ============================================================
# STAGE 9C.1 — Standardisation and shared RBF gamma
# ============================================================

import numpy as np

from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# 1. Fit input scaler on training observations ONLY
# ------------------------------------------------------------

stage9_x_scaler = StandardScaler()

X9_train_scaled = stage9_x_scaler.fit_transform(
    X9_train
)

X9_val_scaled = stage9_x_scaler.transform(
    X9_val
)

# ------------------------------------------------------------
# 2. Fit target scaler on training temperatures ONLY
# ------------------------------------------------------------

stage9_y_scaler = StandardScaler()

y9_train_scaled = (
    stage9_y_scaler.fit_transform(
        y9_train.reshape(-1, 1)
    ).ravel()
)

# The validation targets remain in physical units (°C).
# They are not used to fit any scaler.

# ------------------------------------------------------------
# 3. Verify shapes, finite values and transformations
# ------------------------------------------------------------

assert X9_train_scaled.shape == (23_040, 108)
assert X9_val_scaled.shape == (12_160, 108)

assert y9_train_scaled.shape == (23_040,)

assert np.isfinite(X9_train_scaled).all()
assert np.isfinite(X9_val_scaled).all()
assert np.isfinite(y9_train_scaled).all()

# Training input means should be approximately zero.
np.testing.assert_allclose(
    X9_train_scaled.mean(axis=0),
    np.zeros(108),
    atol=1e-9
)

# The standardised target should have zero mean
# and unit population standard deviation.
np.testing.assert_allclose(
    y9_train_scaled.mean(),
    0.0,
    atol=1e-10
)

np.testing.assert_allclose(
    y9_train_scaled.std(),
    1.0,
    atol=1e-10
)

# Verify that the target scaling is reversible.
reconstructed_y = (
    stage9_y_scaler.inverse_transform(
        y9_train_scaled.reshape(-1, 1)
    ).ravel()
)

np.testing.assert_allclose(
    reconstructed_y,
    y9_train,
    rtol=1e-12,
    atol=1e-9
)

# ------------------------------------------------------------
# 4. Compute the reference RBF kernel parameter
# ------------------------------------------------------------

assert STAGE9_SVR_HP["gamma"] == "scale"

STAGE9_GAMMA = 1.0 / (
    X9_train_scaled.shape[1] *
    X9_train_scaled.var()
)

assert np.isfinite(STAGE9_GAMMA)
assert STAGE9_GAMMA > 0

# For all nonconstant features successfully standardised,
# gamma should be close to 1/108.
constant_features = int(
    np.sum(stage9_x_scaler.var_ == 0)
)

# ------------------------------------------------------------
# 5. Report results
# ------------------------------------------------------------

print("STAGE 9C.1 — STANDARDISATION VERIFICATION")
print("=" * 55)

print("Training X:", X9_train_scaled.shape)
print("Validation X:", X9_val_scaled.shape)

print("\nTarget scaling:")
print("Training mean (°C):", stage9_y_scaler.mean_[0])
print("Training std (°C):", stage9_y_scaler.scale_[0])

print("\nRBF kernel:")
print("Reference gamma:", STAGE9_GAMMA)
print("1 / 108:", 1 / 108)
print("Constant input features:", constant_features)

print("\nPASS: Training-only standardisation verified.")


In [ ]:

# ============================================================
# STAGE 9C.2 — Verify Nyström kernel approximation
# ============================================================

import numpy as np
import pandas as pd

from sklearn.kernel_approximation import Nystroem
from sklearn.metrics.pairwise import rbf_kernel

# ------------------------------------------------------------
# 1. Select a small reproducible training-only probe
# ------------------------------------------------------------

rng = np.random.default_rng(STAGE9_SEED)

probe_indices = rng.choice(
    len(X9_train_scaled),
    size=192,
    replace=False
)

X9_probe = X9_train_scaled[probe_indices]

assert X9_probe.shape == (192, 108)

# ------------------------------------------------------------
# 2. Calculate the exact RBF kernel
# ------------------------------------------------------------

K_exact = rbf_kernel(
    X9_probe,
    X9_probe,
    gamma=STAGE9_GAMMA
)

assert K_exact.shape == (192, 192)

np.testing.assert_allclose(
    K_exact,
    K_exact.T,
    atol=1e-12
)

# ------------------------------------------------------------
# 3. Test different Nyström component counts
# ------------------------------------------------------------

component_counts = (24, 64, 128, 192)

kernel_records = []

for m in component_counts:

    transformer = Nystroem(
        kernel="rbf",
        gamma=STAGE9_GAMMA,
        n_components=m,
        random_state=STAGE9_SEED
    )

    # Fit the Nyström basis and transform the probe.
    Z = transformer.fit_transform(X9_probe)

    assert Z.shape == (192, m)
    assert np.isfinite(Z).all()

    # Reconstruct the approximate kernel.
    K_approx = Z @ Z.T

    relative_error = (
        np.linalg.norm(K_exact - K_approx, ord="fro")
        /
        np.linalg.norm(K_exact, ord="fro")
    )

    assert np.isfinite(relative_error)
    assert relative_error >= 0

    kernel_records.append({
        "Components": m,
        "Kernel_Relative_Error": float(relative_error)
    })

# ------------------------------------------------------------
# 4. Display results
# ------------------------------------------------------------

stage9_kernel_results = pd.DataFrame(kernel_records)

display(stage9_kernel_results)

# Full basis should reconstruct this small kernel
# to within numerical precision.
full_basis_error = stage9_kernel_results.loc[
    stage9_kernel_results["Components"].eq(192),
    "Kernel_Relative_Error"
].iloc[0]

assert full_basis_error < 1e-5, (
    "Full-basis reconstruction error is unexpectedly large."
)

print("\nPASS: Nyström kernel construction verified.")
print("Full-basis relative error:", full_basis_error)
print("No regressors fitted; held-out test data untouched.")


<a id="E08"></a>
> **Evidence E08 — Exact RBF-SVR, Nyström and direct LinearSVR comparisons.** Controlled development-only accuracy/cost screening and grouped-CV comparisons justify exploring LinearSVR as a scalable compromise, not as universally the most accurate SVR formulation.


## 9D — Controlled Screening of Four SVR Approaches

### Objective

Determine whether a Nyström approximation offers a useful
accuracy-versus-computational-cost advantage over exact
RBF-SVR when trained on identical observations.

### Compared methods

1. Exact RBF-SVR.
2. Nyström (128 components) + LinearSVR.
3. Nyström (256 components) + LinearSVR.
4. Direct LinearSVR.

### Experimental controls

All methods use:

- 23,040 identical Fold 1 training observations.
- 12,160 identical Fold 1 validation observations.
- The same 108 engineered motor predictors.
- Training-only feature and target standardisation.
- The same epsilon-insensitive loss convention.
- Fixed initial C and epsilon parameters.
- The same RBF gamma for both nonlinear approaches.

Nyström basis observations are selected only from
the Fold 1 training data.

### Evaluation

Record:

- Validation MSE (°C²).
- Validation RMSE (°C).
- Fitting time (seconds).
- Prediction time (seconds).
- Observed process-memory peak (GiB).
- Convergence warnings.

The recorded memory consumption is process-level
and sampled periodically, not exact model-only memory.

### Interpretation

This initial experiment is a screening comparison,
not the final model selection.

Different feature representations may require
different regularisation parameters. We will only
consider additional hyperparameter tuning after
reviewing the initial accuracy-cost trade-offs.

No held-out test profiles are used.


### Stage 9: the three distinct SVR computational approaches

Stage 9 is an efficiency study using the **freshly selected RBF-SVR configuration from the larger-sample Gaussian-process search completed earlier in this Run All**. We preserve the original matched-kernel comparison procedure, sample budgets, and penalty multipliers. Relative penalty grids are recomputed around the new reference rather than fixed at archived values.

**Exact RBF-SVR** uses the reference Gaussian kernel on standardised full-feature vectors. **Nyström SVR** approximates the kernel Gram matrix with \(K_{nm}K_{mm}^{\dagger}K_{mn}\), fits a lower-dimensional map with \(m\) components, and applies a linear epsilon-insensitive SVR. **Direct LinearSVR** uses a linear decision function in the original 108-feature space and therefore avoids constructing a kernel representation.

In the historical Stage 9 screening, the exact RBF and Nyström approaches use **matched `gamma`**, calculated from *training-only* standardised data. The source then evaluates component counts, convergence, regularisation sensitivity, nested training sizes of 640 and 1,280 observations per profile, grouped-fold error and computational costs. The full-million-row scaling of \(C\) is reported as an extrapolation, **not** a measured full-data Bayesian optimum.

In [ ]:

# ============================================================
# STAGE 9D — Four-method SVR screening
# ============================================================

import os
import gc
import time
import psutil
import warnings
import threading

import numpy as np
import pandas as pd

from sklearn.svm import SVR, LinearSVR
from sklearn.pipeline import Pipeline
from sklearn.kernel_approximation import Nystroem
from sklearn.exceptions import ConvergenceWarning

RUN_STAGE9D = True  # Change to True to run

# ------------------------------------------------------------
# 1. Reference hyperparameters
# ------------------------------------------------------------

C_REF = STAGE9_SVR_HP["C"]
EPSILON_REF = STAGE9_SVR_HP["epsilon"]

def make_linear_svr():
    return LinearSVR(
        C=C_REF,
        epsilon=EPSILON_REF,
        loss="epsilon_insensitive",
        dual=True,
        tol=1e-4,
        max_iter=3000,
        random_state=STAGE9_SEED
    )

def make_nystroem_svr(components):
    return Pipeline([
        (
            "nystroem",
            Nystroem(
                kernel="rbf",
                gamma=STAGE9_GAMMA,
                n_components=components,
                random_state=STAGE9_SEED,
                n_jobs=1
            )
        ),
        ("linear_svr", make_linear_svr())
    ])

stage9d_models = {
    "Exact RBF-SVR": SVR(
        kernel="rbf",
        C=C_REF,
        epsilon=EPSILON_REF,
        gamma=STAGE9_GAMMA,
        cache_size=512
    ),
    "Nystroem-128": make_nystroem_svr(128),
    "Nystroem-256": make_nystroem_svr(256),
    "Direct LinearSVR": make_linear_svr()
}

# ------------------------------------------------------------
# 2. Process-memory sampling helper
# ------------------------------------------------------------

def measure_peak_memory(stop_event, measurements):
    process = psutil.Process(os.getpid())

    while not stop_event.is_set():
        try:
            rss = process.memory_info().rss
            measurements.append(rss)
        except psutil.Error:
            break

        stop_event.wait(0.05)

# ------------------------------------------------------------
# 3. Fit, predict and record
# ------------------------------------------------------------

stage9d_records = []

if RUN_STAGE9D:

    assert X9_train_scaled.shape == (23040, 108)
    assert X9_val_scaled.shape == (12160, 108)

    assert np.isfinite(X9_train_scaled).all()
    assert np.isfinite(X9_val_scaled).all()

    for name, model in stage9d_models.items():

        print("\n" + "=" * 55)
        print("Training:", name)

        gc.collect()

        process = psutil.Process(os.getpid())
        rss_before = process.memory_info().rss

        memory_samples = [rss_before]
        stop_event = threading.Event()

        monitor = threading.Thread(
            target=measure_peak_memory,
            args=(stop_event, memory_samples),
            daemon=True
        )

        monitor.start()

        try:
            # Fit entire estimator, including Nyström mapping
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter(
                    "always",
                    ConvergenceWarning
                )

                t0 = time.perf_counter()

                model.fit(
                    X9_train_scaled,
                    y9_train_scaled
                )

                fit_seconds = time.perf_counter() - t0

            convergence_count = sum(
                issubclass(w.category, ConvergenceWarning)
                for w in caught
            )

            # Predict, including any Nyström transformation
            t1 = time.perf_counter()

            pred_scaled = model.predict(X9_val_scaled)

            pred_celsius = (
                stage9_y_scaler.inverse_transform(
                    pred_scaled.reshape(-1, 1)
                ).ravel()
            )

            predict_seconds = time.perf_counter() - t1

            assert np.isfinite(pred_celsius).all()

            mse = float(
                np.mean((y9_val - pred_celsius) ** 2)
            )

            rmse = float(np.sqrt(mse))

            status = (
                "convergence_warning"
                if convergence_count > 0
                else "success"
            )

            record = {
                "Method": name,
                "Status": status,
                "Training_Rows": len(X9_train_scaled),
                "Validation_Rows": len(X9_val_scaled),
                "MSE_C2": mse,
                "RMSE_C": rmse,
                "Fit_Seconds": fit_seconds,
                "Predict_Seconds": predict_seconds,
                "Convergence_Warnings": convergence_count
            }

            print("MSE (°C²):", round(mse, 4))
            print("RMSE (°C):", round(rmse, 4))
            print("Fit time (s):", round(fit_seconds, 3))
            print("Prediction time (s):",
                  round(predict_seconds, 3))
            print("Status:", status)

        except Exception as exc:
            record = {
                "Method": name,
                "Status": "failed",
                "Training_Rows": len(X9_train_scaled),
                "Validation_Rows": len(X9_val_scaled),
                "MSE_C2": np.nan,
                "RMSE_C": np.nan,
                "Fit_Seconds": np.nan,
                "Predict_Seconds": np.nan,
                "Convergence_Warnings": np.nan,
                "Error": repr(exc)
            }
            print("FAILED:", repr(exc))

        finally:
            stop_event.set()
            monitor.join(timeout=2)

            memory_samples.append(
                process.memory_info().rss
            )

            peak_rss = max(memory_samples)

            record["Peak_RSS_GiB"] = (
                peak_rss / (1024 ** 3)
            )

            record["Peak_RSS_Increase_GiB"] = (
                max(0, peak_rss - rss_before) / (1024 ** 3)
            )

        stage9d_records.append(record)

        # Save after every method, including failed attempts
        stage9d_results = pd.DataFrame(stage9d_records)

        stage9d_results.to_csv(
            STAGE9_DIR / "stage9d_fold1_screening.csv",
            index=False
        )

    print("\nSTAGE 9D — SCREENING RESULTS")
    display(stage9d_results)

else:
    print(
        "Stage 9D is disabled. "
        "Set RUN_STAGE9D = True to execute screening."
    )


In [ ]:

# ============================================================
# STAGE 9D — Accuracy and computational-cost summary
# ============================================================

assert "stage9d_results" in globals(), (
    "Run the Stage 9D screening experiment first."
)

summary_cols = [
    "Method",
    "Status",
    "MSE_C2",
    "RMSE_C",
    "Fit_Seconds",
    "Predict_Seconds",
    "Peak_RSS_GiB",
    "Convergence_Warnings"
]

stage9d_summary = stage9d_results[
    summary_cols
].copy()

display(
    stage9d_summary.sort_values(
        "MSE_C2",
        na_position="last"
    ).round(4)
)

# Plot observed accuracy versus fitting time
import matplotlib.pyplot as plt

valid = stage9d_summary[
    stage9d_summary["MSE_C2"].notna()
]

fig, ax = plt.subplots(figsize=(8, 5))

for _, row in valid.iterrows():
    ax.scatter(
        row["Fit_Seconds"],
        row["MSE_C2"],
        s=80
    )

    ax.annotate(
        row["Method"],
        (row["Fit_Seconds"], row["MSE_C2"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=9
    )

ax.set_xlabel("Model fitting time (seconds)")
ax.set_ylabel("Validation MSE (°C²)")
ax.set_title("Stage 9D: SVR Accuracy–Cost Screening")
ax.grid(alpha=0.25)

plt.show()



## 9E — Regularisation Sensitivity and Convergence

### Motivation

Stage 9D demonstrated that Nyström approximation greatly reduced
computational cost, but its validation MSE was substantially
higher than that of exact RBF-SVR.

Direct LinearSVR achieved comparable accuracy to exact RBF-SVR,
but produced a convergence warning.

These observations motivate a controlled investigation of
regularisation before increasing the kernel approximation size.

### Method

An inner profile-grouped training-validation split is constructed
using only the 36 training profiles from Stage 9D.

The input and target scalers are fitted again using only
the inner training observations.

Three C values are compared, spanning one order of magnitude
below and above the historical reference C.

The epsilon-insensitive margin, sampling strategy, temporal
features, Nyström component counts and random seed are fixed.

### Limitations

The inner validation results are diagnostic and will be used
to select promising regularisation configurations.

They are not directly comparable with Stage 9D's MSE values
because the validation motor profiles differ.

A later experiment will evaluate the selected approaches
using the original grouped validation framework.

No held-out test profiles are accessed.


In [ ]:

# ============================================================
# STAGE 9E.1 — Inner development-only grouped split
# ============================================================

import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

# These are the same 23,040 unscaled training rows
# previously used in Stage 9D.
inner_groups = stage9_train["profile_id"].to_numpy()

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=STAGE9_SEED
)

fit_idx, check_idx = next(
    splitter.split(
        X9_train,
        y9_train,
        groups=inner_groups
    )
)

# Verify group isolation
fit_groups = set(inner_groups[fit_idx])
check_groups = set(inner_groups[check_idx])

assert fit_groups.isdisjoint(check_groups)
assert len(fit_groups) == 27
assert len(check_groups) == 9

# Refit both scalers on inner-training rows ONLY
x_scaler_9e = StandardScaler()
y_scaler_9e = StandardScaler()

X9e_fit = x_scaler_9e.fit_transform(
    X9_train[fit_idx]
)

X9e_check = x_scaler_9e.transform(
    X9_train[check_idx]
)

y9e_fit = y_scaler_9e.fit_transform(
    y9_train[fit_idx].reshape(-1, 1)
).ravel()

y9e_check_celsius = y9_train[check_idx]

# Compute gamma from inner-training features
GAMMA_9E = 1.0 / (
    X9e_fit.shape[1] * X9e_fit.var()
)

assert X9e_fit.shape == (17280, 108)
assert X9e_check.shape == (5760, 108)

assert np.isfinite(X9e_fit).all()
assert np.isfinite(X9e_check).all()
assert np.isfinite(y9e_fit).all()

print("STAGE 9E — INNER SPLIT")
print("Inner training:", X9e_fit.shape)
print("Inner validation:", X9e_check.shape)
print("Training profiles:", len(fit_groups))
print("Validation profiles:", len(check_groups))
print("RBF gamma:", GAMMA_9E)
print("PASS: Grouped split and scaling verified.")


In [ ]:

# ============================================================
# STAGE 9E.2 — C sensitivity screening
# ============================================================

import time
import warnings
import pandas as pd

from sklearn.svm import LinearSVR
from sklearn.kernel_approximation import Nystroem
from sklearn.pipeline import Pipeline
from sklearn.exceptions import ConvergenceWarning

RUN_STAGE9E = True  # Set True to execute

C_GRID_9E = [
    STAGE9_SVR_HP["C"] / 10,
    STAGE9_SVR_HP["C"],
    STAGE9_SVR_HP["C"] * 10
]

METHODS_9E = {
    "Nystroem-128": 128,
    "Nystroem-256": 256,
    "Direct LinearSVR": None
}

def build_stage9e_model(method, components, C):

    linear_model = LinearSVR(
        C=float(C),
        epsilon=STAGE9_SVR_HP["epsilon"],
        loss="epsilon_insensitive",
        dual=True,
        tol=1e-4,
        max_iter=10000,
        random_state=STAGE9_SEED
    )

    if components is None:
        return linear_model

    return Pipeline([
        ("nystroem", Nystroem(
            kernel="rbf",
            gamma=GAMMA_9E,
            n_components=components,
            random_state=STAGE9_SEED,
            n_jobs=1
        )),
        ("linear_svr", linear_model)
    ])

if RUN_STAGE9E:

    stage9e_records = []

    for method, components in METHODS_9E.items():
        for C in C_GRID_9E:

            model = build_stage9e_model(
                method, components, C
            )

            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")

                fit_start = time.perf_counter()
                model.fit(X9e_fit, y9e_fit)
                fit_time = time.perf_counter() - fit_start

            convergence_warnings = sum(
                issubclass(w.category, ConvergenceWarning)
                for w in caught
            )

            prediction_start = time.perf_counter()

            predictions_scaled = model.predict(X9e_check)

            predictions_celsius = (
                y_scaler_9e.inverse_transform(
                    predictions_scaled.reshape(-1, 1)
                ).ravel()
            )

            prediction_time = (
                time.perf_counter() - prediction_start
            )

            mse = float(np.mean(
                (y9e_check_celsius - predictions_celsius) ** 2
            ))

            record = {
                "Method": method,
                "C": float(C),
                "Components": components,
                "MSE_C2": mse,
                "RMSE_C": float(np.sqrt(mse)),
                "Fit_Seconds": fit_time,
                "Predict_Seconds": prediction_time,
                "Convergence_Warnings": convergence_warnings,
                "Status": (
                    "convergence_warning"
                    if convergence_warnings else "success"
                )
            }

            stage9e_records.append(record)

            print(
                f"{method:18s} "
                f"C={C:.5g} "
                f"MSE={mse:.3f} "
                f"Fit={fit_time:.2f}s "
                f"Warnings={convergence_warnings}"
            )

            pd.DataFrame(stage9e_records).to_csv(
                STAGE9_DIR / "stage9e_C_sensitivity.csv",
                index=False
            )

    stage9e_results = pd.DataFrame(stage9e_records)

    print("\nSTAGE 9E RESULTS")
    display(stage9e_results.round(4))

else:
    print("Set RUN_STAGE9E = True to run the experiment.")



## 9F — Nyström Approximation Capacity

### Motivation

Stage 9E showed that direct LinearSVR achieved the lowest
inner-validation MSE, while Nyström-based approaches remained
substantially less accurate.

Two possible explanations for Nyström's behaviour are:

1. The approximate RBF feature representation lacks sufficient
   capacity to represent important nonlinear relationships.
2. The selected regularisation parameters are unsuitable
   for the approximate representation.

### Experiment

Increase the Nyström basis size from 256 to 512 components.

Compare two C values using exactly the same 17,280
inner-training observations and 5,760 inner-validation
observations from Stage 9E.

Retain the same temporal spans, training-only scaling,
epsilon, RBF gamma and random seed.

Measure validation MSE/RMSE, fitting time, prediction
time, convergence warnings and observed process memory.

### Interpretation

An improvement in validation MSE would provide evidence
that approximation capacity matters.

A substantial remaining accuracy gap would support
prioritising direct LinearSVR or computationally
bounded exact RBF-SVR.

These results are diagnostic and will not be treated as
final generalisation estimates.

No held-out test profiles are accessed.


In [ ]:

# ============================================================
# STAGE 9F — Nyström-512 capacity investigation
# ============================================================

import time
import warnings
import gc
import os
import threading

import numpy as np
import pandas as pd
import psutil

from sklearn.exceptions import ConvergenceWarning

RUN_STAGE9F = True  # Change to True to run

C_GRID_9F = [
    STAGE9_SVR_HP["C"],
    STAGE9_SVR_HP["C"] * 10
]

def stage9f_memory_monitor(stop_event, samples):
    process = psutil.Process(os.getpid())

    while not stop_event.is_set():
        try:
            samples.append(process.memory_info().rss)
        except psutil.Error:
            break
        stop_event.wait(0.05)

if RUN_STAGE9F:

    assert X9e_fit.shape == (17280, 108)
    assert X9e_check.shape == (5760, 108)

    stage9f_records = []

    for C in C_GRID_9F:

        model = build_stage9e_model(
            method="Nystroem-512",
            components=512,
            C=C
        )

        gc.collect()

        process = psutil.Process(os.getpid())
        baseline_rss = process.memory_info().rss

        memory_samples = [baseline_rss]
        stop_event = threading.Event()

        monitor = threading.Thread(
            target=stage9f_memory_monitor,
            args=(stop_event, memory_samples),
            daemon=True
        )

        monitor.start()

        try:
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")

                t0 = time.perf_counter()
                model.fit(X9e_fit, y9e_fit)
                fit_seconds = time.perf_counter() - t0

            warning_count = sum(
                issubclass(w.category, ConvergenceWarning)
                for w in caught
            )

            t1 = time.perf_counter()
            predictions_scaled = model.predict(X9e_check)

            predictions_celsius = (
                y_scaler_9e.inverse_transform(
                    predictions_scaled.reshape(-1, 1)
                ).ravel()
            )
            predict_seconds = time.perf_counter() - t1

            mse = float(np.mean(
                (y9e_check_celsius - predictions_celsius) ** 2
            ))

            record = {
                "Method": "Nystroem-512",
                "Components": 512,
                "C": float(C),
                "MSE_C2": mse,
                "RMSE_C": float(np.sqrt(mse)),
                "Fit_Seconds": fit_seconds,
                "Predict_Seconds": predict_seconds,
                "Convergence_Warnings": warning_count,
                "Status": (
                    "convergence_warning"
                    if warning_count else "success"
                )
            }

        except Exception as exc:
            record = {
                "Method": "Nystroem-512",
                "Components": 512,
                "C": float(C),
                "MSE_C2": np.nan,
                "RMSE_C": np.nan,
                "Fit_Seconds": np.nan,
                "Predict_Seconds": np.nan,
                "Convergence_Warnings": np.nan,
                "Status": "failed",
                "Error": repr(exc)
            }

        finally:
            stop_event.set()
            monitor.join(timeout=2)

            memory_samples.append(process.memory_info().rss)

            peak_rss = max(memory_samples)

            record["Peak_Process_RSS_GiB"] = (
                peak_rss / 1024**3
            )

            record["RSS_Increase_GiB"] = (
                max(0, peak_rss - baseline_rss) / 1024**3
            )

        stage9f_records.append(record)

        pd.DataFrame(stage9f_records).to_csv(
            STAGE9_DIR / "stage9f_nystroem_512.csv",
            index=False
        )

        print(
            f"C={C:.6f}, "
            f"MSE={record['MSE_C2']:.4f}, "
            f"Fit={record['Fit_Seconds']:.2f}s, "
            f"Status={record['Status']}"
        )

        del model
        gc.collect()

    stage9f_results = pd.DataFrame(stage9f_records)

    # Compare against existing Stage 9E results
    comparison_columns = [
        "Method", "Components", "C",
        "MSE_C2", "RMSE_C",
        "Fit_Seconds", "Predict_Seconds",
        "Convergence_Warnings", "Status"
    ]

    stage9f_comparison = pd.concat(
        [
            stage9e_results[comparison_columns],
            stage9f_results[comparison_columns]
        ],
        ignore_index=True
    )

    print("\nSTAGE 9F — COMBINED INNER-VALIDATION RESULTS")

    display(
        stage9f_comparison.sort_values(
            "MSE_C2",
            na_position="last"
        ).round(4)
    )

else:
    print("Set RUN_STAGE9F = True to execute.")



## 9G — Confirmation on the Original Grouped Validation Fold

### Objective

Confirm whether the promising configurations identified in the
inner development experiment retain their performance when
evaluated on the original Fold 1 validation profiles.

### Motivation

Stages 9E–9F indicated that:

- Direct LinearSVR achieved substantially lower validation
  MSE than the tested Nyström approximations.
- Its best tested penalty was C = 0.0156175.
- Increasing the Nyström basis to 512 components did not
  improve predictive accuracy in the tested configurations.
- Nyström-256 with C = 1.56175 was the strongest Nyström
  configuration in the inner-validation experiment.

### Experimental procedure

Train the selected LinearSVR and Nyström configurations
using the original 23,040 Fold 1 training observations.

Use the 12,160 original Fold 1 validation observations
for evaluation.

Compare these results against the existing exact RBF-SVR
reference from Stage 9D.

All configurations use the same:

- 108 engineered predictors.
- Training-only standardisation.
- Grouped training and validation profiles.
- Causal temporal features.
- Validation targets in degrees Celsius.

### Evaluation

Measure validation MSE, RMSE, fitting time,
prediction time, convergence and observed process memory.

### Limitation

This is a confirmation on one development-validation fold.
It is not sufficient to establish generalisation across
the full 55-profile development dataset.

Full grouped cross-validation will follow if the
results justify continuing.

The 14 final-test profiles remain untouched.


In [ ]:

# ============================================================
# STAGE 9G — Confirm selected approaches on Fold 1
# ============================================================

import gc
import os
import time
import warnings
import threading

import numpy as np
import pandas as pd
import psutil

from sklearn.svm import LinearSVR
from sklearn.pipeline import Pipeline
from sklearn.kernel_approximation import Nystroem
from sklearn.exceptions import ConvergenceWarning

RUN_STAGE9G = True  # Set True to execute

# ------------------------------------------------------------
# 1. Selected configurations from Stage 9E
# ------------------------------------------------------------

C_LINEAR = STAGE9_SVR_HP["C"] / 10
C_NYSTROEM = STAGE9_SVR_HP["C"] * 10

def stage9g_linear_model(C):
    return LinearSVR(
        C=float(C),
        epsilon=STAGE9_SVR_HP["epsilon"],
        loss="epsilon_insensitive",
        dual=True,
        tol=1e-4,
        max_iter=10000,
        random_state=STAGE9_SEED
    )

models_9g = {
    "Direct LinearSVR (selected)": stage9g_linear_model(
        C_LINEAR
    ),

    "Nystroem-256 (selected)": Pipeline([
        ("nystroem", Nystroem(
            kernel="rbf",
            gamma=STAGE9_GAMMA,
            n_components=256,
            random_state=STAGE9_SEED,
            n_jobs=1
        )),
        ("linear_svr", stage9g_linear_model(C_NYSTROEM))
    ])
}

# ------------------------------------------------------------
# 2. Process-memory monitor
# ------------------------------------------------------------

def stage9g_memory_monitor(stop_event, samples):
    process = psutil.Process(os.getpid())

    while not stop_event.is_set():
        try:
            samples.append(process.memory_info().rss)
        except psutil.Error:
            break

        stop_event.wait(0.05)

# ------------------------------------------------------------
# 3. Train, predict and evaluate
# ------------------------------------------------------------

if RUN_STAGE9G:

    assert X9_train_scaled.shape == (23040, 108)
    assert X9_val_scaled.shape == (12160, 108)

    stage9g_records = []

    for name, model in models_9g.items():

        print("\nTraining:", name)

        gc.collect()

        process = psutil.Process(os.getpid())
        initial_rss = process.memory_info().rss

        samples = [initial_rss]
        stop_event = threading.Event()

        monitor = threading.Thread(
            target=stage9g_memory_monitor,
            args=(stop_event, samples),
            daemon=True
        )
        monitor.start()

        try:
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")

                t0 = time.perf_counter()

                model.fit(
                    X9_train_scaled,
                    y9_train_scaled
                )

                fit_seconds = time.perf_counter() - t0

            warning_count = sum(
                issubclass(w.category, ConvergenceWarning)
                for w in caught
            )

            t1 = time.perf_counter()

            pred_scaled = model.predict(X9_val_scaled)

            pred_celsius = (
                stage9_y_scaler.inverse_transform(
                    pred_scaled.reshape(-1, 1)
                ).ravel()
            )

            predict_seconds = time.perf_counter() - t1

            assert np.isfinite(pred_celsius).all()

            mse = float(np.mean(
                (y9_val - pred_celsius) ** 2
            ))

            record = {
                "Method": name,
                "C": (
                    C_LINEAR if "Direct" in name
                    else C_NYSTROEM
                ),
                "Training_Rows": len(X9_train_scaled),
                "Validation_Rows": len(X9_val_scaled),
                "MSE_C2": mse,
                "RMSE_C": float(np.sqrt(mse)),
                "Fit_Seconds": fit_seconds,
                "Predict_Seconds": predict_seconds,
                "Convergence_Warnings": warning_count,
                "Status": (
                    "convergence_warning"
                    if warning_count else "success"
                )
            }

        except Exception as exc:
            record = {
                "Method": name,
                "MSE_C2": np.nan,
                "RMSE_C": np.nan,
                "Fit_Seconds": np.nan,
                "Predict_Seconds": np.nan,
                "Convergence_Warnings": np.nan,
                "Status": "failed",
                "Error": repr(exc)
            }

        finally:
            stop_event.set()
            monitor.join(timeout=2)

            samples.append(process.memory_info().rss)

            peak_rss = max(samples)

            record["Peak_RSS_GiB"] = peak_rss / 1024**3
            record["RSS_Increase_GiB"] = (
                max(0, peak_rss - initial_rss) / 1024**3
            )

        stage9g_records.append(record)

        pd.DataFrame(stage9g_records).to_csv(
            STAGE9_DIR / "stage9g_fold1_confirmation.csv",
            index=False
        )

        print(
            f"MSE={record['MSE_C2']:.4f}, "
            f"Fit={record['Fit_Seconds']:.3f}s, "
            f"Status={record['Status']}"
        )

        del model
        gc.collect()

    stage9g_results = pd.DataFrame(stage9g_records)

    display(stage9g_results.round(4))

else:
    print("Set RUN_STAGE9G = True to run confirmation.")


In [ ]:

# ============================================================
# STAGE 9G — Same-fold comparison
# ============================================================

assert "stage9g_results" in globals()
assert "stage9d_results" in globals()

reference = stage9d_results.loc[
    stage9d_results["Method"].eq("Exact RBF-SVR")
].copy()

assert len(reference) == 1
assert reference.iloc[0]["Status"] == "success"

comparison_columns = [
    "Method",
    "Status",
    "MSE_C2",
    "RMSE_C",
    "Fit_Seconds",
    "Predict_Seconds",
    "Peak_RSS_GiB"
]

stage9g_comparison = pd.concat([
    reference[comparison_columns],
    stage9g_results[comparison_columns]
], ignore_index=True)

stage9g_comparison = stage9g_comparison.sort_values(
    "MSE_C2",
    na_position="last"
)

print("STAGE 9G — ORIGINAL FOLD 1 COMPARISON")
print("=" * 55)

display(stage9g_comparison.round(4))

stage9g_comparison.to_csv(
    STAGE9_DIR / "stage9g_final_comparison.csv",
    index=False
)



## 9H — Three-Fold Grouped Validation of Efficient SVR

### Objective

Determine whether direct LinearSVR retains competitive predictive
accuracy and computational efficiency across all three original
development-validation folds.

### Methods

1. Exact RBF-SVR with the newly Bayesian-selected
   configuration.
2. Direct LinearSVR with C selected separately inside each
   outer training partition.

Nyström is not advanced because Stages 9D–9G demonstrated a
substantial accuracy deficit for the tested configurations.

### Nested regularisation selection

For each outer fold:

1. Exclude the outer validation profiles.
2. Divide the remaining training profiles into two
   inner grouped folds.
3. Evaluate three candidate LinearSVR penalties.
4. Select the candidate with the lowest mean inner MSE
   among converged fits.
5. Refit LinearSVR on all outer training observations.
6. Evaluate it on the outer validation profiles.

Candidate C values:

- Fresh Bayesian-selected C / 100
- Fresh Bayesian-selected C / 10
- Fresh Bayesian-selected C

The smaller candidate extends Stage 9E's search because the
previous best LinearSVR C was the lowest value tested.

### Computational controls

All methods use the same 640 sampled observations per profile,
108 predictors and causal temporal feature definitions.

Feature and target scaling are fitted independently within
each training partition.

### Evaluation

Record outer-fold MSE, RMSE, fitting time, prediction time,
sampled process memory, convergence warnings and selected C.

These are development-stage validation results, not an
independent final-test assessment.

No held-out test profiles are accessed.


In [ ]:

# ============================================================
# STAGE 9H — Nested grouped validation
# ============================================================

import os
import gc
import time
import warnings
import threading
import psutil
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR, LinearSVR
from sklearn.exceptions import ConvergenceWarning

RUN_STAGE9H = True  # Set True to execute

C_REF = STAGE9_SVR_HP["C"]
EPS_REF = STAGE9_SVR_HP["epsilon"]

C_GRID_9H = [
    C_REF / 100,
    C_REF / 10,
    C_REF
]

def build_linear_9h(C):
    return LinearSVR(
        C=float(C),
        epsilon=EPS_REF,
        loss="epsilon_insensitive",
        dual=True,
        tol=1e-4,
        max_iter=10000,
        random_state=STAGE9_SEED
    )

def scale_partition(X_fit, y_fit, X_check):
    xs = StandardScaler()
    ys = StandardScaler()

    X_fit_s = xs.fit_transform(X_fit)
    X_check_s = xs.transform(X_check)
    y_fit_s = ys.fit_transform(
        y_fit.reshape(-1, 1)
    ).ravel()

    return X_fit_s, X_check_s, y_fit_s, ys

def monitor_memory_9h(stop, values):
    process = psutil.Process(os.getpid())
    while not stop.wait(0.05):
        try:
            values.append(process.memory_info().rss)
        except psutil.Error:
            break

def evaluate_outer_9h(
    model, X_fit, y_fit, X_check, y_check, y_scaler
):
    gc.collect()

    process = psutil.Process(os.getpid())
    initial_rss = process.memory_info().rss
    memory_values = [initial_rss]
    stop = threading.Event()

    monitor = threading.Thread(
        target=monitor_memory_9h,
        args=(stop, memory_values),
        daemon=True
    )
    monitor.start()

    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")

            start = time.perf_counter()
            model.fit(X_fit, y_fit)
            fit_seconds = time.perf_counter() - start

        warning_count = sum(
            issubclass(w.category, ConvergenceWarning)
            for w in caught
        )

        start = time.perf_counter()

        prediction_scaled = model.predict(X_check)

        prediction_celsius = (
            y_scaler.inverse_transform(
                prediction_scaled.reshape(-1, 1)
            ).ravel()
        )

        predict_seconds = time.perf_counter() - start

        assert np.isfinite(prediction_celsius).all()

        squared_errors = (
            y_check - prediction_celsius
        ) ** 2

    finally:
        stop.set()
        monitor.join(timeout=2)
        memory_values.append(process.memory_info().rss)

    mse = float(np.mean(squared_errors))

    return {
        "MSE_C2": mse,
        "RMSE_C": float(np.sqrt(mse)),
        "SSE_C2": float(np.sum(squared_errors)),
        "Validation_Rows": len(y_check),
        "Fit_Seconds": fit_seconds,
        "Predict_Seconds": predict_seconds,
        "Peak_RSS_GiB": max(memory_values) / 1024**3,
        "Convergence_Warnings": warning_count,
        "Status": (
            "convergence_warning"
            if warning_count else "success"
        )
    }

if RUN_STAGE9H:

    stage9h_inner_records = []
    stage9h_outer_records = []

    for outer_fold in (1, 2, 3):

        print(f"\n{'=' * 55}")
        print(f"OUTER FOLD {outer_fold}")

        outer_fit_df = stage9_data.loc[
            stage9_data.cv_fold.ne(outer_fold)
        ]
        outer_val_df = stage9_data.loc[
            stage9_data.cv_fold.eq(outer_fold)
        ]

        assert set(outer_fit_df.profile_id).isdisjoint(
            set(outer_val_df.profile_id)
        )
        assert set(outer_fit_df.profile_id).isdisjoint(
            test_ids
        )
        assert set(outer_val_df.profile_id).isdisjoint(
            test_ids
        )

        X_fit_raw = outer_fit_df[
            stage9_feature_cols
        ].to_numpy(dtype=np.float64)

        y_fit_raw = outer_fit_df[
            "pm_target"
        ].to_numpy(dtype=np.float64)

        X_val_raw = outer_val_df[
            stage9_feature_cols
        ].to_numpy(dtype=np.float64)

        y_val_raw = outer_val_df[
            "pm_target"
        ].to_numpy(dtype=np.float64)

        groups = outer_fit_df.profile_id.to_numpy()

        # -----------------------------------------------
        # Inner grouped CV for LinearSVR regularisation
        # -----------------------------------------------

        inner_cv = GroupKFold(n_splits=2)
        inner_scores = []

        for C in C_GRID_9H:

            C_errors = []
            C_warning_counts = []

            for inner_fold, (idx_fit, idx_check) in enumerate(
                inner_cv.split(X_fit_raw, y_fit_raw, groups),
                start=1
            ):

                assert set(groups[idx_fit]).isdisjoint(
                    set(groups[idx_check])
                )

                Xi_fit, Xi_check, yi_fit, yi_scaler = (
                    scale_partition(
                        X_fit_raw[idx_fit],
                        y_fit_raw[idx_fit],
                        X_fit_raw[idx_check]
                    )
                )

                inner_model = build_linear_9h(C)

                with warnings.catch_warnings(
                    record=True
                ) as caught:
                    warnings.simplefilter("always")

                    start = time.perf_counter()
                    inner_model.fit(Xi_fit, yi_fit)
                    inner_fit_seconds = (
                        time.perf_counter() - start
                    )

                count = sum(
                    issubclass(w.category, ConvergenceWarning)
                    for w in caught
                )

                predicted = (
                    yi_scaler.inverse_transform(
                        inner_model.predict(
                            Xi_check
                        ).reshape(-1, 1)
                    ).ravel()
                )

                inner_mse = float(np.mean(
                    (
                        y_fit_raw[idx_check] - predicted
                    ) ** 2
                ))

                stage9h_inner_records.append({
                    "Outer_Fold": outer_fold,
                    "Inner_Fold": inner_fold,
                    "C": C,
                    "MSE_C2": inner_mse,
                    "Fit_Seconds": inner_fit_seconds,
                    "Convergence_Warnings": count
                })

                C_errors.append(inner_mse)
                C_warning_counts.append(count)

                del inner_model

            inner_scores.append({
                "C": C,
                "Mean_MSE": np.mean(C_errors),
                "Warnings": sum(C_warning_counts)
            })

        # Select only converged configurations
        eligible = [
            score for score in inner_scores
            if score["Warnings"] == 0
        ]

        if not eligible:
            raise RuntimeError(
                "No converged LinearSVR candidate."
            )

        best = min(
            eligible,
            key=lambda row: row["Mean_MSE"]
        )

        selected_C = float(best["C"])

        print("Selected LinearSVR C:", selected_C)
        print("Inner CV MSE:", round(best["Mean_MSE"], 4))

        # -----------------------------------------------
        # Refit common outer-fold preprocessing
        # -----------------------------------------------

        Xo_fit, Xo_val, yo_fit, yo_scaler = (
            scale_partition(
                X_fit_raw,
                y_fit_raw,
                X_val_raw
            )
        )

        outer_gamma = 1.0 / (
            Xo_fit.shape[1] * Xo_fit.var()
        )

        assert Xo_fit.shape[1] == 108

        models = {
            "Exact RBF-SVR": SVR(
                kernel="rbf",
                C=C_REF,
                epsilon=EPS_REF,
                gamma=outer_gamma,
                cache_size=512
            ),
            "Direct LinearSVR": build_linear_9h(
                selected_C
            )
        }

        # -----------------------------------------------
        # Common outer-fold evaluation
        # -----------------------------------------------

        for method, model in models.items():

            result = evaluate_outer_9h(
                model,
                Xo_fit,
                yo_fit,
                Xo_val,
                y_val_raw,
                yo_scaler
            )

            result.update({
                "Outer_Fold": outer_fold,
                "Method": method,
                "C": (
                    C_REF if method == "Exact RBF-SVR"
                    else selected_C
                ),
                "Training_Rows": len(Xo_fit),
                "Training_Profiles":
                    outer_fit_df.profile_id.nunique(),
                "Validation_Profiles":
                    outer_val_df.profile_id.nunique()
            })

            stage9h_outer_records.append(result)

            print(
                f"{method}: "
                f"MSE={result['MSE_C2']:.3f}, "
                f"fit={result['Fit_Seconds']:.2f}s, "
                f"status={result['Status']}"
            )

            del model

        # Save after every completed outer fold
        pd.DataFrame(
            stage9h_inner_records
        ).to_csv(
            STAGE9_DIR / "stage9h_inner_cv.csv",
            index=False
        )

        pd.DataFrame(
            stage9h_outer_records
        ).to_csv(
            STAGE9_DIR / "stage9h_outer_cv.csv",
            index=False
        )

    stage9h_results = pd.DataFrame(
        stage9h_outer_records
    )

    print("\nSTAGE 9H — OUTER CV RESULTS")
    display(stage9h_results.round(4))

else:
    print("Set RUN_STAGE9H = True to execute.")


In [ ]:

# ============================================================
# STAGE 9H — Three-fold performance summary
# ============================================================

assert "stage9h_results" in globals()

# Do not silently include failed or nonconverged fits.
assert (
    stage9h_results["Status"].eq("success")
).all(), "Investigate convergence before summarising."

assert (
    stage9h_results.groupby("Method")
    .Outer_Fold.nunique().eq(3)
).all()

summary = (
    stage9h_results.groupby("Method")
    .agg(
        Mean_MSE_C2=("MSE_C2", "mean"),
        Std_MSE_C2=("MSE_C2", "std"),
        Mean_RMSE_C=("RMSE_C", "mean"),
        Total_Fit_S=("Fit_Seconds", "sum"),
        Total_Predict_S=("Predict_Seconds", "sum"),
        Max_Observed_RSS_GiB=("Peak_RSS_GiB", "max"),
        Total_SSE_C2=("SSE_C2", "sum"),
        Total_Validation_Rows=("Validation_Rows", "sum")
    )
)

# Pooled out-of-fold error, weighted by observations
summary["Pooled_MSE_C2"] = (
    summary["Total_SSE_C2"]
    / summary["Total_Validation_Rows"]
)

summary["Pooled_RMSE_C"] = np.sqrt(
    summary["Pooled_MSE_C2"]
)

print("STAGE 9H — THREE-FOLD SUMMARY")
display(
    summary.sort_values("Mean_MSE_C2").round(4)
)

summary.to_csv(
    STAGE9_DIR / "stage9h_three_fold_summary.csv"
)

print("\nSelected LinearSVR penalties by fold:")
display(
    stage9h_results.loc[
        stage9h_results.Method.eq("Direct LinearSVR"),
        ["Outer_Fold", "C", "MSE_C2", "Fit_Seconds"]
    ].round(5)
)


**Grouped-CV inference.** Compare **fold-level** errors, not just a pooled mean. LinearSVR trades some prediction accuracy for much faster inference; these development-fold scores motivated further scaling tests, not a post-test change in the final model.

### Figure 9.1 — Fold-to-fold accuracy of exact RBF-SVR versus direct LinearSVR

One model may be faster but less accurate on average; fold-level bars expose whether the gap is consistent. **All data here belong to the development profiles**, and LinearSVR's internal `C` selection is reported separately.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

_pivot = stage9h_results.pivot(index="Outer_Fold",columns="Method",values="MSE_C2")
_candidates = [k for k in ["Exact RBF-SVR","Direct LinearSVR"] if k in _pivot.columns]
assert len(_candidates)==2, f"Unexpected Stage 9H labels: {list(_pivot.columns)}"
_x = np.arange(len(_pivot))
_width=.36
fig,ax=plt.subplots(figsize=(8.7,4.1))
for j,label in enumerate(_candidates):
    ax.bar(_x+(j-.5)*_width,_pivot[label],width=_width,label=label)
ax.set(xticks=_x,xticklabels=[f"Fold {f}" for f in _pivot.index],
       ylabel="Development validation MSE (°C²)",
       title="Three grouped folds: nonlinear kernel accuracy versus linear efficiency")
ax.legend(frameon=False)
ax.grid(axis="y",alpha=.2)
fig.tight_layout()
fig.savefig(Path(OUTPUT_DIR)/"figures"/"chapter9_foldwise_svr_mse.png",dpi=180)
plt.show()


<a id="E09"></a>
> **Evidence E09 — LinearSVR regularisation and sample-size sensitivity.** Bounded C searches and 640/1280-per-profile sensitivity motivated a sample-size-aware extrapolation for the final full-data LinearSVR C; that extrapolation is a heuristic, not a full-data CV optimum.


## 9I — Final Hyperparameter Selection for LinearSVR

### Objective

Select a fixed regularisation parameter for the computationally
efficient LinearSVR formulation, using grouped cross-validation
on development data only.

### Motivation

Stage 9H established that exact RBF-SVR achieved lower mean
validation MSE than LinearSVR, but LinearSVR required
substantially less fitting and prediction time.

LinearSVR has therefore been selected as the practical
computational alternative, accepting the documented
accuracy-efficiency trade-off.

However, the preferred regularisation parameter varied
between the three development folds.

### Methodology

Five candidate C values are evaluated using the original
three profile-grouped development folds.

For each fold:

1. Select the training and validation motor profiles.
2. Fit feature and target scalers using training rows only.
3. Fit LinearSVR for each candidate C.
4. Predict permanent-magnet temperature on validation profiles.
5. Inverse-transform predictions into degrees Celsius.
6. Calculate validation MSE and RMSE.
7. Record fitting time, prediction time, convergence diagnostics
   and sampled process-memory consumption.

All candidates use identical observations within each fold.

The epsilon, loss function, temporal spans, sampling strategy
and solver settings remain fixed.

### Hyperparameter selection

Select the converged candidate with the lowest mean MSE
across the three validation folds.

The pooled MSE, variation between folds and computational
costs are also reported.

### Methodological limitation

The development CV scores are used for hyperparameter
selection and should not be treated as an independent
generalisation estimate.

The 14 held-out test profiles remain untouched.

The selected hyperparameter will subsequently be checked
for computational feasibility at larger training sizes.


In [ ]:

# ============================================================
# STAGE 9I — LinearSVR hyperparameter selection
# ============================================================

import gc
import json
import os
import time
import warnings
import threading

import numpy as np
import pandas as pd
import psutil

from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVR
from sklearn.exceptions import ConvergenceWarning

RUN_STAGE9I = True  # Change to True to execute

C_REF = STAGE9_SVR_HP["C"]
EPS_REF = STAGE9_SVR_HP["epsilon"]

# Logarithmically spaced C values.
C_GRID_9I = C_REF * np.array([
    0.01,
    0.0316227766,
    0.1,
    0.3162277660,
    1.0
])

def make_stage9i_model(C):
    return LinearSVR(
        C=float(C),
        epsilon=EPS_REF,
        loss="epsilon_insensitive",
        dual=True,
        tol=1e-4,
        max_iter=10000,
        random_state=STAGE9_SEED
    )

# Sample process RSS during model fitting and prediction.
def stage9i_monitor(stop_event, samples):
    process = psutil.Process(os.getpid())

    while not stop_event.wait(0.05):
        try:
            samples.append(process.memory_info().rss)
        except psutil.Error:
            break

if RUN_STAGE9I:

    assert len(stage9_data) == 55 * 640
    assert stage9_data.profile_id.nunique() == 55

    assert set(stage9_data.profile_id).isdisjoint(test_ids)
    assert set(stage9_data.cv_fold.unique()) == {1, 2, 3}

    stage9i_records = []

    for fold in (1, 2, 3):

        print("\n" + "=" * 60)
        print(f"OUTER DEVELOPMENT FOLD {fold}")

        # ------------------------------------------------
        # 1. Select original grouped development fold
        # ------------------------------------------------

        train_df = stage9_data.loc[
            stage9_data.cv_fold.ne(fold)
        ]

        val_df = stage9_data.loc[
            stage9_data.cv_fold.eq(fold)
        ]

        assert set(train_df.profile_id).isdisjoint(
            set(val_df.profile_id)
        )

        assert set(train_df.profile_id).isdisjoint(test_ids)
        assert set(val_df.profile_id).isdisjoint(test_ids)

        X_train_raw = train_df[
            stage9_feature_cols
        ].to_numpy(dtype=np.float64)

        X_val_raw = val_df[
            stage9_feature_cols
        ].to_numpy(dtype=np.float64)

        y_train_raw = train_df[
            "pm_target"
        ].to_numpy(dtype=np.float64)

        y_val_raw = val_df[
            "pm_target"
        ].to_numpy(dtype=np.float64)

        # ------------------------------------------------
        # 2. Fit scalers on training observations ONLY
        # ------------------------------------------------

        x_scaler = StandardScaler()
        y_scaler = StandardScaler()

        X_train = x_scaler.fit_transform(X_train_raw)
        X_val = x_scaler.transform(X_val_raw)

        y_train = y_scaler.fit_transform(
            y_train_raw.reshape(-1, 1)
        ).ravel()

        assert X_train.shape[1] == 108
        assert X_val.shape[1] == 108

        print("Training observations:", len(X_train))
        print("Validation observations:", len(X_val))

        # ------------------------------------------------
        # 3. Compare each regularisation candidate
        # ------------------------------------------------

        for C in C_GRID_9I:

            gc.collect()

            model = make_stage9i_model(C)

            process = psutil.Process(os.getpid())
            rss_initial = process.memory_info().rss

            memory_samples = [rss_initial]
            stop_event = threading.Event()

            monitor = threading.Thread(
                target=stage9i_monitor,
                args=(stop_event, memory_samples),
                daemon=True
            )

            monitor.start()

            try:
                with warnings.catch_warnings(
                    record=True
                ) as caught:

                    warnings.simplefilter(
                        "always",
                        ConvergenceWarning
                    )

                    t0 = time.perf_counter()

                    model.fit(X_train, y_train)

                    fit_seconds = (
                        time.perf_counter() - t0
                    )

                warning_count = sum(
                    issubclass(
                        w.category,
                        ConvergenceWarning
                    )
                    for w in caught
                )

                # ----------------------------------------
                # 4. Validation prediction
                # ----------------------------------------

                t1 = time.perf_counter()

                predictions_scaled = model.predict(X_val)

                predictions_celsius = (
                    y_scaler.inverse_transform(
                        predictions_scaled.reshape(-1, 1)
                    ).ravel()
                )

                predict_seconds = (
                    time.perf_counter() - t1
                )

                assert np.isfinite(predictions_celsius).all()

                squared_errors = (
                    y_val_raw - predictions_celsius
                ) ** 2

                mse = float(np.mean(squared_errors))

                record = {
                    "Fold": fold,
                    "C": float(C),
                    "Training_Rows": len(X_train),
                    "Validation_Rows": len(X_val),
                    "MSE_C2": mse,
                    "RMSE_C": float(np.sqrt(mse)),
                    "SSE_C2": float(np.sum(squared_errors)),
                    "Fit_Seconds": fit_seconds,
                    "Predict_Seconds": predict_seconds,
                    "Iterations": int(
                        np.max(np.atleast_1d(model.n_iter_))
                    ),
                    "Convergence_Warnings": warning_count,
                    "Status": (
                        "convergence_warning"
                        if warning_count else "success"
                    )
                }

            except Exception as exc:
                # Unexpected errors must not be treated
                # as legitimate validation results.
                print(f"FAILED: Fold={fold}, C={C}")
                raise

            finally:
                stop_event.set()
                monitor.join(timeout=2)

                memory_samples.append(
                    process.memory_info().rss
                )

            peak_rss = max(memory_samples)

            record["Peak_Process_RSS_GiB"] = (
                peak_rss / 1024**3
            )

            record["RSS_Increase_GiB"] = (
                max(0, peak_rss-rss_initial) / 1024**3
            )

            stage9i_records.append(record)

            print(
                f"C={C:.6f} | "
                f"MSE={record['MSE_C2']:.4f} | "
                f"Fit={fit_seconds:.3f}s | "
                f"Warnings={warning_count}"
            )

            # Save each successful trial immediately.
            pd.DataFrame(stage9i_records).to_csv(
                STAGE9_DIR / "stage9i_linear_C_trials.csv",
                index=False
            )

            del model
            gc.collect()

    stage9i_trials = pd.DataFrame(stage9i_records)

    assert len(stage9i_trials) == 15

    print("\nSTAGE 9I — ALL TRIAL RESULTS")
    display(stage9i_trials.round(5))

else:
    print("Set RUN_STAGE9I = True to run the C search.")


In [ ]:

# ============================================================
# STAGE 9I — Hyperparameter selection and summary
# ============================================================

assert "stage9i_trials" in globals()
assert len(stage9i_trials) == 15

# ------------------------------------------------------------
# 1. Aggregate across three development folds
# ------------------------------------------------------------

stage9i_summary = (
    stage9i_trials
    .groupby("C", as_index=False)
    .agg(
        Mean_MSE_C2=("MSE_C2", "mean"),
        Std_MSE_C2=("MSE_C2", "std"),
        Mean_RMSE_C=("RMSE_C", "mean"),
        Total_Fit_Seconds=("Fit_Seconds", "sum"),
        Total_Predict_Seconds=("Predict_Seconds", "sum"),
        Total_SSE_C2=("SSE_C2", "sum"),
        Total_Validation_Rows=("Validation_Rows", "sum"),
        Convergence_Warnings=(
            "Convergence_Warnings", "sum"
        ),
        Completed_Folds=("Fold", "nunique"),
        Max_Process_RSS_GiB=(
            "Peak_Process_RSS_GiB", "max"
        )
    )
)

stage9i_summary["Pooled_MSE_C2"] = (
    stage9i_summary["Total_SSE_C2"]
    / stage9i_summary["Total_Validation_Rows"]
)

stage9i_summary["Pooled_RMSE_C"] = np.sqrt(
    stage9i_summary["Pooled_MSE_C2"]
)

# ------------------------------------------------------------
# 2. Select only completed, converged configurations
# ------------------------------------------------------------

eligible = stage9i_summary.loc[
    stage9i_summary.Completed_Folds.eq(3)
    & stage9i_summary.Convergence_Warnings.eq(0)
].copy()

assert len(eligible) > 0, (
    "No fully converged configuration. "
    "Investigate solver convergence before selection."
)

eligible = eligible.sort_values(
    ["Mean_MSE_C2", "Total_Fit_Seconds", "C"]
)

STAGE9I_SELECTED_C = float(eligible.iloc[0]["C"])

# ------------------------------------------------------------
# 3. Report results
# ------------------------------------------------------------

print("STAGE 9I — LINEARSVR HYPERPARAMETER SUMMARY")
print("=" * 60)

display(
    stage9i_summary.sort_values("Mean_MSE_C2").round(5)
)

print("\nSELECTED REGULARISATION")
print("C =", STAGE9I_SELECTED_C)
print(
    "Mean development CV MSE =",
    round(float(eligible.iloc[0]["Mean_MSE_C2"]), 4),
    "°C²"
)

print(
    "Pooled development CV MSE =",
    round(float(eligible.iloc[0]["Pooled_MSE_C2"]), 4),
    "°C²"
)

print("\nPer-fold performance of selected C:")

display(
    stage9i_trials.loc[
        stage9i_trials.C.eq(STAGE9I_SELECTED_C),
        [
            "Fold",
            "C",
            "MSE_C2",
            "RMSE_C",
            "Fit_Seconds",
            "Convergence_Warnings"
        ]
    ].round(5)
)

# ------------------------------------------------------------
# 4. Save the selection and experimental record
# ------------------------------------------------------------

stage9i_summary.to_csv(
    STAGE9_DIR / "stage9i_linear_C_summary.csv",
    index=False
)

selection_manifest = {
    "stage": "9I",
    "method": "LinearSVR",
    "selected_C": STAGE9I_SELECTED_C,
    "epsilon": float(EPS_REF),
    "loss": "epsilon_insensitive",
    "dual": True,
    "max_iter": 10000,
    "tol": 1e-4,
    "temporal_spans": list(STAGE9_SPANS),
    "features": 108,
    "observations_per_profile": 640,
    "cv_folds": 3,
    "selection_metric": "mean_fold_MSE_C2",
    "test_profiles_accessed": False
}

with open(
    STAGE9_DIR / "stage9i_selection_manifest.json",
    "w"
) as f:
    json.dump(selection_manifest, f, indent=2)

print("\nPASS: Stage 9I completed.")
print("Selected C saved with reproducibility metadata.")


**Hyperparameter selection note.** The best observed `C` in this bounded grid should be reported together with runtime and fold variation. CV is used for selection; its score is not an independent post-selection generalisation estimate.

### Figure 9.2 — Regularisation choice as an accuracy–cost trade-off

Each point represents one `C` evaluated over all three grouped development folds. Left and lower is preferable for both resource use and error, but *only* among settings that converged. The selected hyperparameter is documented independently of the final-test score.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

_cplot=stage9i_summary.sort_values("C")
fig,ax=plt.subplots(figsize=(8.7,4.6))
ax.plot(_cplot["Total_Fit_Seconds"],_cplot["Mean_MSE_C2"],"o-",color="#4D7994")
for r in _cplot.itertuples():
    ax.annotate(f"C={r.C:.4g}",(r.Total_Fit_Seconds,r.Mean_MSE_C2),
                xytext=(5,7),textcoords="offset points",fontsize=8)
_selected=_cplot.iloc[(abs(_cplot.C-STAGE9I_SELECTED_C)).argmin()]
ax.scatter([_selected.Total_Fit_Seconds],[_selected.Mean_MSE_C2],s=115,
           facecolors="none",edgecolors="#C86D3C",linewidths=2,label="Selected by mean fold MSE")
ax.set(xlabel="Total three-fold fitting time (s)",ylabel="Mean grouped-CV MSE (°C²)",
       title="LinearSVR: regularisation and compute cost")
ax.grid(alpha=.2)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(Path(OUTPUT_DIR)/"figures"/"chapter9_regularisation_tradeoff.png",dpi=180)
plt.show()



## 9J — LinearSVR Training-Size Scalability

### Objective

Investigate how doubling the training observations affects
LinearSVR prediction accuracy, fitting time, inference time,
memory consumption and numerical convergence.

### Motivation

Stage 9I recomputes its `C` grid using the **fresh Stage 9 RBF reference**. The original Stage 9J protocol evaluates two pre-specified grid positions, `C_GRID_9I[3]` and `C_GRID_9I[4]`, with 1,280 training samples/profile. Their values and observed scores must be read from this execution's trial tables; historical numeric results are not inherited.

### Method

1. Increase training samples from 640 to 1280 per profile.
2. Preserve the original 640 validation samples per profile.
3. Maintain the same three grouped development folds.
4. Generate temporal features from full chronological histories.
5. Fit scalers on each training partition only.
6. Compare the two candidate C values.

### Evaluation

Record mean validation MSE/RMSE, fitting time,
prediction time, convergence status, iterations and
sampled process memory.

Compare the results with the existing Stage 9I
640-observation training baseline.

### Limitation

The preferred C may change as the training dataset grows.
Consequently, the present comparison is a scalability
investigation, not final proof of optimal full-data
regularisation.

The 14 held-out test profiles remain untouched.


In [ ]:

# ============================================================
# STAGE 9J.1 — Expand training sampling to 1280/profile
# ============================================================

import time
import numpy as np
import pandas as pd

STAGE9J_BUDGET = 1280

t0 = time.perf_counter()
stage9j_parts = []

fold_lookup = (
    profile_cv.set_index("profile_id")["cv_fold"]
    .astype(int).to_dict()
)

for pid, profile in development_df.groupby(
    "profile_id", sort=False
):
    order = nested_sample_order(
        len(profile), STAGE9J_BUDGET
    )

    assert len(order) == STAGE9J_BUDGET
    assert len(set(order)) == STAGE9J_BUDGET

    positions = np.sort(np.asarray(order, dtype=int))

    rank_lookup = {
        int(pos): rank
        for rank, pos in enumerate(order)
    }

    base = make_paper_inputs(profile)

    sample = vectorised_temporal_sample(
        base, positions, STAGE9_SPANS
    )

    sample = sample.loc[
        :, stage9_feature_cols
    ].copy()

    sample["pm_target"] = (
        profile["pm"].iloc[positions].to_numpy()
    )
    sample["profile_id"] = int(pid)
    sample["source_row"] = sample.index.to_numpy()
    sample["nest_rank"] = [
        rank_lookup[int(p)] for p in positions
    ]
    sample["cv_fold"] = fold_lookup[int(pid)]

    stage9j_parts.append(sample)

stage9j_data = pd.concat(
    stage9j_parts, ignore_index=True
)

assert len(stage9j_data) == 55 * 1280
assert stage9j_data.profile_id.nunique() == 55
assert set(stage9j_data.profile_id).isdisjoint(test_ids)

assert np.isfinite(
    stage9j_data[
        stage9_feature_cols + ["pm_target"]
    ].to_numpy(dtype=np.float64)
).all()

# Verify that the original Stage 9B training/validation
# sample is exactly preserved by the larger nested sample.
cols = [
    "profile_id", "source_row",
    "nest_rank", "pm_target"
]

old = (
    stage9_data
    .sort_values(["profile_id", "nest_rank"])
    .reset_index(drop=True)
)

new = (
    stage9j_data.loc[stage9j_data.nest_rank.lt(640)]
    .sort_values(["profile_id", "nest_rank"])
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    old[cols],
    new[cols],
    check_exact=True
)

np.testing.assert_allclose(
    old[stage9_feature_cols].to_numpy(),
    new[stage9_feature_cols].to_numpy(),
    rtol=0,
    atol=1e-10
)

print("PASS: Original 640 observations/profile preserved.")
print("Expanded observations:", len(stage9j_data))
print("Features:", len(stage9_feature_cols))
print("Preparation time:", round(time.perf_counter()-t0, 2), "s")
print("Held-out test profiles untouched.")


In [ ]:

# ============================================================
# STAGE 9J.2 — LinearSVR scaling comparison
# ============================================================

import gc
import os
import time
import threading
import warnings
import psutil
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVR
from sklearn.exceptions import ConvergenceWarning

RUN_STAGE9J = True  # Change to True to execute

# Reuse the EXACT two C values from Stage 9I.
C_GRID_9J = [
    float(C_GRID_9I[3]),
    float(C_GRID_9I[4])
]

def stage9j_memory_monitor(stop, samples):
    process = psutil.Process(os.getpid())

    while not stop.wait(0.05):
        try:
            samples.append(process.memory_info().rss)
        except psutil.Error:
            break

if RUN_STAGE9J:

    stage9j_records = []

    for fold in (1, 2, 3):

        train_df = stage9j_data.loc[
            stage9j_data.cv_fold.ne(fold)
        ]

        val_df = stage9j_data.loc[
            stage9j_data.cv_fold.eq(fold)
            & stage9j_data.nest_rank.lt(640)
        ]

        assert set(train_df.profile_id).isdisjoint(
            set(val_df.profile_id)
        )

        X_train_raw = train_df[
            stage9_feature_cols
        ].to_numpy(dtype=np.float64)

        X_val_raw = val_df[
            stage9_feature_cols
        ].to_numpy(dtype=np.float64)

        y_train_raw = train_df[
            "pm_target"
        ].to_numpy(dtype=np.float64)

        y_val_raw = val_df[
            "pm_target"
        ].to_numpy(dtype=np.float64)

        x_scaler = StandardScaler()
        y_scaler = StandardScaler()

        X_train = x_scaler.fit_transform(X_train_raw)
        X_val = x_scaler.transform(X_val_raw)

        y_train = y_scaler.fit_transform(
            y_train_raw.reshape(-1, 1)
        ).ravel()

        print(f"\nFOLD {fold}")
        print("Training rows:", len(X_train))
        print("Validation rows:", len(X_val))

        for C in C_GRID_9J:

            gc.collect()

            model = LinearSVR(
                C=float(C),
                epsilon=STAGE9_SVR_HP["epsilon"],
                loss="epsilon_insensitive",
                dual=True,
                tol=1e-4,
                max_iter=10000,
                random_state=STAGE9_SEED
            )

            process = psutil.Process(os.getpid())
            baseline = process.memory_info().rss

            memory_samples = [baseline]
            stop = threading.Event()

            monitor = threading.Thread(
                target=stage9j_memory_monitor,
                args=(stop, memory_samples),
                daemon=True
            )
            monitor.start()

            try:
                with warnings.catch_warnings(
                    record=True
                ) as caught:
                    warnings.simplefilter(
                        "always", ConvergenceWarning
                    )

                    fit_start = time.perf_counter()
                    model.fit(X_train, y_train)
                    fit_seconds = (
                        time.perf_counter() - fit_start
                    )

                warning_count = sum(
                    issubclass(w.category, ConvergenceWarning)
                    for w in caught
                )

                predict_start = time.perf_counter()

                prediction = (
                    y_scaler.inverse_transform(
                        model.predict(X_val).reshape(-1, 1)
                    ).ravel()
                )

                predict_seconds = (
                    time.perf_counter() - predict_start
                )

                assert np.isfinite(prediction).all()

                squared_errors = (y_val_raw-prediction)**2
                mse = float(np.mean(squared_errors))

                record = {
                    "Fold": fold,
                    "C": float(C),
                    "Training_Rows": len(X_train),
                    "Validation_Rows": len(X_val),
                    "MSE_C2": mse,
                    "RMSE_C": float(np.sqrt(mse)),
                    "SSE_C2": float(np.sum(squared_errors)),
                    "Fit_Seconds": fit_seconds,
                    "Predict_Seconds": predict_seconds,
                    "Iterations": int(model.n_iter_),
                    "Convergence_Warnings": warning_count,
                    "Status": (
                        "success" if warning_count == 0
                        else "convergence_warning"
                    )
                }

            finally:
                stop.set()
                monitor.join(timeout=2)
                memory_samples.append(
                    process.memory_info().rss
                )

            record["Peak_Process_RSS_GiB"] = (
                max(memory_samples) / 1024**3
            )

            stage9j_records.append(record)

            print(
                f"C={C:.6f}, "
                f"MSE={mse:.4f}, "
                f"Fit={fit_seconds:.2f}s, "
                f"Warnings={warning_count}"
            )

            # Save after every completed fit.
            pd.DataFrame(stage9j_records).to_csv(
                STAGE9_DIR / "stage9j_scalability_trials.csv",
                index=False
            )

            del model
            gc.collect()

    stage9j_results = pd.DataFrame(stage9j_records)

    print("\nSTAGE 9J — EXPANDED TRAINING RESULTS")
    display(stage9j_results.round(4))

else:
    print("Set RUN_STAGE9J = True to execute.")


In [ ]:

# ============================================================
# STAGE 9J — Accuracy and scalability summary
# ============================================================

assert "stage9j_results" in globals()
assert len(stage9j_results) == 6

stage9j_summary = (
    stage9j_results.groupby("C", as_index=False)
    .agg(
        Mean_MSE_C2=("MSE_C2", "mean"),
        Mean_RMSE_C=("RMSE_C", "mean"),
        Total_Fit_S=("Fit_Seconds", "sum"),
        Total_Predict_S=("Predict_Seconds", "sum"),
        Warnings=("Convergence_Warnings", "sum"),
        Max_RSS_GiB=("Peak_Process_RSS_GiB", "max")
    )
)

baseline = stage9i_summary[
    ["C", "Mean_MSE_C2", "Total_Fit_Seconds"]
].rename(columns={
    "Mean_MSE_C2": "MSE_640",
    "Total_Fit_Seconds": "Fit_640_S"
})

stage9j_summary = stage9j_summary.merge(
    baseline, on="C", validate="one_to_one"
)

stage9j_summary["MSE_Change_C2"] = (
    stage9j_summary.Mean_MSE_C2
    - stage9j_summary.MSE_640
)

stage9j_summary["Fit_Time_Ratio"] = (
    stage9j_summary.Total_Fit_S
    / stage9j_summary.Fit_640_S
)

print("STAGE 9J — SCALABILITY COMPARISON")
display(stage9j_summary.round(4))

stage9j_summary.to_csv(
    STAGE9_DIR / "stage9j_scalability_summary.csv",
    index=False
)


## 9K — Sample-Size-Aware Regularisation

For epsilon-insensitive LinearSVR, increasing the number of training rows at fixed `C` changes the relative weight of the data-fit term and regularisation. To test this **hypothesis**, Stage 9K repeats the original grouped three-fold experiment using 1,280 training observations/profile and 640 validation observations/profile.

It tests two **freshly calculated** regularisation values: `C_GRID_9I[3] × (640 / 1280)` and `C_GRID_9I[4] × (640 / 1280)`. These are derived from the current Stage 9I candidate grid; no archived values of `C` or historical findings are reused. Compare all four conditions from Stages 9J–9K for development-validation MSE/RMSE, convergence, runtime and observed process memory. The full-development extrapolation remains an explicitly labelled heuristic, not a measured million-row optimum.

In [ ]:

# ============================================================
# STAGE 9K — Sample-size-aware C sensitivity
# ============================================================

import gc
import os
import time
import warnings
import threading

import numpy as np
import pandas as pd
import psutil

from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVR
from sklearn.exceptions import ConvergenceWarning

RUN_STAGE9K = True  # Change to True when ready

# Reference: 640 -> 1280 training observations/profile
SAMPLE_RATIO_9K = 640 / 1280

C_GRID_9K = [
    float(C_GRID_9I[3] * SAMPLE_RATIO_9K),
    float(C_GRID_9I[4] * SAMPLE_RATIO_9K)
]

print("New C candidates:", C_GRID_9K)

if RUN_STAGE9K:

    assert len(stage9j_data) == 55 * 1280
    assert set(stage9j_data.profile_id).isdisjoint(test_ids)

    stage9k_records = []

    for fold in (1, 2, 3):

        train_df = stage9j_data.loc[
            stage9j_data.cv_fold.ne(fold)
        ]

        val_df = stage9j_data.loc[
            stage9j_data.cv_fold.eq(fold)
            & stage9j_data.nest_rank.lt(640)
        ]

        assert set(train_df.profile_id).isdisjoint(
            set(val_df.profile_id)
        )

        X_train_raw = train_df[
            stage9_feature_cols
        ].to_numpy(dtype=np.float64)

        X_val_raw = val_df[
            stage9_feature_cols
        ].to_numpy(dtype=np.float64)

        y_train_raw = train_df[
            "pm_target"
        ].to_numpy(dtype=np.float64)

        y_val_raw = val_df[
            "pm_target"
        ].to_numpy(dtype=np.float64)

        # Training-fold-only preprocessing
        xs = StandardScaler()
        ys = StandardScaler()

        X_train = xs.fit_transform(X_train_raw)
        X_val = xs.transform(X_val_raw)

        y_train = ys.fit_transform(
            y_train_raw.reshape(-1, 1)
        ).ravel()

        print(f"\nFOLD {fold}")

        for C in C_GRID_9K:

            gc.collect()

            model = LinearSVR(
                C=C,
                epsilon=STAGE9_SVR_HP["epsilon"],
                loss="epsilon_insensitive",
                dual=True,
                tol=1e-4,
                max_iter=10000,
                random_state=STAGE9_SEED
            )

            process = psutil.Process(os.getpid())
            rss_before = process.memory_info().rss

            memory_samples = [rss_before]
            stop = threading.Event()

            monitor = threading.Thread(
                target=stage9j_memory_monitor,
                args=(stop, memory_samples),
                daemon=True
            )

            monitor.start()

            try:
                with warnings.catch_warnings(
                    record=True
                ) as caught:

                    warnings.simplefilter(
                        "always",
                        ConvergenceWarning
                    )

                    t0 = time.perf_counter()
                    model.fit(X_train, y_train)
                    fit_seconds = time.perf_counter() - t0

                warning_count = sum(
                    issubclass(w.category, ConvergenceWarning)
                    for w in caught
                )

                t1 = time.perf_counter()

                y_pred = ys.inverse_transform(
                    model.predict(X_val).reshape(-1, 1)
                ).ravel()

                predict_seconds = time.perf_counter() - t1

                assert np.isfinite(y_pred).all()

                squared_errors = (y_val_raw - y_pred) ** 2
                mse = float(np.mean(squared_errors))

                record = {
                    "Fold": fold,
                    "C": C,
                    "Training_Rows": len(X_train),
                    "Validation_Rows": len(X_val),
                    "MSE_C2": mse,
                    "RMSE_C": float(np.sqrt(mse)),
                    "Fit_Seconds": fit_seconds,
                    "Predict_Seconds": predict_seconds,
                    "Iterations": int(model.n_iter_),
                    "Convergence_Warnings": warning_count,
                    "Status": (
                        "success" if warning_count == 0
                        else "convergence_warning"
                    )
                }

            finally:
                stop.set()
                monitor.join(timeout=2)
                memory_samples.append(
                    process.memory_info().rss
                )

            record["Peak_Process_RSS_GiB"] = (
                max(memory_samples) / 1024**3
            )

            stage9k_records.append(record)

            print(
                f"C={C:.6f} | "
                f"MSE={mse:.4f} | "
                f"Fit={fit_seconds:.2f}s | "
                f"Warnings={warning_count}"
            )

            pd.DataFrame(stage9k_records).to_csv(
                STAGE9_DIR / "stage9k_C_scaling_trials.csv",
                index=False
            )

            del model
            gc.collect()

    stage9k_results = pd.DataFrame(stage9k_records)

    assert len(stage9k_results) == 6

    print("\nSTAGE 9K RESULTS")
    display(stage9k_results.round(5))

else:
    print("Stage 9K disabled. Set RUN_STAGE9K=True.")


In [ ]:

# ============================================================
# STAGE 9K — Combined regularisation comparison
# ============================================================

assert "stage9k_results" in globals()
assert "stage9j_results" in globals()

combined = pd.concat(
    [stage9j_results, stage9k_results],
    ignore_index=True
)

assert len(combined) == 12

assert (
    combined.groupby("C")["Fold"].nunique().eq(3)
).all()

stage9k_summary = (
    combined.groupby("C", as_index=False)
    .agg(
        Mean_MSE_C2=("MSE_C2", "mean"),
        Mean_RMSE_C=("RMSE_C", "mean"),
        Total_Fit_S=("Fit_Seconds", "sum"),
        Total_Predict_S=("Predict_Seconds", "sum"),
        Warnings=("Convergence_Warnings", "sum"),
        Max_RSS_GiB=("Peak_Process_RSS_GiB", "max")
    )
)

print("STAGE 9K — COMBINED RESULTS")
display(
    stage9k_summary
    .sort_values("Mean_MSE_C2")
    .round(5)
)

stage9k_summary.to_csv(
    STAGE9_DIR / "stage9k_combined_summary.csv",
    index=False
)


**Sample-size-aware regularisation.** The decrease in error as `C` is reduced at the 1,280/profile budget is evidence for this tested grid, not proof of a universal inverse-`N` law. The chosen full-data `C` is still a heuristic extrapolation.

### Figure 9.3 — Scaled regularisation at 1,280 training samples per profile

The four points are the already tested `C` settings from Stages 9J–9K. Their labels indicate **total three-fold fit time**. The plot supports the documented small-sample sensitivity finding; it does not validate an optimum for the full 1,055,198-row development fit.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

_scaled=stage9k_summary.sort_values("C")
fig,ax=plt.subplots(figsize=(8.7,4.4))
ax.semilogx(_scaled["C"],_scaled["Mean_MSE_C2"],"o-",color="#4A8790")
for r in _scaled.itertuples():
    ax.annotate(f"{r.Total_Fit_S:.1f}s",(r.C,r.Mean_MSE_C2),
                xytext=(4,9),textcoords="offset points",fontsize=9)
ax.set(xlabel="Regularisation parameter C (log scale)",ylabel="Mean grouped-CV MSE (°C²)",
       title="Sample-size-aware C screening (times annotated)")
ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(Path(OUTPUT_DIR)/"figures"/"chapter9_scaled_regularisation.png",dpi=180)
plt.show()


---
# 10. Export development-only configuration manifest

This export reads **this run's actual Bayesian trial CSV dataframes and Stage 9 sensitivity scores**. It never reads the previous finite-grid manifest, it never uses historical test scores to choose settings, and it does not fit any full-development final estimator. The freshly selected RBF-SVR Bayesian reference used by Stage 9 is recorded and verified against the new SVR trial CSV.

**LinearSVR caution:** The historical Stage 9 full-data C extrapolation uses the ratio of development-CV training rows to 1,055,198. This is a documented regularisation heuristic, not a newly cross-validated million-row optimum.

In [ ]:
import json, hashlib, sys, platform, time, sklearn, pandas, numpy
from pathlib import Path

# Do not release a configuration manifest if a required historical search did not run.
assert len(full_trials)==20
assert len(scaled_results)==5
assert all(len(scaled_results[name])==12 for name in MODEL_ORDER)
assert len(stage9k_summary)==4 and len(stage9k_results)==6
assert len(stage9i_trials)==15 and len(stage9h_results)>0
assert DATASET_SHA256_ACTUAL==DATASET_SHA256_EXPECTED
assert sorted(test_ids)==LOCKED_TEST_IDS

def _selected_row(table):
    """Select minimum actual CV-MSE row from this run's recorded candidates."""
    valid=table.loc[np.isfinite(table['mean_cv_mse'])]
    assert not valid.empty
    return valid.loc[valid['mean_cv_mse'].idxmin()]

ols_best=_selected_row(full_trials)
selected_configs={
    'OLS':{
        'reference_source':'historical full-development QR Bayes search (20 evaluations)',
        'temporal_spans':[int(ols_best[f's{j}']) for j in range(1,5)],
        'hyperparameters':{},
        'mean_development_cv_mse_c2':float(ols_best['mean_cv_mse']),
        'training_budget':'all 55 development profiles in QR-compressed CV'
    }
}
for _name in MODEL_ORDER:
    _table=scaled_results[_name]
    _best=_selected_row(_table)
    selected_configs[_name]={
        'reference_source':'historical larger-sample GP Bayesian search (12 evaluations)',
        'temporal_spans':[int(_best[f's{j}']) for j in range(1,5)],
        'hyperparameters':json.loads(_best['hyperparameters_json']),
        'mean_development_cv_mse_c2':float(_best['mean_cv_mse']),
        'training_samples_per_profile':int(TRAIN_SAMPLES[_name]),
        'validation_samples_per_profile':int(VALIDATION_SAMPLES),
        'warnings_across_search':int(_table['convergence_warnings'].sum())
    }

# Stage 9 now uses the fresh Bayesian RBF optimum from this Run All.
assert tuple(STAGE9_SPANS) == tuple(selected_configs['SVR']['temporal_spans'])
assert np.isclose(STAGE9_SVR_HP['C'], selected_configs['SVR']['hyperparameters']['C'])
assert np.isclose(STAGE9_SVR_HP['epsilon'], selected_configs['SVR']['hyperparameters']['epsilon'])
_eligible=stage9k_summary.loc[stage9k_summary['Warnings'].eq(0)].sort_values('Mean_MSE_C2')
assert len(_eligible)>0,'No fully converged Stage 9K candidate'
_refC=float(_eligible.iloc[0]['C'])
_refTrain=float(stage9k_results.drop_duplicates('Fold')['Training_Rows'].mean())
_Cfull=float(_refC * _refTrain / len(development_df))
assert _Cfull>0
selected_configs['LinearSVR']={
    'reference_source':'historical Stage 9I–9K 640/1280 profile grouped-CV study',
    'temporal_spans':list(STAGE9_SPANS),
    'hyperparameters':{
        'C':_Cfull, 'epsilon':float(STAGE9_SVR_HP['epsilon']),
        'loss':'epsilon_insensitive','dual':True,'tol':1e-4,
        'max_iter':10000,'random_state':int(STAGE9_SEED)
    },
    'cv_selected_C':_refC,
    'cv_mean_training_rows':_refTrain,
    'full_C_is_sample_size_extrapolation':True,
    'cv_mean_mse_c2':float(_eligible.iloc[0]['Mean_MSE_C2'])
}
# The historical GP search includes exact RBF-SVR, but the final full-development
# six-model continuation substitutes the later Stage-9 LinearSVR method.
# Keep the RBF-SVR Bayesian results as a separate development reference.
final_six_names=('OLS','k-NN','Random Forest','Extra Trees','MLP','LinearSVR')
final_six_configs={name:selected_configs[name] for name in final_six_names}
assert len(final_six_configs)==6
manifest={
    'status':'RESTORED_HISTORICAL_PROTOCOL_DEVELOPMENT_SELECTED',
    'full_run_completed':True,
    'source_dataset_sha256':DATASET_SHA256_ACTUAL,
    'source_primary_sha256':'2f8141c04f0635a9b7504240058f6b27981359d607e530032d9cd5ae18adeeca',
    'source_stage9_sha256':'de45cd4384597860b38817c03bc2e13851140dfc682231b26f483ba69e337a89',
    'source_profile_partition':{'development_rows':int(len(development_df)),'test_rows_excluded':int(len(heldout_df)),'development_profiles':development_ids,'excluded_test_profiles':LOCKED_TEST_IDS},
    'profile_folds':{str(int(r.profile_id)):int(r.cv_fold) for r in profile_cv.itertuples()},
    'original_gp_searches':{'OLS_pilot':8,'OLS_expansion_added':20,'five_family_pilot_per_model':8,'OLS_full_data_total':20,'five_family_larger_per_model':12},
    'sample_budgets':{'pilot_per_profile':80,'larger_model_train_per_profile':TRAIN_SAMPLES,'larger_model_validation_per_profile':VALIDATION_SAMPLES,'stage9_initial_per_profile':640,'stage9_expanded_per_profile':1280},
    'selected_model_configurations':final_six_configs,
    'bayesian_rbf_svr_development_reference':selected_configs['SVR'],
    'stage9_fresh_bayesian_rbf_reference':{'spans':list(STAGE9_SPANS),'svr_parameters':STAGE9_SVR_HP,'is_fresh_bayesian_winner':True,'source_trial':int(_stage9_best['trial'])},
    'environment':{'python':sys.version.split()[0],'platform':platform.platform(),'numpy':numpy.__version__,'pandas':pandas.__version__,'sklearn':sklearn.__version__,'scipy':scipy.__version__},
    'heldout_data_used_for_tuning':False,
    'previous_historical_holdout_results_examined':True,
    'methodology_difference_from_archives':'Original algorithms/search spaces/folds/budgets retained. Provenance handoffs altered to use fresh Stage5/Stage9 SVR winners and OLS full-data seed; development-only export and documentation added',
    'methodology_difference_from_replaced_finite_grid':'restores original GP Bayesian searches, historical sample budgets, and matched-kernel SVR comparison; every selected setting now comes from current run'
}
OUTPUT_EXPORT=Path(BASE_DIR)/'mma3001_restored_historical_protocol'
OUTPUT_EXPORT.mkdir(parents=True,exist_ok=True)
manifest_path=OUTPUT_EXPORT/'selected_model_configurations_historical_protocol.json'
manifest_path.write_text(json.dumps(manifest,indent=2,default=str))
print('PASS: restored-protocol manifest exported:',manifest_path)
for _k,_v in selected_configs.items():
    print(_k,_v['temporal_spans'],_v['hyperparameters'])


## 11. Export complete development evidence ZIP

Include actual CSV trials, held-out-safe verification reports, Stage 9 figures and the manifest. This operation **does not run any model fit**. Save the generated ZIP and this executed `.ipynb` outside the Colab runtime.

## 11.1 Generate browsable HTML API documentation (pdoc)

The brief explicitly requires **HTML source-code documentation**. This cell extracts the source-derived functions verified by the repository's AST parity tests, installs pdoc in Colab if necessary, and generates actual HTML under `mma3001_restored_historical_protocol/docs/pdoc/`. The HTML is included in the results ZIP. It does not fit or select any models.

In [ ]:
# MMA3001 repository compliance: produce and verify source modules, genuine pdoc HTML and pytest report.
# This does not train or evaluate an estimator; the modules mirror existing notebook function ASTs.
from pathlib import Path
import json, os, sys, subprocess, importlib.util, hashlib
_DOC_MODULES = json.loads('{"pmsm_project/data_features.py": "\\"\\"\\"Reproducible PMSM algorithms extracted from historical Notebook 02.\\n\\nFunctions here match the executed notebook\'s executable AST statements. They\\nare separately importable for unit testing and browsable HTML documentation.\\nThe original notebook executes its own copy of the identical function bodies.\\n\\"\\"\\"\\nimport numpy as np\\nimport pandas as pd\\nimport heapq\\nfrom sklearn.pipeline import Pipeline\\nfrom sklearn.preprocessing import StandardScaler\\nfrom sklearn.svm import SVR\\nfrom sklearn.neighbors import KNeighborsRegressor\\nfrom sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor\\nfrom sklearn.neural_network import MLPRegressor\\n\\n\\ndef make_paper_inputs(data):\\n    \\"\\"\\"Construct the 12 instantaneous PMSM engineering predictors.\\n\\n    Parameters\\n    ----------\\n    data : pandas.DataFrame\\n        Original 2-Hz measurements from one or more motor profiles; must\\n        contain original electrical, mechanical and temperature sensors.\\n\\n    Returns\\n    -------\\n    pandas.DataFrame\\n        Twelve paper-defined predictors in the original feature order.\\n        Engineering units follow the source sensors and derived quantities.\\n\\n    Notes\\n    -----\\n    Does not use the permanent-magnet target to construct predictors.\\n    \\"\\"\\"\\n\\n    X = pd.DataFrame(index=data.index)\\n\\n    # Seven measured inputs\\n    X[\\"theta_a\\"] = data[\\"ambient\\"]\\n    X[\\"theta_c\\"] = data[\\"coolant\\"]\\n    X[\\"u_d\\"] = data[\\"u_d\\"]\\n    X[\\"u_q\\"] = data[\\"u_q\\"]\\n    X[\\"i_d\\"] = data[\\"i_d\\"]\\n    X[\\"i_q\\"] = data[\\"i_q\\"]\\n    X[\\"n_mech\\"] = data[\\"motor_speed\\"]\\n\\n    # Five derived inputs\\n    X[\\"u_s\\"] = np.hypot(X[\\"u_d\\"], X[\\"u_q\\"])\\n    X[\\"i_s\\"] = np.hypot(X[\\"i_d\\"], X[\\"i_q\\"])\\n\\n    # Apparent electrical power\\n    X[\\"S_el\\"] = 1.5 * X[\\"u_s\\"] * X[\\"i_s\\"]\\n\\n    # Angular speed in rad/s\\n    omega = 2 * np.pi * X[\\"n_mech\\"] / 60\\n\\n    X[\\"i_s_omega\\"] = X[\\"i_s\\"] * omega\\n    X[\\"S_el_omega\\"] = X[\\"S_el\\"] * omega\\n\\n    return X\\n\\n\\ndef make_temporal_features(X, profile_ids, spans):\\n    # Calculate temporal features independently per profile\\n    \\"\\"\\"Add causal exponentially weighted mean and standard-deviation features.\\n\\n    Parameters\\n    ----------\\n    X : pandas.DataFrame\\n        Instantaneous engineering predictors in their chronological order.\\n    profile_ids : array-like\\n        Per-row recording identifiers used to reset histories between\\n        motor measurement profiles; must align with ``X``.\\n    spans : sequence[int]\\n        Positive exponentially weighted spans measured in observations.\\n        Each span produces EWMA and EWMS features for every base predictor.\\n\\n    Returns\\n    -------\\n    pandas.DataFrame\\n        Twelve instantaneous plus 24 features per requested span when\\n        there are twelve base predictors; four spans produce 108 columns.\\n\\n    Notes\\n    -----\\n    Uses Pandas ``adjust=True`` exponentially weighted means and\\n    population standard deviations with ``bias=True``. Compute on\\n    complete chronological profiles before any training subsampling.\\n    \\"\\"\\"\\n    features = {}\\n\\n    for column in X.columns:\\n        grouped = X[column].groupby(\\n            profile_ids, sort=False\\n        )\\n\\n        for span in spans:\\n            features[f\\"{column}_ewma_s{span}\\"] = (\\n                grouped.transform(\\n                    lambda z: z.ewm(\\n                        span=span, adjust=True\\n                    ).mean()\\n                )\\n            )\\n\\n            features[f\\"{column}_ewms_s{span}\\"] = (\\n                grouped.transform(\\n                    lambda z: z.ewm(\\n                        span=span, adjust=True\\n                    ).std(bias=True)\\n                )\\n            )\\n\\n    return pd.concat(\\n        [X, pd.DataFrame(features, index=X.index)],\\n        axis=1\\n    )\\n\\n\\ndef vectorised_temporal_sample(X_original, positions, spans):\\n    \\"\\"\\"Sample causal EWM features with 12-column vectorised Pandas filters.\\n\\n    Returns exactly the same columns/values as make_temporal_features,\\n    but avoids looping separately over 96 individual feature series.\\n    Every EWM is computed over the full original profile *before* sampling.\\n    \\"\\"\\"\\n    sampled_columns = {col: X_original[col].iloc[positions].to_numpy()\\n                       for col in X_original.columns}\\n    for s in spans:\\n        ewm = X_original.ewm(span=int(s), adjust=True)\\n        historical_mean = ewm.mean().iloc[positions]\\n        historical_std = ewm.std(bias=True).iloc[positions]\\n        for col in X_original.columns:\\n            sampled_columns[f\'{col}_ewma_s{s}\'] = historical_mean[col].to_numpy()\\n            sampled_columns[f\'{col}_ewms_s{s}\'] = historical_std[col].to_numpy()\\n    return pd.DataFrame(sampled_columns,index=X_original.index[positions])\\n\\n\\ndef stage6_feature_names(columns, spans):\\n    \\"\\"\\"Return the original ordered instantaneous and causal temporal feature names.\\n\\n    Parameters\\n    ----------\\n    columns : iterable[str]\\n        Names of the base electrical and thermal features.\\n    spans : sequence[int]\\n        Causal history spans used to construct EWMA and EWMS columns.\\n\\n    Returns\\n    -------\\n    list[str]\\n        Exact model feature order for reference and vectorised implementations.\\n    \\"\\"\\"\\n    return list(columns) + [\\n        name for col in columns for span in spans\\n        for name in (f\'{col}_ewma_s{span}\', f\'{col}_ewms_s{span}\')\\n    ]\\n", "pmsm_project/models.py": "\\"\\"\\"Reproducible PMSM algorithms extracted from historical Notebook 02.\\n\\nFunctions here match the executed notebook\'s executable AST statements. They\\nare separately importable for unit testing and browsable HTML documentation.\\nThe original notebook executes its own copy of the identical function bodies.\\n\\"\\"\\"\\nimport numpy as np\\nimport pandas as pd\\nimport heapq\\nfrom sklearn.pipeline import Pipeline\\nfrom sklearn.preprocessing import StandardScaler\\nfrom sklearn.svm import SVR\\nfrom sklearn.neighbors import KNeighborsRegressor\\nfrom sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor\\nfrom sklearn.neural_network import MLPRegressor\\n\\n\\ndef decode_spans(parameters):\\n    \\"\\"\\"Decode the four Bayesian search coordinates into four ordered spans.\\n\\n    Parameters\\n    ----------\\n    parameters : sequence[int]\\n        ``(span_1, positive_gap_2, positive_gap_3, positive_gap_4)``.\\n\\n    Returns\\n    -------\\n    list[int]\\n        Four strictly increasing EWMA/EWMS smoothing spans.\\n    \\"\\"\\"\\n    a, d2, d3, d4 = map(int, parameters)\\n\\n    return [\\n        a,\\n        a + d2,\\n        a + d2 + d3,\\n        a + d2 + d3 + d4\\n    ]\\n\\n\\ndef encode_spans(spans):\\n    \\"\\"\\"Encode four ordered smoothing spans as a base span and positive gaps.\\n\\n    Parameters\\n    ----------\\n    spans : sequence[int]\\n        Four strictly increasing temporal smoothing spans.\\n\\n    Returns\\n    -------\\n    list[int]\\n        Coordinates accepted by the original Bayesian search space.\\n\\n    Raises\\n    ------\\n    ValueError\\n        If spans are not strictly increasing.\\n    \\"\\"\\"\\n    a, b, c, d = map(int, spans)\\n\\n    if not a < b < c < d:\\n        raise ValueError(\\"Spans must be strictly increasing\\")\\n\\n    return [a, b-a, c-b, d-c]\\n\\n\\ndef make_regressor(model_name, hp):\\n    \\"\\"\\"Construct the historical, model-specific scikit-learn regression estimator.\\n\\n    Parameters\\n    ----------\\n    model_name : str\\n        One of ``SVR``, ``k-NN``, ``Random Forest``, ``Extra Trees``, or ``MLP``.\\n    hp : dict\\n        Original Gaussian-process search hyperparameters for that family.\\n\\n    Returns\\n    -------\\n    sklearn.base.RegressorMixin\\n        Unfitted estimator with the original model-specific solver settings.\\n        Feature and target preprocessing is applied separately by the\\n        cross-validation objective, never inside this function.\\n\\n    Raises\\n    ------\\n    ValueError\\n        If ``model_name`` is not a supported family.\\n    \\"\\"\\"\\n\\n    if model_name == \\"SVR\\":\\n        return SVR(\\n            kernel=\\"rbf\\",\\n            C=float(hp[\\"C\\"]),\\n            epsilon=float(hp[\\"epsilon\\"]),\\n            gamma=\\"scale\\"\\n        )\\n\\n    if model_name == \\"k-NN\\":\\n        return KNeighborsRegressor(\\n            n_neighbors=int(hp[\\"n_neighbors\\"]),\\n            weights=hp[\\"weights\\"],\\n            n_jobs=2\\n        )\\n\\n    if model_name in [\\"Random Forest\\", \\"Extra Trees\\"]:\\n\\n        estimator_class = (\\n            RandomForestRegressor\\n            if model_name == \\"Random Forest\\"\\n            else ExtraTreesRegressor\\n        )\\n\\n        return estimator_class(\\n            n_estimators=int(hp[\\"n_estimators\\"]),\\n            max_depth=int(hp[\\"max_depth\\"]),\\n            min_samples_split=int(hp[\\"min_samples_split\\"]),\\n            min_samples_leaf=int(hp[\\"min_samples_leaf\\"]),\\n            bootstrap=(hp[\\"bootstrap\\"] == \\"yes\\"),\\n            max_features=\\"sqrt\\",\\n            random_state=42,\\n            n_jobs=2\\n        )\\n\\n    if model_name == \\"MLP\\":\\n\\n        n_layers = int(hp[\\"n_layers\\"])\\n        n_units = int(hp[\\"n_units\\"])\\n\\n        return MLPRegressor(\\n            hidden_layer_sizes=(n_units,) * n_layers,\\n            activation=\\"relu\\",\\n            solver=\\"adam\\",\\n            alpha=float(hp[\\"alpha\\"]),\\n            learning_rate_init=float(\\n                hp[\\"learning_rate_init\\"]\\n            ),\\n            batch_size=64,\\n            max_iter=200,\\n            random_state=42,\\n            early_stopping=False\\n        )\\n\\n    raise ValueError(f\\"Unknown model: {model_name}\\")\\n\\n\\ndef make_json_safe(values):\\n    \\"\\"\\"Convert NumPy scalar trial coordinates into JSON-compatible Python values.\\n\\n    Parameters\\n    ----------\\n    values : iterable\\n        Bayesian search coordinates including NumPy scalar types.\\n\\n    Returns\\n    -------\\n    list\\n        Equivalent values suitable for JSON and ``skopt.Optimizer.tell``.\\n    \\"\\"\\"\\n    return [\\n        v.item() if isinstance(v, np.generic) else v\\n        for v in values\\n    ]\\n", "pmsm_project/__init__.py": "\\"\\"\\"Original-method PMSM utilities for reproducible Notebook 02 experiments.\\n\\nThe notebook continues to execute its original code cells. These modules\\npublish exact, source-derived reusable functions as importable documentation\\nand lightweight test targets. No archived model fits or historical CV winners\\nare loaded here.\\n\\"\\"\\"\\n", "pmsm_project/sampling.py": "\\"\\"\\"Reproducible PMSM algorithms extracted from historical Notebook 02.\\n\\nFunctions here match the executed notebook\'s executable AST statements. They\\nare separately importable for unit testing and browsable HTML documentation.\\nThe original notebook executes its own copy of the identical function bodies.\\n\\"\\"\\"\\nimport numpy as np\\nimport pandas as pd\\nimport heapq\\nfrom sklearn.pipeline import Pipeline\\nfrom sklearn.preprocessing import StandardScaler\\nfrom sklearn.svm import SVR\\nfrom sklearn.neighbors import KNeighborsRegressor\\nfrom sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor\\nfrom sklearn.neural_network import MLPRegressor\\n\\n\\ndef nested_sample_order(profile_length: int, max_samples: int) -> list[int]:\\n    \\"\\"\\"Return nested sample positions in selection order (not time order).\\n\\n    Repeatedly bisects the largest interval, preserving first and last\\n    positions. For every n in (80,160,320,640), the first n selections\\n    are a strict subset of the first 2n selections when those exist.\\n    \\"\\"\\"\\n    if not (2 <= max_samples <= profile_length):\\n        raise ValueError(\'Require 2 <= max_samples <= profile_length\')\\n    selected = [0, profile_length - 1]\\n    intervals = [(-(profile_length - 1), 0, profile_length - 1)]\\n    while len(selected) < max_samples:\\n        _, left, right = heapq.heappop(intervals)\\n        midpoint = (left + right) // 2\\n        if midpoint in (left, right):\\n            continue\\n        selected.append(midpoint)\\n        for a,b in ((left,midpoint),(midpoint,right)):\\n            if b-a > 1:\\n                heapq.heappush(intervals, (-(b-a),a,b))\\n    assert len(selected) == len(set(selected)) == max_samples\\n    return selected\\n", "pmsm_project/selection.py": "\\"\\"\\"Fresh-run result selection and handoff verification.\\n\\nThe functions accept trial records produced in the *current* execution and\\nraise rather than using an archived best parameter as a fallback.\\n\\"\\"\\"\\nimport json\\nimport numpy as np\\nimport pandas as pd\\n\\n\\ndef select_fresh_winner(trials: pd.DataFrame) -> dict:\\n    \\"\\"\\"Select the minimum mean grouped-CV MSE from a fresh Bayesian trial table.\\n\\n    Parameters\\n    ----------\\n    trials : pandas.DataFrame\\n        Full record of the current Bayesian search. Required columns are\\n        ``trial``, ``mean_cv_mse``, ``s1..s4``, and ``hyperparameters_json``.\\n\\n    Returns\\n    -------\\n    dict\\n        Four causal spans, decoded hyperparameters, trial number and CV MSE.\\n\\n    Raises\\n    ------\\n    ValueError\\n        If any required field is missing, scores are invalid or trials empty.\\n    \\"\\"\\"\\n    required = {\'trial\',\'mean_cv_mse\',\'s1\',\'s2\',\'s3\',\'s4\',\'hyperparameters_json\'}\\n    if not required.issubset(trials.columns) or len(trials)==0:\\n        raise ValueError(\'Incomplete fresh Bayesian trial data\')\\n    if not np.isfinite(trials[\'mean_cv_mse\'].astype(float)).all():\\n        raise ValueError(\'Nonfinite Bayesian trial scores\')\\n    best = trials.loc[trials[\'mean_cv_mse\'].idxmin()]\\n    spans = [int(best[f\'s{i}\']) for i in range(1,5)]\\n    if spans != sorted(spans) or len(set(spans))!=4:\\n        raise ValueError(\'Invalid ordered temporal spans\')\\n    hp = json.loads(best[\'hyperparameters_json\'])\\n    return {\'spans\':spans,\'hyperparameters\':hp,\'trial\':int(best[\'trial\']),\\n            \'mean_cv_mse\':float(best[\'mean_cv_mse\'])}\\n"}')
_DOC_TEST_SOURCE = '"""Fast tests for original-method, source-derived computational helpers."""\nimport ast, hashlib, json\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport pytest\nfrom pmsm_project.data_features import make_paper_inputs, make_temporal_features, vectorised_temporal_sample, stage6_feature_names\nfrom pmsm_project.models import decode_spans, encode_spans, make_regressor\nfrom pmsm_project.sampling import nested_sample_order\nfrom pmsm_project.selection import select_fresh_winner\n\n\ndef test_spans_encoding_roundtrip():\n    spans=[31,89,770,2531]\n    assert decode_spans(encode_spans(spans))==spans\n\n\ndef test_spans_reject_duplicates():\n    with pytest.raises(ValueError):\n        encode_spans([3,3,8,10])\n\n\ndef test_nested_sample_unique_boundaries_and_subset():\n    order=nested_sample_order(1500,640)\n    assert len(order)==len(set(order))==640\n    assert order[:2]==[0,1499]\n    for m in [80,160,320]:\n        assert set(order[:m]).issubset(set(order[:m*2]))\n\n\ndef test_ewm_vectorised_matches_original():\n    x=pd.DataFrame({\'i_q\':np.linspace(-2,8,30), \'u_d\':np.sin(np.arange(30))})\n    ids=pd.Series(np.ones(30,dtype=int))\n    spans=[3,10]\n    positions=np.array([0,1,3,8,29])\n    slow=make_temporal_features(x,ids,spans)\n    fast=vectorised_temporal_sample(x,positions,spans)\n    np.testing.assert_allclose(fast[slow.columns].to_numpy(),slow.iloc[positions].to_numpy(),atol=1e-11)\n    for col in x.columns:\n        assert np.isclose(fast[f\'{col}_ewma_s3\'].iloc[0],x[col].iloc[0])\n        assert np.isclose(fast[f\'{col}_ewms_s3\'].iloc[0],0)\n\n\ndef test_temporal_features_causal_prefix():\n    x=pd.DataFrame({\'i_q\':np.arange(10,dtype=float),\'u_d\':np.sin(np.arange(10))})\n    prefix=make_temporal_features(x.iloc[:6],pd.Series([1]*6),[4])\n    extended=make_temporal_features(x,pd.Series([1]*10),[4])\n    np.testing.assert_allclose(prefix.to_numpy(),extended.iloc[:6].to_numpy(),atol=1e-12)\n\n\ndef test_tree_estimator_matches_original_controls():\n    h={\'n_estimators\':80,\'max_depth\':10,\'min_samples_split\':4,\'min_samples_leaf\':2,\'bootstrap\':\'yes\'}\n    m=make_regressor(\'Random Forest\',h)\n    assert m.n_estimators==80 and m.max_features==\'sqrt\' and m.bootstrap is True\n\n\ndef test_svr_kernel_fixed_but_parameters_are_inputs():\n    m=make_regressor(\'SVR\',{\'C\':0.031,\'epsilon\':0.09})\n    assert m.kernel==\'rbf\' and m.gamma==\'scale\'\n    assert m.C==0.031 and m.epsilon==0.09\n\n\ndef test_fresh_selection_uses_current_trial_records():\n    rows=pd.DataFrame([\n        {\'trial\':1,\'mean_cv_mse\':45,\'s1\':10,\'s2\':50,\'s3\':250,\'s4\':800,\'hyperparameters_json\':json.dumps({\'C\':1.,\'epsilon\':0.2})},\n        {\'trial\':2,\'mean_cv_mse\':35,\'s1\':12,\'s2\':55,\'s3\':260,\'s4\':900,\'hyperparameters_json\':json.dumps({\'C\':2.,\'epsilon\':0.1})},\n    ])\n    selection=select_fresh_winner(rows)\n    assert selection[\'trial\']==2 and selection[\'hyperparameters\'][\'C\']==2.\n    with pytest.raises(ValueError):\n        select_fresh_winner(pd.DataFrame())\n\n\ndef test_fresh_selection_refuses_invalid_scores():\n    row=pd.DataFrame([{\'trial\':1,\'mean_cv_mse\':float(\'nan\'), \'s1\':1,\'s2\':2,\'s3\':3,\'s4\':4,\'hyperparameters_json\':\'{}\'}])\n    with pytest.raises(ValueError): select_fresh_winner(row)\n\n\ndef test_executable_ast_matches_notebook_functions():\n    # The documented function bodies MUST remain in sync with executable cells.\n    import nbformat\n    root=Path(__file__).resolve().parents[1]\n    nb_file = root/\'notebooks/02_Experimental_Development_Fresh_Historical_Method.ipynb\'\n    if not nb_file.exists(): pytest.skip(\'Notebook not deployed with this standalone Colab source copy\')\n    nb=nbformat.read(nb_file,as_version=4)\n    mapping=json.loads((root/\'source_function_hashes.json\').read_text())\n    for name,record in mapping.items():\n        module_file=root/\'src/pmsm_project\'/record[\'module\']\n        c = nb.cells[record[\'historical_source_cell\'] + 1] # +1 documentation cell inserted at index 2\n        notebook_nodes=[x for x in ast.parse(c.source.replace(\'%pip -q install scikit-optimize\',\'\')).body if isinstance(x,ast.FunctionDef) and x.name==name]\n        assert notebook_nodes, name\n        module_nodes=[x for x in ast.parse(module_file.read_text()).body if isinstance(x,ast.FunctionDef) and x.name==name]\n        assert module_nodes,name\n        def executable_ast(node):\n            node=__import__(\'copy\').deepcopy(node)\n            if node.body and isinstance(node.body[0],ast.Expr) and isinstance(node.body[0].value,ast.Constant) and isinstance(node.body[0].value.value,str):\n                node.body.pop(0)\n            return ast.dump(node,include_attributes=False)\n        assert executable_ast(notebook_nodes[0])==executable_ast(module_nodes[0]),name\n'
_DOC_FRESH_TEST_SOURCE = '"""Contract tests: no old selected numerical values can be reused as winners."""\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport nbformat\n\n\ndef _notebook():\n    path = Path(__file__).resolve().parents[1] / \'notebooks\' / \'02_Experimental_Development_Fresh_Historical_Method.ipynb\'\n    return nbformat.read(path, as_version=4) if path.exists() else None\n\n\ndef test_stage5_uses_fresh_pilot_winner():\n    n = _notebook()\n    if n is None: return\n    cell = next(c.source for c in n.cells if c.cell_type == \'code\' and \'STAGE5_CONFIG_SOURCE\' in c.source)\n    source = cell.split(\'assert len(STAGE5_SPANS)\')[0]\n    trial_rows = []\n    for i in range(8):\n        trial_rows.append({\'trial\':i+1,\'mean_cv_mse\':float(90-i),\n            \'s1\':7+i,\'s2\':90+i,\'s3\':300+i,\'s4\':1000+i,\n            \'hyperparameters_json\':json.dumps({\'C\':0.04+i/10,\'epsilon\':0.11+i/100})})\n    data={\'np\':np,\'pd\':pd,\'json\':json,\'os\':__import__(\'os\'),\n          \'OUTPUT_DIR\':Path(\'/tmp\'), \'N_TRIALS\':8,\n          \'all_results\':{\'SVR\':pd.DataFrame(trial_rows)},\n          \'development_df\':pd.DataFrame({\'profile_id\':[2]*1055198})}\n    # Avoid any data allocation and post-provenance assumptions.\n    exec(source.split(\'import os, json, heapq\')[1].split(\'STAGE5_DIR =\')[0] if False else source[source.index(\'# SAME Stage-5\'):], data)\n    assert data[\'STAGE5_SPANS\'] == (14,97,307,1007)\n    assert np.isclose(data[\'STAGE5_SVR_HP\'][\'C\'],0.74)\n    assert np.isclose(data[\'STAGE5_SVR_HP\'][\'epsilon\'],0.18)\n\n\ndef test_stage9_uses_fresh_scaled_search_winner():\n    n = _notebook()\n    if n is None: return\n    cell = next(c.source for c in n.cells if c.cell_type==\'code\' and \'STAGE9_CONFIG_SOURCE\' in c.source)\n    source=cell.split(\'STAGE9_SEED =\')[0]\n    trial_rows=[]\n    for i in range(12):\n        trial_rows.append({\'trial\':i+1,\'mean_cv_mse\':float(120-i),\n          \'s1\':6+i,\'s2\':100+i,\'s3\':700+i,\'s4\':2000+i,\n          \'hyperparameters_json\':json.dumps({\'C\':0.015+i/100,\'epsilon\':0.03+i/200})})\n    data={\'np\':np,\'pd\':pd,\'json\':json,\'scaled_results\':{\'SVR\':pd.DataFrame(trial_rows)},\'SCALED_TRIALS\':12}\n    exec(source, data)\n    assert data[\'STAGE9_SPANS\']==(17,111,711,2011)\n    assert np.isclose(data[\'STAGE9_SVR_HP\'][\'C\'],0.125)\n    assert np.isclose(data[\'STAGE9_SVR_HP\'][\'epsilon\'],0.085)\n    assert data[\'STAGE9_CONFIG_SOURCE\'].startswith(\'current execution\')\n\n\ndef test_full_ols_seed_is_fresh_result_not_historical_constant():\n    n=_notebook()\n    if n is None: return\n    ols_source=next(c.source for c in n.cells if c.cell_type==\'code\' and \'candidate_spans = list(map(int, best_spans))\' in c.source)\n    assert \'candidate_spans = [89, 265, 622, 3829]\' not in ols_source\n    cfg=next(c.source for c in n.cells if c.cell_type==\'code\' and \'SEED_SPANS = list(map(int, candidate_spans))\' in c.source)\n    assert \'SEED_SPANS = [89, 265, 622, 3829]\' not in cfg\n'
_DOC_ROOT = Path(BASE_DIR) / 'mma3001_restored_historical_protocol'
_DOC_SRC = _DOC_ROOT / 'src'
for _rel, _source in _DOC_MODULES.items():
    _dest = _DOC_SRC / _rel
    _dest.parent.mkdir(parents=True, exist_ok=True)
    _dest.write_text(_source)
# Fail rather than emit purported pdoc compliance without actual pdoc output.
if importlib.util.find_spec('pdoc') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pdoc'], check=True)
_DOCS = _DOC_ROOT / 'docs' / 'pdoc'
_env = {**os.environ, 'PYTHONPATH': str(_DOC_SRC)}
subprocess.run([sys.executable, '-m', 'pdoc', '-o', str(_DOCS), 'pmsm_project'], check=True, env=_env)
assert (_DOCS / 'pmsm_project.html').exists() or any(_DOCS.rglob('*.html')), 'pdoc did not generate HTML'
# Generate a pytest XML report using these same source-derived methods.
_DOC_TEST_FILE = _DOC_ROOT / 'tests' / 'test_original_methods.py'
_DOC_TEST_FILE.parent.mkdir(parents=True, exist_ok=True)
_DOC_TEST_FILE.write_text(_DOC_TEST_SOURCE)
if importlib.util.find_spec('pytest') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytest', 'nbformat'], check=True)
_DOC_REPORT = _DOC_ROOT / 'tests' / 'test_report.xml'
subprocess.run([sys.executable, '-m', 'pytest', '-q', str(_DOC_TEST_FILE), '--junitxml='+str(_DOC_REPORT)], check=True, env=_env)
assert _DOC_REPORT.is_file()
print('PASS — genuine pdoc HTML documentation:', _DOCS)
print('PASS — executed pytest XML report:', _DOC_REPORT)


In [ ]:
from pathlib import Path
import zipfile, json, hashlib, os
zip_path=OUTPUT_EXPORT/'Notebook02_historical_protocol_development_results.zip'
input_roots=[Path(BASE_DIR)/'bayesian_model_pilots',Path(BASE_DIR)/'scaled_model_bayesian',Path(BASE_DIR)/'ols_full_bayesian',Path(BASE_DIR)/'training_feasibility']
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as zf:
    zf.write(manifest_path,arcname='manifests/'+manifest_path.name)
    _docs_root = OUTPUT_EXPORT/'docs'/'pdoc'
    assert _docs_root.is_dir() and any(_docs_root.rglob('*.html'))
    for _file in _docs_root.rglob('*'):
        if _file.is_file():
            zf.write(_file,arcname='docs/pdoc/'+str(_file.relative_to(_docs_root)))
    for _file in (OUTPUT_EXPORT/'src').rglob('*.py'):
        zf.write(_file,arcname='src/'+str(_file.relative_to(OUTPUT_EXPORT/'src')))
    zf.write(OUTPUT_EXPORT/'tests'/'test_report.xml', arcname='tests/test_report.xml')
    zf.write(OUTPUT_EXPORT/'tests'/'test_original_methods.py', arcname='tests/test_original_methods.py')
    zf.write(OUTPUT_EXPORT/'tests'/'test_fresh_handoff.py', arcname='tests/test_fresh_handoff.py')
    for root in input_roots:
        if root.is_dir():
            for file in root.rglob('*'):
                if file.is_file() and file.resolve()!=zip_path.resolve():
                    zf.write(file,arcname='evidence/'+root.name+'/'+str(file.relative_to(root)))
    f=Path('profile_cv_assignments.csv')
    if f.is_file(): zf.write(f,arcname='evidence/profile_cv_assignments.csv')
assert zip_path.exists() and zip_path.stat().st_size>1000
print('COMPLETE — original historical development protocol; no held-out scoring performed.')
print('Manifest:',manifest_path)
print('Download ZIP:',zip_path)
print('ZIP bytes:',zip_path.stat().st_size)
